<a href="https://colab.research.google.com/github/ibelalov/rlmw/blob/main/rlmw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rlmw

Single-notebook development environment for low-weight span-vector search.

Core components:

1. Binary linear algebra over F_2
2. Planted span instance generator
3. Bitset/popcount utilities
4. Direction bank construction
5. Exact discrete-gradient descent
6. Local-minimum support-intersection constraints
7. Failed-local-minimum archive
8. Neural direction model
9. Cross-entropy sampler
10. Strategy-level Q-controller
11. Exact local-minimum intersection solver
12. Training/evaluation dashboard

In [17]:
# @title
# --- 00. Environment setup ---

import os
import sys
import math
import time
import json
import random
import pathlib
import itertools
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
PyTorch available: True
PyTorch: 2.10.0+cpu
CUDA available: False


In [18]:
# @title Environment detection and storage backend selection
import os


def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


# Small self-tests for environment-flag parsing
assert _env_flag("__RLMW_TEST_UNSET__", "0") is False
assert _env_flag("__RLMW_TEST_UNSET__", "1") is True

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

HEADLESS = _env_flag("RLMW_HEADLESS", "0")
SMOKE = _env_flag("RLMW_SMOKE", "0")
USE_DRIVE = IN_COLAB and (not HEADLESS)

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [19]:
# @title Project paths
import os
from pathlib import Path

if USE_DRIVE:
    PROJECT_ROOT = Path("/content/drive/MyDrive") / "rlmw"
else:
    PROJECT_ROOT = Path(os.environ.get("RLMW_PROJECT_ROOT", "/tmp/rlmw"))

DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
EXPORT_DIR = PROJECT_ROOT / "exports"
CACHE_DIR = PROJECT_ROOT / "cache"

for p in [PROJECT_ROOT, DATA_DIR, RUNS_DIR, CHECKPOINT_DIR, EXPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("HEADLESS:", HEADLESS)
print("SMOKE:", SMOKE)
print("USE_DRIVE:", USE_DRIVE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("CACHE_DIR:", CACHE_DIR)

IN_COLAB: True
HEADLESS: False
SMOKE: False
USE_DRIVE: True
PROJECT_ROOT: /content/drive/MyDrive/rlmw
DATA_DIR: /content/drive/MyDrive/rlmw/data
RUNS_DIR: /content/drive/MyDrive/rlmw/runs
CHECKPOINT_DIR: /content/drive/MyDrive/rlmw/checkpoints
EXPORT_DIR: /content/drive/MyDrive/rlmw/exports
CACHE_DIR: /content/drive/MyDrive/rlmw/cache


In [ ]:
# @title Optional Colab dependency bootstrap
import subprocess


def _module_available(module_name: str) -> bool:
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


if IN_COLAB and not HEADLESS:
    if not _module_available("ortools"):
        print("Installing missing Colab dependency: ortools")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ortools"])
        print("Installed ortools. Continue running the notebook from top to bottom.")
    else:
        print("Colab dependency available: ortools")
else:
    print("Dependency bootstrap skipped outside interactive Colab.")


## 01. Binary linear algebra over F_2

In [20]:
# @title GF(2) utilities (exact, uint8/binary)
import numpy as np


def as_binary_uint8(x, *, name="array", copy=True) -> np.ndarray:
    """Return `x` as a uint8 NumPy array containing only 0/1."""
    arr = np.array(x, copy=True) if copy else np.asarray(x)

    if arr.dtype == np.bool_:
        return arr.astype(np.uint8, copy=copy)

    if not np.issubdtype(arr.dtype, np.integer):
        raise TypeError(f"{name} must have boolean or integer dtype, got {arr.dtype!r}")

    mask = (arr == 0) | (arr == 1)
    if arr.size and not mask.all():
        bad = np.unique(arr[~mask])
        raise ValueError(f"{name} must be binary (0/1) only; found values {bad.tolist()}")

    return arr.astype(np.uint8, copy=copy)


def hamming_weight(x) -> int:
    arr = as_binary_uint8(x, name="x", copy=False)
    return int(arr.sum(dtype=np.int64))


def gf2_matvec(A, u) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u, name="u", copy=False)

    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")
    if u_bin.ndim != 1:
        raise ValueError(f"u must be 1D, got shape {u_bin.shape}")
    if A_bin.shape[1] != u_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, u is {u_bin.shape}")

    y = (A_bin.astype(np.int64) @ u_bin.astype(np.int64)) & 1
    return y.astype(np.uint8)


def gf2_matmul(A, B) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    B_bin = as_binary_uint8(B, name="B", copy=False)

    if A_bin.ndim != 2 or B_bin.ndim != 2:
        raise ValueError(f"A and B must be 2D, got {A_bin.shape} and {B_bin.shape}")
    if A_bin.shape[1] != B_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, B is {B_bin.shape}")

    C = (A_bin.astype(np.int64) @ B_bin.astype(np.int64)) & 1
    return C.astype(np.uint8)


def gf2_rank(A) -> int:
    M = as_binary_uint8(A, name="A", copy=True)
    if M.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {M.shape}")

    rows, cols = M.shape
    r = 0
    for c in range(cols):
        pivot = None
        for i in range(r, rows):
            if M[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            continue

        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]

        for i in range(r + 1, rows):
            if M[i, c] == 1:
                M[i, :] ^= M[r, :]

        r += 1
        if r == rows:
            break

    return int(r)


def gf2_random_matrix(rows, cols, rng=None, density=0.5) -> np.ndarray:
    if rows < 0 or cols < 0:
        raise ValueError("rows and cols must be nonnegative")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng() if rng is None else rng
    A = (rng.random((rows, cols)) < density).astype(np.uint8)
    return as_binary_uint8(A, name="A", copy=False)


def gf2_random_invertible(n, rng=None, max_tries=1000) -> np.ndarray:
    if n < 0:
        raise ValueError("n must be nonnegative")
    if max_tries <= 0:
        raise ValueError("max_tries must be positive")

    rng = np.random.default_rng() if rng is None else rng

    for _ in range(max_tries):
        Q = gf2_random_matrix(n, n, rng=rng, density=0.5)
        if gf2_rank(Q) == n:
            return Q

    raise RuntimeError(f"failed to sample invertible {n}x{n} matrix within {max_tries} tries")


def gf2_inverse(A) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=True)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")

    n, m = A_bin.shape
    if n != m:
        raise ValueError(f"A must be square, got shape {A_bin.shape}")

    I = np.eye(n, dtype=np.uint8)
    aug = np.concatenate([A_bin, I], axis=1)

    for c in range(n):
        pivot = None
        for i in range(c, n):
            if aug[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            raise ValueError("A is singular over F_2")

        if pivot != c:
            aug[[c, pivot]] = aug[[pivot, c]]

        for i in range(n):
            if i != c and aug[i, c] == 1:
                aug[i, :] ^= aug[c, :]

    inv = aug[:, n:]
    return as_binary_uint8(inv, name="A_inv", copy=False)


def support_intersection(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return int(np.bitwise_and(c_bin, d_bin).sum(dtype=np.int64))


def directional_delta(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return hamming_weight(d_bin) - 2 * support_intersection(c_bin, d_bin)


def run_f2_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # Binary/uint8 conversion checks
    v = as_binary_uint8([0, 1, 1, 0], name="v")
    assert v.dtype == np.uint8 and np.isin(v, [0, 1]).all()
    b = as_binary_uint8(np.array([True, False, True]), name="b")
    assert b.dtype == np.uint8 and np.array_equal(b, np.array([1, 0, 1], dtype=np.uint8))

    try:
        as_binary_uint8(np.array([0, 2], dtype=np.int64), name="bad")
        raise AssertionError("expected ValueError for non-binary integers")
    except ValueError:
        pass

    try:
        as_binary_uint8(np.array([0.0, 1.0]), name="float_bad")
        raise AssertionError("expected TypeError for non-integer dtype")
    except TypeError:
        pass


    for overflow_bad in [
        np.array([256], dtype=np.int64),
        np.array([-256], dtype=np.int64),
        np.array([0, 1, 257], dtype=np.int64),
        np.array([256], dtype=np.uint64),
    ]:
        try:
            as_binary_uint8(overflow_bad, name="overflow_bad")
            raise AssertionError("expected ValueError for overflow-prone non-binary integers")
        except ValueError:
            pass

    # Edge vectors
    z = as_binary_uint8(np.zeros(6, dtype=np.uint8), name="z")
    oh = as_binary_uint8(np.array([0, 0, 1, 0, 0, 0], dtype=np.uint8), name="oh")
    ones = as_binary_uint8(np.ones(6, dtype=np.uint8), name="ones")
    assert hamming_weight([0, 1, 1, 0]) == 2
    y_list = gf2_matvec([[1, 0], [1, 1]], [1, 1])
    assert y_list.dtype == np.uint8 and np.array_equal(y_list, np.array([1, 0], dtype=np.uint8))

    assert hamming_weight(z) == 0
    assert hamming_weight(oh) == 1
    assert hamming_weight(ones) == 6

    # Matvec: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 7))
        r = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        u = gf2_random_matrix(r, 1, rng=rng, density=0.5).reshape(-1)
        y = gf2_matvec(A, u)

        y_ref = np.zeros(n, dtype=np.uint8)
        for i in range(n):
            s = 0
            for j in range(r):
                s ^= int(A[i, j] & u[j])
            y_ref[i] = s

        assert y.dtype == np.uint8 and y.shape == (n,)
        assert np.array_equal(y, y_ref)

    # Matmul: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 6))
        r = int(rng.integers(1, 6))
        k = int(rng.integers(1, 6))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        B = gf2_random_matrix(r, k, rng=rng, density=0.5)
        C = gf2_matmul(A, B)

        C_ref = np.zeros((n, k), dtype=np.uint8)
        for i in range(n):
            for j in range(k):
                s = 0
                for t in range(r):
                    s ^= int(A[i, t] & B[t, j])
                C_ref[i, j] = s

        assert C.dtype == np.uint8 and C.shape == (n, k)
        assert np.array_equal(C, C_ref)

    # Rank edge cases
    Z = np.zeros((4, 5), dtype=np.uint8)
    assert gf2_rank(Z) == 0

    I5 = np.eye(5, dtype=np.uint8)
    assert gf2_rank(I5) == 5

    D = np.array([
        [1, 0, 1, 1],
        [1, 0, 1, 1],
        [0, 1, 1, 0],
    ], dtype=np.uint8)
    assert gf2_rank(D) == 2

    for _ in range(20):
        n = int(rng.integers(1, 7))
        m = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, m, rng=rng, density=0.5)
        rnk = gf2_rank(A)
        assert isinstance(rnk, int)
        assert 0 <= rnk <= min(n, m)

    # Small rectangular matrices
    A_rect = gf2_random_matrix(2, 5, rng=rng)
    B_rect = gf2_random_matrix(5, 3, rng=rng)
    C_rect = gf2_matmul(A_rect, B_rect)
    assert C_rect.shape == (2, 3)
    assert C_rect.dtype == np.uint8 and np.isin(C_rect, [0, 1]).all()

    # Invertible sampling and inverse checks
    for n in [1, 2, 3, 5]:
        Q = gf2_random_invertible(n, rng=rng, max_tries=2000)
        Q_inv = gf2_inverse(Q)
        I = np.eye(n, dtype=np.uint8)
        assert np.array_equal(gf2_matmul(Q, Q_inv), I)
        assert np.array_equal(gf2_matmul(Q_inv, Q), I)

    # singular / non-square inverse errors
    try:
        gf2_inverse(np.array([[1, 0, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for non-square matrix")
    except ValueError:
        pass

    try:
        gf2_inverse(np.array([[1, 1], [1, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for singular matrix")
    except ValueError:
        pass

    # Directional delta identities
    for _ in range(40):
        n = int(rng.integers(1, 20))
        c = gf2_random_matrix(n, 1, rng=rng).reshape(-1)
        d = gf2_random_matrix(n, 1, rng=rng).reshape(-1)

        delta = directional_delta(c, d)
        lhs1 = hamming_weight(c ^ d) - hamming_weight(c)
        lhs2 = hamming_weight(d) - 2 * support_intersection(c, d)

        assert delta == lhs1
        assert delta == lhs2

    # explicit edge cases for directional_delta
    assert directional_delta(z, z) == 0
    assert directional_delta(z, oh) == 1
    assert directional_delta(ones, ones) == -6

    print("run_f2_self_tests: all tests passed")

In [21]:
run_f2_self_tests()

run_f2_self_tests: all tests passed


## 02. Planted span-instance generator

In [22]:
# @title Planted span-instance generator (exact, uint8/binary)
from dataclasses import dataclass


@dataclass
class SpanInstance:
    A: np.ndarray
    c_star: np.ndarray
    u_star: np.ndarray
    W: int
    metadata: dict

    def matvec(self, u) -> np.ndarray:
        return gf2_matvec(self.A, u)

    def weight(self, c) -> int:
        return hamming_weight(c)

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        c_star = as_binary_uint8(self.c_star, name="c_star", copy=False)
        u_star = as_binary_uint8(self.u_star, name="u_star", copy=False)

        if A.ndim != 2:
            raise ValueError(f"A must be 2D, got shape {A.shape}")
        N, r = A.shape

        if c_star.ndim != 1 or c_star.shape != (N,):
            raise ValueError(f"c_star must have shape ({N},), got {c_star.shape}")
        if u_star.ndim != 1 or u_star.shape != (r,):
            raise ValueError(f"u_star must have shape ({r},), got {u_star.shape}")

        if hamming_weight(u_star) == 0:
            raise ValueError("u_star must be nonzero")
        if int(self.W) != hamming_weight(c_star):
            raise ValueError("W must equal hamming_weight(c_star)")

        if not np.array_equal(gf2_matvec(A, u_star), c_star):
            raise ValueError("constraint violated: A u_star != c_star")


def generate_planted_span_instance(
    N: int,
    r: int,
    W: int,
    seed=None,
    density: float = 0.5,
    hide: bool = True,
) -> SpanInstance:
    if N <= 0:
        raise ValueError("N must be positive")
    if r <= 0:
        raise ValueError("r must be positive")
    if not (1 <= W <= N):
        raise ValueError("W must satisfy 1 <= W <= N")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng(seed)

    support = rng.choice(N, size=W, replace=False)
    c_star = np.zeros(N, dtype=np.uint8)
    c_star[support] = 1

    u_star = np.zeros(r, dtype=np.uint8)
    while hamming_weight(u_star) == 0:
        u_star = (rng.random(r) < 0.5).astype(np.uint8)

    one_positions = np.flatnonzero(u_star)
    k = int(rng.choice(one_positions))

    A = gf2_random_matrix(N, r, rng=rng, density=density)

    parity = np.zeros(N, dtype=np.uint8)
    for j in one_positions:
        if j != k:
            parity ^= A[:, j]
    A[:, k] = c_star ^ parity

    if hide:
        Q = gf2_random_invertible(r, rng=rng)
        Q_inv = gf2_inverse(Q)
        A = gf2_matmul(A, Q)
        u_star = gf2_matvec(Q_inv, u_star)

    if not np.array_equal(gf2_matvec(A, u_star), c_star):
        raise RuntimeError("internal construction error: A u_star != c_star")

    metadata = {
        "N": int(N),
        "r": int(r),
        "W": int(W),
        "seed": seed,
        "density": float(density),
        "hide": bool(hide),
    }
    inst = SpanInstance(A=A, c_star=c_star, u_star=u_star, W=int(W), metadata=metadata)
    inst.verify()
    return inst


def run_planted_instance_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    for hide in [False, True]:
        for _ in range(8):
            N = int(rng.integers(4, 13))
            r = int(rng.integers(2, 11))
            W = int(rng.integers(1, N + 1))
            density = float(rng.uniform(0.0, 1.0))
            s = int(rng.integers(0, 10**9))

            inst = generate_planted_span_instance(N=N, r=r, W=W, seed=s, density=density, hide=hide)
            inst.verify()

            assert inst.weight(inst.c_star) == W
            assert np.array_equal(inst.matvec(inst.u_star), inst.c_star)

            for arr in [inst.A, inst.c_star, inst.u_star]:
                assert arr.dtype == np.uint8
                assert np.isin(arr, [0, 1]).all()

    params = dict(N=10, r=6, W=3, seed=12345, density=0.4, hide=True)
    a = generate_planted_span_instance(**params)
    b = generate_planted_span_instance(**params)
    assert np.array_equal(a.A, b.A)
    assert np.array_equal(a.c_star, b.c_star)
    assert np.array_equal(a.u_star, b.u_star)

    invalid_cases = [
        dict(N=10, r=5, W=0, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=11, seed=0, density=0.5, hide=False),
        dict(N=0, r=5, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=0, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=-0.1, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=1.1, hide=False),
    ]
    for kwargs in invalid_cases:
        try:
            generate_planted_span_instance(**kwargs)
            raise AssertionError(f"expected ValueError for invalid kwargs={kwargs}")
        except ValueError:
            pass

    print("run_planted_instance_self_tests: all tests passed")

In [23]:
run_planted_instance_self_tests()

run_planted_instance_self_tests: all tests passed


## 03. Direction bank

In [24]:
# @title Direction bank (exact span directions)

class DirectionBank:
    def __init__(self, A):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got ndim={A_bin.ndim}")

        N, r = A_bin.shape
        self.A = A_bin
        self.V = np.zeros((0, r), dtype=np.uint8)
        self.D = np.zeros((0, N), dtype=np.uint8)
        self.weights = np.zeros((0,), dtype=np.int64)
        self.hashes = set()
        self.usage_counts = np.zeros((0,), dtype=np.int64)
        self.ages = np.zeros((0,), dtype=np.int64)

    def _direction_key(self, d: np.ndarray) -> bytes:
        return np.ascontiguousarray(d, dtype=np.uint8).tobytes()

    def add_one(self, v):
        v_bin = as_binary_uint8(v, name="v", copy=False)
        if v_bin.ndim != 1 or v_bin.shape[0] != self.A.shape[1]:
            raise ValueError(f"v must have shape [{self.A.shape[1]}], got {v_bin.shape}")

        d = gf2_matvec(self.A, v_bin)
        if hamming_weight(d) == 0:
            return None

        key = self._direction_key(d)
        if key in self.hashes:
            return None

        idx = self.D.shape[0]
        self.V = np.vstack([self.V, v_bin.reshape(1, -1)])
        self.D = np.vstack([self.D, d.reshape(1, -1)])
        self.weights = np.append(self.weights, hamming_weight(d))
        self.usage_counts = np.append(self.usage_counts, 0)
        self.ages = np.append(self.ages, 0)
        self.hashes.add(key)
        return idx

    def add_coefficients(self, V_new):
        Vb = as_binary_uint8(V_new, name="V_new", copy=False)
        if Vb.ndim == 1:
            rows = [Vb]
        elif Vb.ndim == 2 and Vb.shape[1] == self.A.shape[1]:
            rows = [Vb[i] for i in range(Vb.shape[0])]
        else:
            raise ValueError(f"V_new must have shape [{self.A.shape[1]}] or [K, {self.A.shape[1]}], got {Vb.shape}")

        accepted = []
        for row in rows:
            idx = self.add_one(row)
            if idx is not None:
                accepted.append(idx)
        return accepted

    def add_random_span_directions(self, K, rng, max_attempts=None):
        if K < 0:
            raise ValueError("K must be nonnegative")
        if max_attempts is None:
            max_attempts = max(100, 20 * K)

        accepted = 0
        attempts = 0
        r = self.A.shape[1]
        while accepted < K and attempts < max_attempts:
            v = rng.integers(0, 2, size=(r,), dtype=np.uint8)
            idx = self.add_one(v)
            if idx is not None:
                accepted += 1
            attempts += 1
        return accepted

    def deltas(self, c) -> np.ndarray:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        if c_bin.ndim != 1 or c_bin.shape[0] != self.A.shape[0]:
            raise ValueError(f"c must have shape [{self.A.shape[0]}], got {c_bin.shape}")

        M = self.D.shape[0]
        if M == 0:
            return np.zeros((0,), dtype=np.int64)

        intersections = np.sum(self.D & c_bin.reshape(1, -1), axis=1, dtype=np.int64)
        return self.weights.astype(np.int64) - 2 * intersections

    def best_descent(self, c) -> Optional[int]:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        deltas = self.deltas(c_bin)
        if deltas.size == 0:
            return None

        candidates = np.where(deltas < 0)[0]
        if candidates.size == 0:
            return None

        for idx in candidates[np.argsort(deltas[candidates])]:
            c_new = c_bin ^ self.D[idx]
            if hamming_weight(c_new) != 0:
                return int(idx)
        return None

    def features_for_ranker(self, c) -> np.ndarray:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        M = self.D.shape[0]
        if M == 0:
            return np.zeros((0, 6), dtype=np.float32)

        N = float(self.A.shape[0])
        intersections = np.sum(self.D & c_bin.reshape(1, -1), axis=1, dtype=np.int64)
        deltas = self.weights.astype(np.int64) - 2 * intersections

        age_denom = max(1.0, float(np.max(self.ages)) if self.ages.size else 1.0)
        use_denom = max(1.0, float(np.max(self.usage_counts)) if self.usage_counts.size else 1.0)

        feats = np.stack([
            self.weights / N,
            intersections / N,
            deltas / N,
            (deltas < 0).astype(np.float32),
            self.ages / age_denom,
            self.usage_counts / use_denom,
        ], axis=1)
        return feats.astype(np.float32)

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        if A.ndim != 2:
            raise ValueError("A must be 2D")
        N, r = A.shape
        M = self.D.shape[0]

        if self.V.shape != (M, r):
            raise ValueError(f"V shape mismatch: expected {(M, r)}, got {self.V.shape}")
        if self.D.shape != (M, N):
            raise ValueError(f"D shape mismatch: expected {(M, N)}, got {self.D.shape}")

        _ = as_binary_uint8(self.V, name="V", copy=False)
        _ = as_binary_uint8(self.D, name="D", copy=False)

        if len(self.weights) != M or len(self.usage_counts) != M or len(self.ages) != M:
            raise ValueError("weights/usage_counts/ages length mismatch")

        if len(self.hashes) != M:
            raise ValueError("hash set size mismatch")

        rebuilt_hashes = set()
        for i in range(M):
            d = gf2_matvec(A, self.V[i])
            if not np.array_equal(d, self.D[i]):
                raise ValueError(f"direction mismatch at index {i}")
            w = hamming_weight(self.D[i])
            if w == 0:
                raise ValueError(f"zero direction at index {i}")
            if int(self.weights[i]) != w:
                raise ValueError(f"weight mismatch at index {i}")
            key = self._direction_key(self.D[i])
            if key in rebuilt_hashes:
                raise ValueError("duplicate direction hash")
            rebuilt_hashes.add(key)

        if rebuilt_hashes != self.hashes:
            raise ValueError("stored hashes do not match directions")


def run_direction_bank_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    randvec = lambda n: rng.integers(0, 2, size=(n,), dtype=np.uint8)
    A = gf2_random_matrix(12, 8, rng=rng)
    bank = DirectionBank(A)

    accepted = bank.add_random_span_directions(20, rng=rng)
    assert accepted >= 1
    bank.verify()

    for i in range(bank.D.shape[0]):
        assert np.array_equal(bank.D[i], gf2_matvec(bank.A, bank.V[i]))
        assert hamming_weight(bank.D[i]) > 0

    v = bank.V[0].copy()
    M_before = bank.D.shape[0]
    assert bank.add_one(v) is None
    assert bank.D.shape[0] == M_before

    z_found = None
    for _ in range(256):
        z = randvec(A.shape[1])
        if hamming_weight(z) > 0 and hamming_weight(gf2_matvec(A, z)) == 0:
            z_found = z
            break
    if z_found is not None:
        assert bank.add_one(v ^ z_found) is None

    empty = DirectionBank(A)
    c0 = randvec(A.shape[0])
    assert empty.deltas(c0).shape == (0,)
    assert empty.best_descent(c0) is None

    c = randvec(A.shape[0])
    deltas = bank.deltas(c)
    for i in range(bank.D.shape[0]):
        assert int(deltas[i]) == directional_delta(c, bank.D[i])

    feats = bank.features_for_ranker(c)
    assert feats.shape[0] == bank.D.shape[0]
    assert feats.shape[1] >= 4
    assert np.isfinite(feats).all()


run_direction_bank_self_tests()

## 04. Exact discrete-gradient descent

In [25]:
# @title Exact discrete-gradient descent

class GradientEngine:
    def delta(self, c, d) -> int:
        c_bin = as_binary_uint8(c, name="c", copy=False)
        d_bin = as_binary_uint8(d, name="d", copy=False)
        if c_bin.shape != d_bin.shape:
            raise ValueError(f"c and d must share shape, got {c_bin.shape} vs {d_bin.shape}")

        delta_1 = directional_delta(c_bin, d_bin)
        delta_2 = hamming_weight(c_bin ^ d_bin) - hamming_weight(c_bin)
        if int(delta_1) != int(delta_2):
            raise ValueError("delta formula mismatch")
        return int(delta_1)

    def apply_best(self, c, bank) -> tuple[np.ndarray, dict]:
        c_bin = as_binary_uint8(c, name="c", copy=True)
        if c_bin.ndim != 1 or c_bin.shape[0] != bank.A.shape[0]:
            raise ValueError(f"c must have shape [{bank.A.shape[0]}], got {c_bin.shape}")

        idx = bank.best_descent(c_bin)
        if idx is None:
            return c_bin.copy(), {"moved": False, "reason": "no_descent"}

        d = bank.D[idx]
        delta = self.delta(c_bin, d)
        old_weight = hamming_weight(c_bin)
        c_new = c_bin ^ d
        new_weight = hamming_weight(c_new)

        if new_weight == 0:
            raise ValueError("invalid move: moved to zero")
        if new_weight >= old_weight or delta >= 0:
            raise ValueError("invalid move: weight did not strictly decrease")

        intersection = int(np.sum(c_bin & d))
        bank.usage_counts[idx] += 1

        meta = {
            "moved": True,
            "idx": int(idx),
            "delta": int(delta),
            "old_weight": int(old_weight),
            "new_weight": int(new_weight),
            "direction_weight": int(bank.weights[idx]),
            "intersection": intersection,
        }
        return c_new, meta

    def descend(self, c, bank, max_steps) -> tuple[np.ndarray, list[dict]]:
        if max_steps < 0:
            raise ValueError("max_steps must be nonnegative")
        current = as_binary_uint8(c, name="c", copy=True)
        history = []

        for _ in range(max_steps):
            nxt, meta = self.apply_best(current, bank)
            history.append(meta)
            if not meta.get("moved", False):
                break
            if hamming_weight(nxt) >= hamming_weight(current):
                raise ValueError("descend step failed to strictly decrease")
            bank.ages += 1
            bank.ages[meta["idx"]] = 0
            current = nxt
        return current, history


def run_gradient_engine_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    randvec = lambda n: rng.integers(0, 2, size=(n,), dtype=np.uint8)
    engine = GradientEngine()

    c = randvec(20)
    d = randvec(20)
    delta = engine.delta(c, d)
    assert delta == hamming_weight(c ^ d) - hamming_weight(c)
    assert delta == hamming_weight(d) - 2 * int(np.sum(c & d))

    A = np.eye(4, dtype=np.uint8)
    bank = DirectionBank(A)
    c_zero_block = np.array([1, 1, 0, 0], dtype=np.uint8)
    bank.add_one(c_zero_block)
    c2, m2 = engine.apply_best(c_zero_block, bank)
    assert np.array_equal(c2, c_zero_block)
    assert m2["moved"] is False and m2["reason"] == "no_descent"

    A3 = gf2_random_matrix(16, 10, rng=rng)
    bank3 = DirectionBank(A3)
    bank3.add_random_span_directions(40, rng=rng)
    u = randvec(A3.shape[1])
    c3 = gf2_matvec(A3, u)
    if hamming_weight(c3) == 0:
        for _ in range(256):
            u = randvec(A3.shape[1])
            c3 = gf2_matvec(A3, u)
            if hamming_weight(c3) > 0:
                break


    c_next, meta = engine.apply_best(c3, bank3)
    if meta["moved"]:
        assert hamming_weight(c_next) < hamming_weight(c3)

    u0 = randvec(A3.shape[1])
    v0 = randvec(A3.shape[1])
    left = gf2_matvec(A3, u0) ^ gf2_matvec(A3, v0)
    right = gf2_matvec(A3, u0 ^ v0)
    assert np.array_equal(left, right)

    final_c, hist = engine.descend(c3, bank3, max_steps=25)
    prev_w = hamming_weight(c3)
    for h in hist:
        if h.get("moved", False):
            assert h["new_weight"] < h["old_weight"]
            assert h["old_weight"] <= prev_w
            prev_w = h["new_weight"]
    assert hamming_weight(final_c) <= hamming_weight(c3)


run_gradient_engine_self_tests()

## 05. Local-minimum intersection solver

In [26]:
# @title Local-minimum intersection solver (exact CP-SAT feasibility)
try:
    from ortools.sat.python import cp_model
    ORTOOLS_AVAILABLE = True
except Exception as e:
    cp_model = None
    ORTOOLS_AVAILABLE = False
    ORTOOLS_IMPORT_ERROR = e


@dataclass
class SolverResult:
    status: str
    found: bool
    u: Optional[np.ndarray]
    c: Optional[np.ndarray]
    weight: Optional[int]
    runtime_s: float
    num_variables: int
    num_constraints: int
    num_directions: int
    metadata: dict


class LocalMinimumSolver:
    def __init__(
        self,
        num_search_workers: int = 1,
        random_seed: int = 0,
        log_search_progress: bool = False,
    ):
        self.num_search_workers = int(num_search_workers)
        self.random_seed = int(random_seed)
        self.log_search_progress = bool(log_search_progress)

    def solve(self, A, W, direction_bank, time_limit_s: float = 10.0) -> SolverResult:
        if not ORTOOLS_AVAILABLE:
            raise RuntimeError(
                f"OR-Tools unavailable; cannot run LocalMinimumSolver.solve. Import error: {ORTOOLS_IMPORT_ERROR!r}"
            )

        A_bin = as_binary_uint8(A, name='A', copy=False)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError(f"A must have positive shape [N, r], got {A_bin.shape}")

        if int(W) != W:
            raise ValueError('W must be an integer')
        W = int(W)
        if not (1 <= W <= N):
            raise ValueError(f'W must satisfy 1 <= W <= N={N}, got {W}')

        direction_bank.verify()
        if direction_bank.A.shape != A_bin.shape or not np.array_equal(direction_bank.A, A_bin):
            raise ValueError('direction_bank.A must match A exactly')

        D = direction_bank.D
        weights = direction_bank.weights.astype(np.int64, copy=False)
        num_directions = int(D.shape[0])

        model = cp_model.CpModel()
        num_constraints = 0

        u_vars = [model.NewBoolVar(f'u_{j}') for j in range(r)]
        c_vars = [model.NewBoolVar(f'c_{i}') for i in range(N)]

        for i in range(N):
            S_i = np.flatnonzero(A_bin[i])
            if S_i.size == 0:
                model.Add(c_vars[i] == 0)
                num_constraints += 1
            else:
                model.AddBoolXOr([u_vars[int(j)] for j in S_i] + [c_vars[i].Not()])
                num_constraints += 1

        model.Add(sum(c_vars) >= 1)
        model.Add(sum(c_vars) <= W)
        num_constraints += 2

        for j in range(num_directions):
            supp = np.flatnonzero(D[j])
            rhs = int(weights[j] // 2)
            model.Add(sum(c_vars[int(i)] for i in supp) <= rhs)
            num_constraints += 1

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = float(time_limit_s)
        solver.parameters.num_search_workers = self.num_search_workers
        solver.parameters.random_seed = self.random_seed
        solver.parameters.log_search_progress = self.log_search_progress

        status_code = solver.Solve(model)
        status_name = solver.StatusName(status_code)
        runtime_s = float(solver.WallTime())

        if status_code in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            u = np.array([solver.Value(v) for v in u_vars], dtype=np.uint8)
            c = np.array([solver.Value(v) for v in c_vars], dtype=np.uint8)

            if not np.array_equal(c, gf2_matvec(A_bin, u)):
                raise ValueError('solver returned inconsistent assignment: c != A u over F_2')
            if hamming_weight(c) == 0:
                raise ValueError('solver returned zero codeword despite wt(c) >= 1 constraint')
            if hamming_weight(c) > W:
                raise ValueError('solver returned wt(c) > W')

            for j in range(num_directions):
                lhs = int(np.sum(c & D[j]))
                rhs = int(weights[j] // 2)
                if lhs > rhs:
                    raise ValueError(f'local-minimum inequality violated at direction {j}: {lhs} > {rhs}')

            return SolverResult(
                status=status_name,
                found=True,
                u=u,
                c=c,
                weight=hamming_weight(c),
                runtime_s=runtime_s,
                num_variables=int(r + N),
                num_constraints=num_constraints,
                num_directions=num_directions,
                metadata={'status_code': int(status_code), 'status_name': status_name},
            )

        return SolverResult(
            status=status_name,
            found=False,
            u=None,
            c=None,
            weight=None,
            runtime_s=runtime_s,
            num_variables=int(r + N),
            num_constraints=num_constraints,
            num_directions=num_directions,
            metadata={'status_code': int(status_code), 'status_name': status_name},
        )


def verify_solver_result(A, W, direction_bank, result) -> None:
    if not result.found:
        return

    A_bin = as_binary_uint8(A, name='A', copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f'A must have shape [N, r], got {A_bin.shape}')
    N, r = A_bin.shape

    c = as_binary_uint8(result.c, name='result.c', copy=False)
    u = as_binary_uint8(result.u, name='result.u', copy=False)

    if c.ndim != 1 or c.shape != (N,):
        raise ValueError(f'result.c must have shape ({N},), got {c.shape}')
    if u.ndim != 1 or u.shape != (r,):
        raise ValueError(f'result.u must have shape ({r},), got {u.shape}')

    if not np.array_equal(c, gf2_matvec(A_bin, u)):
        raise ValueError('verification failed: c != A u over F_2')
    if hamming_weight(c) == 0:
        raise ValueError('verification failed: c must be nonzero')
    if hamming_weight(c) > int(W):
        raise ValueError('verification failed: wt(c) > W')

    direction_bank.verify()
    if direction_bank.A.shape != A_bin.shape or not np.array_equal(direction_bank.A, A_bin):
        raise ValueError('verification failed: direction_bank.A must match A exactly')

    for j in range(direction_bank.D.shape[0]):
        lhs = int(np.sum(c & direction_bank.D[j]))
        rhs = int(direction_bank.weights[j] // 2)
        if lhs > rhs:
            raise ValueError(f'verification failed at direction {j}: {lhs} > {rhs}')


def run_local_minimum_solver_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    inst = generate_planted_span_instance(N=20, r=8, W=3, seed=int(rng.integers(0, 10**9)), hide=True)
    empty_bank = DirectionBank(inst.A)
    solver = LocalMinimumSolver(num_search_workers=1, random_seed=seed)
    result = solver.solve(inst.A, W=inst.W, direction_bank=empty_bank, time_limit_s=10.0)
    assert result.found is True
    verify_solver_result(inst.A, inst.W, empty_bank, result)
    assert result.weight <= inst.W

    A4 = np.eye(4, dtype=np.uint8)
    W4 = 1
    bank4 = DirectionBank(A4)
    bank4.add_one(np.array([1, 0, 0, 0], dtype=np.uint8))
    bank4.add_one(np.array([0, 1, 0, 0], dtype=np.uint8))
    bank4.add_one(np.array([0, 0, 1, 0], dtype=np.uint8))
    result4 = solver.solve(A4, W=W4, direction_bank=bank4, time_limit_s=10.0)
    assert result4.found is True
    verify_solver_result(A4, W4, bank4, result4)
    assert np.array_equal(result4.c, np.array([0, 0, 0, 1], dtype=np.uint8))

    A3 = np.eye(3, dtype=np.uint8)
    W3 = 1
    bank3 = DirectionBank(A3)
    bank3.add_one(np.array([1, 0, 0], dtype=np.uint8))
    bank3.add_one(np.array([0, 1, 0], dtype=np.uint8))
    bank3.add_one(np.array([0, 0, 1], dtype=np.uint8))
    result3 = solver.solve(A3, W=W3, direction_bank=bank3, time_limit_s=10.0)
    assert result3.found is False
    assert result3.status in ('INFEASIBLE', 'MODEL_INVALID', 'UNKNOWN')

    sanity = result4
    assert np.array_equal(gf2_matvec(A4, sanity.u), sanity.c)
    assert 1 <= hamming_weight(sanity.c) <= W4
    for j in range(bank4.D.shape[0]):
        assert int(np.sum(sanity.c & bank4.D[j])) <= int(bank4.weights[j] // 2)

    try:
        solver.solve(A4, W=0, direction_bank=bank4, time_limit_s=1.0)
        raise AssertionError('expected ValueError for W=0')
    except ValueError:
        pass

    try:
        solver.solve(A4, W=5, direction_bank=bank4, time_limit_s=1.0)
        raise AssertionError('expected ValueError for W>N')
    except ValueError:
        pass

    try:
        solver.solve(np.array([1, 0, 1], dtype=np.uint8), W=1, direction_bank=bank3, time_limit_s=1.0)
        raise AssertionError('expected ValueError for non-2D A')
    except ValueError:
        pass

    bad_bank = DirectionBank(np.eye(5, dtype=np.uint8))
    try:
        solver.solve(A4, W=1, direction_bank=bad_bank, time_limit_s=1.0)
        raise AssertionError('expected ValueError for direction-bank mismatch')
    except ValueError:
        pass

    print('LocalMinimumSolver self-tests passed.')


In [27]:
if ORTOOLS_AVAILABLE:
    run_local_minimum_solver_self_tests()
else:
    msg = (
        "OR-Tools unavailable; cannot run local-minimum solver self-tests. "
        "Install with: %pip install ortools"
    )
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


LocalMinimumSolver self-tests passed.


In [28]:
inst = generate_planted_span_instance(N=20, r=8, W=3, seed=0, hide=True)
bank = DirectionBank(inst.A)

solver = LocalMinimumSolver(num_search_workers=1, random_seed=0)
result = solver.solve(inst.A, inst.W, bank, time_limit_s=10)

print(result.status, result.found, result.weight)
verify_solver_result(inst.A, inst.W, bank, result)

OPTIMAL True 3


## 06. Failed-local-minima archive and attack utilities

In [ ]:

# @title Failed-local-minima archive and exact attack utilities
from dataclasses import field
from typing import List
import hashlib


def vector_hash(x) -> str:
    xb = as_binary_uint8(x, name="x", copy=False)
    if xb.ndim != 1:
        raise ValueError(f"x must be 1D, got {xb.shape}")
    return hashlib.sha256(np.ascontiguousarray(xb, dtype=np.uint8).tobytes()).hexdigest()


def bank_contains_direction(direction_bank, d) -> bool:
    db = as_binary_uint8(d, name="d", copy=False)
    if db.ndim != 1 or db.shape[0] != direction_bank.A.shape[0]:
        raise ValueError(f"d must have shape [{direction_bank.A.shape[0]}], got {db.shape}")
    key = np.ascontiguousarray(db, dtype=np.uint8).tobytes()
    return key in direction_bank.hashes


def descent_margin(c, d) -> int:
    cb = as_binary_uint8(c, name="c", copy=False)
    db = as_binary_uint8(d, name="d", copy=False)
    return int(2 * support_intersection(cb, db) - hamming_weight(db))


def is_descent_direction(c, d) -> bool:
    cb = as_binary_uint8(c, name="c", copy=False)
    db = as_binary_uint8(d, name="d", copy=False)
    return bool(directional_delta(cb, db) < 0 and hamming_weight(cb ^ db) != 0)


def local_minimum_report(c, direction_bank) -> dict:
    cb = as_binary_uint8(c, name="c", copy=False)
    direction_bank.verify()
    deltas = direction_bank.deltas(cb)
    active = []
    near_active = []
    num_valid_neg = 0
    for j in range(direction_bank.D.shape[0]):
        d = direction_bank.D[j]
        inter = support_intersection(cb, d)
        half = hamming_weight(d) // 2
        gap = half - inter
        if gap == 0:
            active.append(j)
        if gap in (0, 1):
            near_active.append(j)
        if deltas[j] < 0 and hamming_weight(cb ^ d) != 0:
            num_valid_neg += 1
    return {
        "weight": hamming_weight(cb),
        "num_directions": int(direction_bank.D.shape[0]),
        "min_delta": int(deltas.min()) if deltas.size else 0,
        "num_negative_directions": int(np.sum(deltas < 0)),
        "num_valid_negative_directions": int(num_valid_neg),
        "is_local_minimum": bool(num_valid_neg == 0),
        "active_constraints": active,
        "near_active_constraints": near_active,
    }


@dataclass
class FailedMinimumEntry:
    u_bad: np.ndarray
    c_bad: np.ndarray
    weight: int
    active_constraints: List[int]
    near_active_constraints: List[int]
    timestamp: int
    origin_action: str
    hash: str
    metadata: dict = field(default_factory=dict)


class FailedMinimaArchive:
    def __init__(self, A, W):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError("A must have N>0 and r>0")
        W_int = int(W)
        if not (1 <= W_int <= N):
            raise ValueError("W must satisfy 1 <= W <= N")
        self.A = A_bin
        self.W = W_int
        self.entries = []
        self.hashes = set()
        self.timestamp_counter = 0

    def add(self, u_bad, c_bad, direction_bank, origin_action: str = "unknown", metadata: Optional[dict] = None) -> Optional[FailedMinimumEntry]:
        direction_bank.verify()
        if not np.array_equal(direction_bank.A, self.A):
            raise ValueError("direction_bank.A must match archive A")
        N, r = self.A.shape
        ub = as_binary_uint8(u_bad, name="u_bad", copy=True)
        cb = as_binary_uint8(c_bad, name="c_bad", copy=True)
        if ub.ndim != 1 or ub.shape != (r,):
            raise ValueError(f"u_bad must have shape ({r},), got {ub.shape}")
        if cb.ndim != 1 or cb.shape != (N,):
            raise ValueError(f"c_bad must have shape ({N},), got {cb.shape}")
        if not np.array_equal(gf2_matvec(self.A, ub), cb):
            raise ValueError("c_bad must equal gf2_matvec(A, u_bad)")
        wt = hamming_weight(cb)
        if wt == 0:
            raise ValueError("c_bad must be nonzero")
        if wt <= self.W:
            raise ValueError("c_bad must satisfy hamming_weight(c_bad) > W")
        report = local_minimum_report(cb, direction_bank)
        if not report["is_local_minimum"]:
            raise ValueError("state is not a failed local minimum under current bank")
        h = vector_hash(cb)
        if h in self.hashes:
            return None
        ent = FailedMinimumEntry(
            u_bad=ub,
            c_bad=cb,
            weight=wt,
            active_constraints=list(report["active_constraints"]),
            near_active_constraints=list(report["near_active_constraints"]),
            timestamp=int(self.timestamp_counter),
            origin_action=str(origin_action),
            hash=h,
            metadata={} if metadata is None else dict(metadata),
        )
        self.entries.append(ent)
        self.hashes.add(h)
        self.timestamp_counter += 1
        return ent

    def __len__(self) -> int:
        return len(self.entries)

    def get(self, idx: int) -> FailedMinimumEntry:
        return self.entries[idx]

    def latest(self) -> Optional[FailedMinimumEntry]:
        return None if len(self.entries) == 0 else self.entries[-1]

    def verify_entry(self, entry, direction_bank=None) -> None:
        N, r = self.A.shape
        ub = as_binary_uint8(entry.u_bad, name="entry.u_bad", copy=False)
        cb = as_binary_uint8(entry.c_bad, name="entry.c_bad", copy=False)
        if ub.ndim != 1 or ub.shape != (r,):
            raise ValueError(f"entry.u_bad must have shape ({r},), got {ub.shape}")
        if cb.ndim != 1 or cb.shape != (N,):
            raise ValueError(f"entry.c_bad must have shape ({N},), got {cb.shape}")
        if not np.array_equal(gf2_matvec(self.A, ub), cb):
            raise ValueError("entry must satisfy c_bad == gf2_matvec(A, u_bad)")
        if hamming_weight(cb) == 0:
            raise ValueError("entry.c_bad must be nonzero")
        if int(entry.weight) != hamming_weight(cb):
            raise ValueError("entry.weight must equal hamming_weight(c_bad)")
        if int(entry.weight) <= self.W:
            raise ValueError("entry.weight must be > W")
        if vector_hash(cb) != entry.hash:
            raise ValueError("entry hash mismatch")
        if direction_bank is not None:
            if not np.array_equal(direction_bank.A, self.A):
                raise ValueError("direction_bank.A must match archive A")
            report = local_minimum_report(cb, direction_bank)
            if not report["is_local_minimum"]:
                raise ValueError("entry is not a local minimum under provided bank")
            if list(entry.active_constraints) != list(report["active_constraints"]):
                raise ValueError("active constraints mismatch")
            if list(entry.near_active_constraints) != list(report["near_active_constraints"]):
                raise ValueError("near-active constraints mismatch")

    def verify(self, direction_bank=None) -> None:
        recomputed_hashes = set()
        for entry in self.entries:
            self.verify_entry(entry, direction_bank=direction_bank)
            h = vector_hash(entry.c_bad)
            if h in recomputed_hashes:
                raise ValueError("duplicate c_bad hash in entries")
            recomputed_hashes.add(h)
        if recomputed_hashes != self.hashes:
            raise ValueError("archive hash set mismatch")


def random_attack_failed_minimum(A, failed_entry, direction_bank, rng, num_trials: int = 1000) -> Optional[dict]:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got {A_bin.shape}")
    if num_trials < 0:
        raise ValueError("num_trials must be nonnegative")
    c_bad = as_binary_uint8(failed_entry.c_bad, name="failed_entry.c_bad", copy=False)
    if c_bad.shape != (A_bin.shape[0],):
        raise ValueError("failed entry incompatible with A")
    if not np.array_equal(direction_bank.A, A_bin):
        raise ValueError("direction_bank.A must equal A")
    direction_bank.verify()

    r = A_bin.shape[1]
    for t in range(1, int(num_trials) + 1):
        v = rng.integers(0, 2, size=(r,), dtype=np.uint8)
        d = gf2_matvec(A_bin, v)
        if hamming_weight(d) == 0:
            continue
        if bank_contains_direction(direction_bank, d):
            continue
        if is_descent_direction(c_bad, d):
            delta = directional_delta(c_bad, d)
            new_weight = hamming_weight(c_bad ^ d)
            return {
                "v": v,
                "d": d,
                "delta": int(delta),
                "margin": int(descent_margin(c_bad, d)),
                "new_weight": int(new_weight),
                "old_weight": int(hamming_weight(c_bad)),
                "trials": int(t),
            }
    return None


def planted_attack_direction(inst, u_current) -> dict:
    inst.verify()
    u_curr = as_binary_uint8(u_current, name="u_current", copy=False)
    if u_curr.ndim != 1 or u_curr.shape != inst.u_star.shape:
        raise ValueError(f"u_current must have shape {inst.u_star.shape}, got {u_curr.shape}")
    c_current = inst.matvec(u_curr)
    v_star = u_curr ^ inst.u_star
    d_star = inst.matvec(v_star)
    c_target = as_binary_uint8(inst.c_star, name="inst.c_star", copy=False)
    if not np.array_equal(c_current ^ d_star, c_target):
        raise ValueError("exactness failed: c_current XOR d_star must equal c_star")
    delta = directional_delta(c_current, d_star)
    margin = descent_margin(c_current, d_star)
    return {
        "v": v_star,
        "d": d_star,
        "c_current": c_current,
        "c_target": c_target,
        "delta": int(delta),
        "margin": int(margin),
        "old_weight": int(hamming_weight(c_current)),
        "target_weight": int(hamming_weight(c_target)),
        "valid_descent": bool(is_descent_direction(c_current, d_star)),
    }


def run_failed_minima_archive_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)
    # 1) explicit failed local minimum
    A = np.eye(4, dtype=np.uint8)
    W = 1
    c_bad = np.array([1, 1, 0, 0], dtype=np.uint8)
    u_bad = c_bad.copy()
    bank = DirectionBank(A)
    bank.add_one(np.array([0, 0, 1, 0], dtype=np.uint8))
    bank.add_one(np.array([0, 0, 0, 1], dtype=np.uint8))
    archive = FailedMinimaArchive(A, W)
    e = archive.add(u_bad, c_bad, bank, origin_action="unit-test")
    assert e is not None
    assert len(archive) == 1
    assert archive.add(u_bad, c_bad, bank) is None
    archive.verify(bank)

    # 2) reject non-local state
    bad_bank = DirectionBank(A)
    bad_bank.add_one(np.array([1, 0, 0, 0], dtype=np.uint8))
    archive2 = FailedMinimaArchive(A, W)
    try:
        archive2.add(u_bad, c_bad, bad_bank)
        raise AssertionError("expected ValueError for non-local state")
    except ValueError:
        pass

    # 3) reject weight <= W
    archive3 = FailedMinimaArchive(A, 2)
    try:
        archive3.add(u_bad, c_bad, bank)
        raise AssertionError("expected ValueError for weight <= W")
    except ValueError:
        pass

    # 4) zero-target descent invalid
    c = np.array([1, 1, 0, 0], dtype=np.uint8)
    d = c.copy()
    assert directional_delta(c, d) < 0
    assert is_descent_direction(c, d) is False

    # 5) random attack utility
    prop = random_attack_failed_minimum(A, e, bank, rng, num_trials=200)
    assert prop is not None
    d_prop = prop["d"]
    assert hamming_weight(d_prop) > 0
    assert not bank_contains_direction(bank, d_prop)
    assert np.array_equal(gf2_matvec(A, prop["v"]), d_prop)
    assert directional_delta(c_bad, d_prop) < 0
    assert hamming_weight(c_bad ^ d_prop) > 0

    # 6) planted attack direction
    inst = generate_planted_span_instance(N=8, r=5, W=2, seed=int(rng.integers(10**9)), hide=True)
    for _ in range(128):
        u_current = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
        out = planted_attack_direction(inst, u_current)
        if not np.array_equal(out["c_current"], inst.c_star) and out["old_weight"] > inst.W:
            break
    assert np.array_equal(out["c_current"] ^ out["d"], inst.c_star)
    assert np.array_equal(gf2_matvec(inst.A, out["v"]), out["d"])
    assert out["delta"] == out["target_weight"] - out["old_weight"]
    if out["old_weight"] > inst.W and out["target_weight"] > 0:
        assert out["valid_descent"] is True

    # 7) local minimum report consistency
    report = local_minimum_report(c_bad, bank)
    for key in [
        "weight", "num_directions", "min_delta", "num_negative_directions",
        "num_valid_negative_directions", "is_local_minimum", "active_constraints", "near_active_constraints"
    ]:
        assert key in report
    deltas = [directional_delta(c_bad, bank.D[j]) for j in range(bank.D.shape[0])]
    assert report["num_negative_directions"] == int(sum(int(dlt < 0) for dlt in deltas))
    valid_neg = 0
    for j in range(bank.D.shape[0]):
        if directional_delta(c_bad, bank.D[j]) < 0 and hamming_weight(c_bad ^ bank.D[j]) != 0:
            valid_neg += 1
    assert report["num_valid_negative_directions"] == valid_neg

    # descent margin identity
    for j in range(bank.D.shape[0]):
        assert descent_margin(c_bad, bank.D[j]) == -directional_delta(c_bad, bank.D[j])

    print("run_failed_minima_archive_self_tests: PASS")


In [ ]:
run_failed_minima_archive_self_tests()

## 07. Neural direction ranker

In [ ]:
# @title Neural direction ranker (Torch scoring over exact direction features)
from typing import Optional

if TORCH_AVAILABLE:
    import torch.nn as nn

    class DirectionRanker(nn.Module):
        def __init__(self, dir_feat_dim, global_feat_dim, hidden_dim=128):
            super().__init__()
            self.dir_mlp = nn.Sequential(
                nn.Linear(dir_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.global_mlp = nn.Sequential(
                nn.Linear(global_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.score = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, direction_features, global_features, mask=None):
            B, M, _ = direction_features.shape
            d_emb = self.dir_mlp(direction_features)
            g_emb = self.global_mlp(global_features).unsqueeze(1).expand(B, M, -1)
            scores = self.score(torch.cat([d_emb, g_emb], dim=-1)).squeeze(-1)
            if mask is not None:
                scores = scores.masked_fill(~mask.bool(), -1e9)
            return scores
else:
    class DirectionRanker:  # type: ignore[no-redef]
        def __init__(self, *args, **kwargs):
            raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")


def global_features_for_ranker(A, W, c, bank) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    c_bin = as_binary_uint8(c, name="c", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if c_bin.ndim != 1 or c_bin.shape[0] != N:
        raise ValueError(f"c must have shape [{N}], got {c_bin.shape}")

    M = int(bank.D.shape[0])
    wt_c = hamming_weight(c_bin)
    deltas = bank.deltas(c_bin) if M > 0 else np.zeros((0,), dtype=np.int64)
    min_delta = int(np.min(deltas)) if deltas.size else 0
    num_negative = int(np.sum(deltas < 0)) if deltas.size else 0
    frac_negative = float(num_negative / max(1, M))

    feats = np.array([
        float(N),
        float(r),
        float(W),
        float(W) / max(1.0, float(N)),
        float(r) / max(1.0, float(N)),
        float(M),
        float(M) / max(1.0, float(N)),
        float(wt_c),
        float(wt_c) / max(1.0, float(W)),
        float(wt_c) / max(1.0, float(N)),
        float(min_delta),
        float(num_negative),
        frac_negative,
    ], dtype=np.float32)
    return feats


def rank_directions_with_model(model, c, bank, global_features, device="cpu", mask=None) -> np.ndarray:
    if bank.D.shape[0] == 0:
        return np.zeros((0,), dtype=np.int64)
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    dir_features = bank.features_for_ranker(c).astype(np.float32, copy=False)
    global_features = np.asarray(global_features, dtype=np.float32)

    d_t = torch.from_numpy(dir_features).unsqueeze(0).to(device)
    g_t = torch.from_numpy(global_features).unsqueeze(0).to(device)
    m_t = None
    if mask is not None:
        m_t = torch.as_tensor(mask, device=device).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        scores = model(d_t, g_t, m_t).squeeze(0)
    ranking = torch.argsort(scores, descending=True)
    return ranking.detach().cpu().numpy().astype(np.int64)


def exact_best_direction_from_ranking(c, bank, ranking) -> Optional[int]:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    for idx in np.asarray(ranking, dtype=np.int64):
        d = bank.D[int(idx)]
        if directional_delta(c_bin, d) < 0 and hamming_weight(c_bin ^ d) != 0:
            return int(idx)
    return None


In [ ]:
# @title Direction ranker self-tests (small deterministic smoke checks)
def make_direction_ranker_toy_batch(seed=0):
    rng = np.random.default_rng(seed)
    inst = generate_planted_span_instance(N=24, r=10, W=5, seed=seed, hide=True)
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(K=20, rng=rng)

    u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    while hamming_weight(u) == 0:
        u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    c = gf2_matvec(inst.A, u)

    dfeat = bank.features_for_ranker(c).astype(np.float32, copy=False)
    gfeat = global_features_for_ranker(inst.A, inst.W, c, bank)
    deltas = bank.deltas(c).astype(np.float32, copy=False)
    target = (-deltas / max(1.0, float(inst.A.shape[0]))).astype(np.float32)
    return inst, bank, c, dfeat, gfeat, target


def run_direction_ranker_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    torch.manual_seed(seed)

    B, M, Fd, Fg = 2, 5, 6, 13
    model = DirectionRanker(Fd, Fg, hidden_dim=32)
    d = torch.randn(B, M, Fd)
    g = torch.randn(B, Fg)
    mask = torch.tensor([[1, 1, 0, 1, 0], [1, 0, 0, 1, 1]], dtype=torch.bool)
    scores = model(d, g, mask=mask)
    assert scores.shape == (B, M)
    assert torch.all(scores[~mask] <= -1e8)

    inst, bank, c, dfeat, gfeat, _ = make_direction_ranker_toy_batch(seed=seed)
    rk_model = DirectionRanker(dfeat.shape[1], gfeat.shape[0], hidden_dim=64)
    ranking = rank_directions_with_model(rk_model, c, bank, gfeat, device="cpu")
    assert ranking.shape == (bank.D.shape[0],)
    assert np.array_equal(np.sort(ranking), np.arange(bank.D.shape[0]))

    check_model = DirectionRanker(dfeat.shape[1], gfeat.shape[0], hidden_dim=32)
    forced = np.arange(bank.D.shape[0], dtype=np.int64)[::-1]
    idx = exact_best_direction_from_ranking(c, bank, forced)
    if idx is not None:
        assert directional_delta(c, bank.D[idx]) < 0
        assert hamming_weight(c ^ bank.D[idx]) != 0

    # Scores are only heuristic ranking signals; exact acceptance still requires delta<0 and nonzero next state.
    assert exact_best_direction_from_ranking(c, bank, ranking) is None or isinstance(
        exact_best_direction_from_ranking(c, bank, ranking), int
    )

    inst2, bank2, c2, dfeat2, gfeat2, target2 = make_direction_ranker_toy_batch(seed=seed + 1)
    train_model = DirectionRanker(dfeat2.shape[1], gfeat2.shape[0], hidden_dim=64)
    opt = torch.optim.Adam(train_model.parameters(), lr=1e-2)
    d_t = torch.from_numpy(dfeat2).unsqueeze(0)
    g_t = torch.from_numpy(gfeat2).unsqueeze(0)
    t_t = torch.from_numpy(target2).unsqueeze(0)

    train_model.train()
    with torch.no_grad():
        init_loss = torch.mean((train_model(d_t, g_t) - t_t) ** 2).item()
    for _ in range(30):
        opt.zero_grad()
        pred = train_model(d_t, g_t)
        loss = torch.mean((pred - t_t) ** 2)
        loss.backward()
        opt.step()
    final_loss = float(loss.item())
    assert final_loss < init_loss, (init_loss, final_loss)


if TORCH_AVAILABLE:
    run_direction_ranker_self_tests()
else:
    msg = "Torch unavailable; cannot run DirectionRanker self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


## 08. Neural coefficient generator

In [ ]:
# @title Neural coefficient generator (Torch logits over exact GF(2) coefficients)
from typing import List

if TORCH_AVAILABLE:
    import torch.nn as nn


if TORCH_AVAILABLE:
    class CoefficientGenerator(nn.Module):
        def __init__(self, basis_feat_dim, global_feat_dim, hidden_dim=128):
            super().__init__()
            self.basis_mlp = nn.Sequential(
                nn.Linear(basis_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.global_mlp = nn.Sequential(
                nn.Linear(global_feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.out = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, basis_features, global_features):
            B, r, _ = basis_features.shape
            b = self.basis_mlp(basis_features)
            g = self.global_mlp(global_features).unsqueeze(1).expand(B, r, -1)
            return self.out(torch.cat([b, g], dim=-1)).squeeze(-1)


def basis_features_for_generator(A, u_current, c_current) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    c_bin = as_binary_uint8(c_current, name="c_current", copy=False)

    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")
    if c_bin.shape != (N,):
        raise ValueError(f"c_current must have shape ({N},), got {c_bin.shape}")
    c_from_u = gf2_matvec(A_bin, u_bin)
    if not np.array_equal(c_from_u, c_bin):
        raise ValueError("c_current must equal gf2_matvec(A, u_current)")

    denom_n = max(1.0, float(N))
    denom_r = max(1.0, float(r - 1))
    feats = np.zeros((r, 5), dtype=np.float32)
    for j in range(r):
        col = A_bin[:, j]
        feats[j, 0] = hamming_weight(col) / denom_n
        feats[j, 1] = support_intersection(c_bin, col) / denom_n
        feats[j, 2] = directional_delta(c_bin, col) / denom_n
        feats[j, 3] = float(u_bin[j])
        feats[j, 4] = float(j) / denom_r
    return feats


def global_features_for_generator(A, W, u_current, c_current, bank=None) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    c_bin = as_binary_uint8(c_current, name="c_current", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")
    if c_bin.shape != (N,):
        raise ValueError(f"c_current must have shape ({N},), got {c_bin.shape}")

    wc = hamming_weight(c_bin)
    vals = [
        float(N), float(r), float(W),
        float(W) / max(1.0, float(N)),
        float(r) / max(1.0, float(N)),
        float(wc),
        float(wc) / max(1.0, float(W)),
        float(wc) / max(1.0, float(N)),
    ]

    if bank is None:
        vals.extend([0.0, 0.0, 0.0, 0.0])
    else:
        M = int(bank.D.shape[0])
        if M == 0:
            vals.extend([0.0, 0.0, 0.0, 0.0])
        else:
            deltas = bank.deltas(c_bin)
            min_delta = float(np.min(deltas))
            neg_count = int(np.sum(deltas < 0))
            vals.extend([float(M), min_delta, float(neg_count), float(neg_count) / float(M)])
    return np.asarray(vals, dtype=np.float32)


def planted_coefficient_target(inst, u_current) -> np.ndarray:
    inst.verify()
    u_bin = as_binary_uint8(u_current, name="u_current", copy=False)
    r = inst.A.shape[1]
    if u_bin.shape != (r,):
        raise ValueError(f"u_current must have shape ({r},), got {u_bin.shape}")

    v_star = (u_bin ^ inst.u_star).astype(np.uint8, copy=False)
    lhs = gf2_matvec(inst.A, v_star)
    rhs = gf2_matvec(inst.A, u_bin) ^ inst.c_star
    if not np.array_equal(lhs, rhs):
        raise ValueError("Planted coefficient target identity failed")
    return v_star


def coefficient_direction_from_v(A, v, bank=None) -> dict:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    v_bin = as_binary_uint8(v, name="v", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
    N, r = A_bin.shape
    if v_bin.shape != (r,):
        raise ValueError(f"v must have shape ({r},), got {v_bin.shape}")

    d = gf2_matvec(A_bin, v_bin)
    zero = hamming_weight(d) == 0
    duplicate = False if bank is None else bank_contains_direction(bank, d)
    return {
        "v": v_bin.copy(),
        "d": d,
        "direction_weight": hamming_weight(d),
        "is_zero_direction": bool(zero),
        "duplicate_in_bank": bool(duplicate),
    }


def sample_coefficients_from_logits(logits, num_samples=1, seed=None, temperature=1.0) -> np.ndarray:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot sample from logits.")
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    t_logits = torch.as_tensor(logits, dtype=torch.float32)
    if t_logits.ndim not in (1, 2):
        raise ValueError(f"logits must have shape [r] or [B, r], got {tuple(t_logits.shape)}")
    gen = None
    if seed is not None:
        gen = torch.Generator(device="cpu")
        gen.manual_seed(int(seed))
    probs = torch.sigmoid(t_logits / float(temperature))
    if probs.ndim == 1:
        p = probs.unsqueeze(0).expand(int(num_samples), -1)
        out = torch.bernoulli(p, generator=gen)
        return out.to(dtype=torch.uint8).cpu().numpy()

    if int(num_samples) != 1:
        raise ValueError("For batched logits [B, r], only num_samples=1 is supported")
    out = torch.bernoulli(probs, generator=gen)
    return out.to(dtype=torch.uint8).cpu().numpy()


def propose_coefficients_with_model(
    model,
    A,
    W,
    u_current,
    c_current,
    bank=None,
    num_samples=1,
    device="cpu",
    seed=None,
    temperature=1.0,
) -> List[dict]:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot propose coefficients with model.")
    bf = basis_features_for_generator(A, u_current, c_current)
    gf = global_features_for_generator(A, W, u_current, c_current, bank=bank)

    model = model.to(device)
    model.eval()
    with torch.no_grad():
        bf_t = torch.from_numpy(bf).unsqueeze(0).to(device=device, dtype=torch.float32)
        gf_t = torch.from_numpy(gf).unsqueeze(0).to(device=device, dtype=torch.float32)
        logits_t = model(bf_t, gf_t).squeeze(0).detach().cpu()

    v_samples = sample_coefficients_from_logits(
        logits_t,
        num_samples=num_samples,
        seed=seed,
        temperature=temperature,
    )

    probs = torch.sigmoid(logits_t / float(temperature)).cpu().numpy()
    proposals = []
    for v in v_samples:
        p = coefficient_direction_from_v(A, v, bank=bank)
        p["logits"] = logits_t.numpy().astype(np.float32, copy=True)
        p["probability_mean"] = float(np.mean(probs))
        # Heuristic proposal only; exact acceptance/validity checks happen elsewhere.
        proposals.append(p)
    return proposals


In [ ]:
# @title Coefficient generator self-tests (small deterministic smoke checks)
def run_coefficient_generator_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; install torch or use Colab runtime with PyTorch.")

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    # 1) Shape test
    B, r, Fb, Fg = 3, 7, 5, 12
    model = CoefficientGenerator(Fb, Fg, hidden_dim=32)
    bf = torch.randn(B, r, Fb)
    gf = torch.randn(B, Fg)
    logits = model(bf, gf)
    assert logits.shape == (B, r)

    # 2) Planted target identity
    inst = generate_planted_span_instance(N=24, r=9, W=5, seed=seed, hide=True)
    for _ in range(6):
        u = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
        v_star = planted_coefficient_target(inst, u)
        lhs = gf2_matvec(inst.A, v_star)
        rhs = gf2_matvec(inst.A, u) ^ inst.c_star
        assert np.array_equal(lhs, rhs)

    # 3) Feature tests
    u0 = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    c0 = gf2_matvec(inst.A, u0)
    bfeat = basis_features_for_generator(inst.A, u0, c0)
    gfeat = global_features_for_generator(inst.A, inst.W, u0, c0, bank=None)
    assert bfeat.shape[0] == inst.A.shape[1] and bfeat.dtype == np.float32
    assert gfeat.dtype == np.float32
    assert np.isfinite(bfeat).all() and np.isfinite(gfeat).all()

    # 4) Sampling tests
    test_logits = torch.tensor([0.1, -0.2, 0.0, 2.0], dtype=torch.float32)
    v_s = sample_coefficients_from_logits(test_logits, num_samples=5, seed=seed, temperature=1.0)
    assert v_s.shape == (5, 4)
    assert v_s.dtype == np.uint8 and np.isin(v_s, [0, 1]).all()

    inst2 = generate_planted_span_instance(N=12, r=4, W=3, seed=seed + 1, hide=True)
    d_info = coefficient_direction_from_v(inst2.A, v_s[0])
    assert np.array_equal(d_info["d"], gf2_matvec(inst2.A, v_s[0]))

    # 5) Proposal tests + no mutation
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(K=8, rng=rng)
    d_before = bank.D.copy()
    v_before = bank.V.copy()
    h_before = set(bank.hashes)
    model2 = CoefficientGenerator(bfeat.shape[1], gfeat.shape[0], hidden_dim=32)
    proposals = propose_coefficients_with_model(
        model2, inst.A, inst.W, u0, c0, bank=bank, num_samples=4, device="cpu", seed=seed
    )
    assert len(proposals) == 4
    for p in proposals:
        assert np.isin(p["v"], [0, 1]).all() and np.isin(p["d"], [0, 1]).all()
        assert np.array_equal(p["d"], gf2_matvec(inst.A, p["v"]))
        assert p["is_zero_direction"] == (hamming_weight(p["d"]) == 0)
        assert p["duplicate_in_bank"] == bank_contains_direction(bank, p["d"])
    assert np.array_equal(bank.D, d_before)
    assert np.array_equal(bank.V, v_before)
    assert bank.hashes == h_before

    # 6) Tiny supervised BCE loss-decrease test
    examples = []
    for i in range(2):
        inst_i = generate_planted_span_instance(N=20, r=8, W=4, seed=seed + 10 + i, hide=True)
        for _ in range(8):
            u_i = rng.integers(0, 2, size=(inst_i.A.shape[1],), dtype=np.uint8)
            c_i = gf2_matvec(inst_i.A, u_i)
            x_b = basis_features_for_generator(inst_i.A, u_i, c_i)
            x_g = global_features_for_generator(inst_i.A, inst_i.W, u_i, c_i, bank=None)
            y = planted_coefficient_target(inst_i, u_i).astype(np.float32)
            examples.append((x_b, x_g, y))

    Xb = torch.tensor(np.stack([e[0] for e in examples]), dtype=torch.float32)
    Xg = torch.tensor(np.stack([e[1] for e in examples]), dtype=torch.float32)
    Y = torch.tensor(np.stack([e[2] for e in examples]), dtype=torch.float32)

    train_model = CoefficientGenerator(Xb.shape[-1], Xg.shape[-1], hidden_dim=48)
    opt = torch.optim.Adam(train_model.parameters(), lr=1e-2)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    with torch.no_grad():
        initial_loss = float(loss_fn(train_model(Xb, Xg), Y).item())

    for _ in range(40):
        opt.zero_grad()
        loss = loss_fn(train_model(Xb, Xg), Y)
        loss.backward()
        opt.step()

    with torch.no_grad():
        final_loss = float(loss_fn(train_model(Xb, Xg), Y).item())
    assert final_loss < initial_loss, (initial_loss, final_loss)

    # 7) No validity claims: proposals are heuristic; d must be exact from A and v.
    for p in proposals:
        assert np.array_equal(p["d"], gf2_matvec(inst.A, p["v"]))


if TORCH_AVAILABLE:
    run_coefficient_generator_self_tests()
else:
    msg = "Torch unavailable; cannot run CoefficientGenerator self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


## 09. Cross-entropy sampler

In [ ]:

# @title Cross-entropy sampler (heuristic restart distribution over GF(2) coefficients)
from dataclasses import dataclass, field

@dataclass
class CrossEntropyCandidate:
    u_initial: np.ndarray
    u_final: np.ndarray
    c_final: np.ndarray
    weight: int
    descent_steps: int
    improved_by_descent: bool
    metadata: dict = field(default_factory=dict)


class CrossEntropySampler:
    def __init__(
        self,
        A,
        W: int,
        p_init=0.5,
        p_min: float = 0.01,
        p_max: float = 0.99,
        elite_frac: float = 0.2,
        smoothing: float = 0.5,
        seed: int = 0,
    ):
        A_bin = as_binary_uint8(A, name="A", copy=True)
        if A_bin.ndim != 2:
            raise ValueError(f"A must have shape [N, r], got {A_bin.shape}")
        N, r = A_bin.shape
        if N <= 0 or r <= 0:
            raise ValueError("A must have positive dimensions N and r")

        if not isinstance(W, (int, np.integer)):
            raise TypeError("W must be an integer")
        if not (1 <= int(W) <= N):
            raise ValueError(f"W must satisfy 1 <= W <= N, got W={W}, N={N}")

        if np.isscalar(p_init):
            p_arr = np.full((r,), float(p_init), dtype=np.float64)
        else:
            p_arr = np.asarray(p_init, dtype=np.float64)
            if p_arr.shape != (r,):
                raise ValueError(f"p_init array must have shape [{r}], got {p_arr.shape}")
        if np.any((p_arr < 0.0) | (p_arr > 1.0)):
            raise ValueError("p_init entries must be in [0, 1]")

        p_min = float(p_min)
        p_max = float(p_max)
        if not (0.0 <= p_min < p_max <= 1.0):
            raise ValueError("p_min and p_max must satisfy 0 <= p_min < p_max <= 1")

        elite_frac = float(elite_frac)
        if not (0.0 < elite_frac <= 1.0):
            raise ValueError("elite_frac must be in (0, 1]")

        smoothing = float(smoothing)
        if not (0.0 < smoothing <= 1.0):
            raise ValueError("smoothing must be in (0, 1]")

        self.A = A_bin
        self.N = N
        self.r = r
        self.W = int(W)
        self.p_min = p_min
        self.p_max = p_max
        self.elite_frac = elite_frac
        self.smoothing = smoothing
        self.p = np.clip(p_arr, self.p_min, self.p_max).astype(np.float64, copy=False)
        self.rng = np.random.default_rng(seed)
        self.best_seen: Optional[CrossEntropyCandidate] = None
        self.verify()

    def entropy(self) -> float:
        p = np.clip(self.p.astype(np.float64, copy=False), 1e-12, 1.0 - 1e-12)
        ent = -p * np.log(p) - (1.0 - p) * np.log(1.0 - p)
        return float(np.sum(ent, dtype=np.float64))

    def sample_coefficients(self, num_samples: int, rng=None) -> np.ndarray:
        if not isinstance(num_samples, (int, np.integer)) or int(num_samples) <= 0:
            raise ValueError("num_samples must be a positive integer")
        local_rng = self.rng if rng is None else rng
        U = local_rng.binomial(1, self.p.reshape(1, -1), size=(int(num_samples), self.r)).astype(np.uint8)
        return U

    def evaluate_candidate(self, u, bank=None, max_descent_steps: int = 0) -> CrossEntropyCandidate:
        u0 = as_binary_uint8(u, name="u", copy=True)
        if u0.ndim != 1 or u0.shape[0] != self.r:
            raise ValueError(f"u must have shape [{self.r}], got {u0.shape}")

        c = gf2_matvec(self.A, u0)
        u_work = u0.copy()
        steps = 0

        if bank is not None and int(max_descent_steps) > 0:
            bank.verify()
            if not np.array_equal(bank.A, self.A):
                raise ValueError("bank.A must equal sampler A")
            max_steps = int(max_descent_steps)
            while steps < max_steps:
                idx = bank.best_descent(c)
                if idx is None:
                    break
                d = bank.D[idx]
                v = bank.V[idx]
                c_new = c ^ d
                if hamming_weight(c_new) == 0:
                    break
                if hamming_weight(c_new) >= hamming_weight(c):
                    break
                u_work = u_work ^ v
                c = c_new
                steps += 1
                if not np.array_equal(c, gf2_matvec(self.A, u_work)):
                    raise AssertionError("coefficient-tracked invariant violated: c != A u")

        weight = hamming_weight(c)
        return CrossEntropyCandidate(
            u_initial=u0,
            u_final=u_work,
            c_final=c,
            weight=int(weight),
            descent_steps=int(steps),
            improved_by_descent=bool(steps > 0 and weight < hamming_weight(gf2_matvec(self.A, u0))),
            metadata={"heuristic": "cross_entropy_restart"},
        )

    def evaluate_batch(self, num_samples: int, bank=None, max_descent_steps: int = 0) -> list[CrossEntropyCandidate]:
        U = self.sample_coefficients(num_samples=num_samples)
        return [self.evaluate_candidate(U[i], bank=bank, max_descent_steps=max_descent_steps) for i in range(U.shape[0])]

    def select_elites(self, candidates, elite_frac=None, max_elites=None) -> list[CrossEntropyCandidate]:
        if len(candidates) == 0:
            raise ValueError("candidates must be nonempty")
        frac = self.elite_frac if elite_frac is None else float(elite_frac)
        if not (0.0 < frac <= 1.0):
            raise ValueError("elite_frac must be in (0, 1]")
        ordered = sorted(candidates, key=lambda c: (int(c.weight), -int(c.descent_steps)))
        k = max(1, int(np.ceil(frac * len(ordered))))
        if max_elites is not None:
            if int(max_elites) <= 0:
                raise ValueError("max_elites must be positive when provided")
            k = min(k, int(max_elites))
        return ordered[:k]

    def update_from_elites(self, elites, smoothing=None) -> dict:
        if len(elites) == 0:
            raise ValueError("elites must be nonempty")
        eta = self.smoothing if smoothing is None else float(smoothing)
        if not (0.0 < eta <= 1.0):
            raise ValueError("smoothing must be in (0, 1]")

        Ue = np.stack([as_binary_uint8(e.u_final, name="elite.u_final", copy=False) for e in elites], axis=0)
        if Ue.shape[1] != self.r:
            raise ValueError("elite coefficient dimension mismatch")
        elite_mean = Ue.mean(axis=0, dtype=np.float64)

        p_old = self.p.copy()
        entropy_old = self.entropy()
        p_new = (1.0 - eta) * p_old + eta * elite_mean
        p_new = np.clip(p_new, self.p_min, self.p_max)
        self.p = p_new.astype(np.float64, copy=False)
        entropy_new = self.entropy()
        self.verify()
        return {
            "p_old": p_old,
            "p_new": self.p.copy(),
            "elite_mean": elite_mean,
            "entropy_old": float(entropy_old),
            "entropy_new": float(entropy_new),
            "num_elites": int(len(elites)),
        }

    def best_seen_update(self, candidates) -> Optional[CrossEntropyCandidate]:
        if len(candidates) == 0:
            return self.best_seen
        best_batch = min(candidates, key=lambda c: (int(c.weight), -int(c.descent_steps)))
        if self.best_seen is None or (best_batch.weight, -best_batch.descent_steps) < (self.best_seen.weight, -self.best_seen.descent_steps):
            self.best_seen = best_batch
        return self.best_seen

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        if A.ndim != 2:
            raise ValueError("A must be 2D")
        N, r = A.shape
        if N <= 0 or r <= 0:
            raise ValueError("A dimensions must be positive")
        if not isinstance(self.W, (int, np.integer)) or not (1 <= int(self.W) <= N):
            raise ValueError("W out of range")
        if self.p.shape != (r,):
            raise ValueError(f"p must have shape [{r}], got {self.p.shape}")
        if np.any(~np.isfinite(self.p)):
            raise ValueError("p contains non-finite values")
        if np.any((self.p < self.p_min) | (self.p > self.p_max)):
            raise ValueError("p out of configured bounds")
        if not (0.0 <= self.p_min < self.p_max <= 1.0):
            raise ValueError("invalid p_min/p_max")
        if not (0.0 < self.elite_frac <= 1.0):
            raise ValueError("invalid elite_frac")
        if not (0.0 < self.smoothing <= 1.0):
            raise ValueError("invalid smoothing")


def run_cross_entropy_round(
    sampler,
    num_samples,
    bank=None,
    max_descent_steps=0,
    elite_frac=None,
    smoothing=None,
) -> dict:
    candidates = sampler.evaluate_batch(num_samples=num_samples, bank=bank, max_descent_steps=max_descent_steps)
    elites = sampler.select_elites(candidates, elite_frac=elite_frac)
    update_info = sampler.update_from_elites(elites, smoothing=smoothing)
    best_candidate = sampler.best_seen_update(candidates)
    # Cross-entropy output is heuristic best-so-far only; no certification/optimality claim.
    return {
        "candidates": candidates,
        "elites": elites,
        "update_info": update_info,
        "best_candidate": best_candidate,
    }


In [ ]:

# @title Cross-entropy sampler self-tests (exact invariants + heuristic update checks)
def run_cross_entropy_sampler_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # 1) Initialization and sampling.
    inst = generate_planted_span_instance(N=18, r=9, W=4, seed=seed, hide=True)
    sampler = CrossEntropySampler(inst.A, W=inst.W, p_init=0.5, seed=seed)
    sampler.verify()
    U = sampler.sample_coefficients(16)
    assert U.shape == (16, inst.A.shape[1])
    assert U.dtype == np.uint8
    assert np.all((U == 0) | (U == 1))

    # 2) Candidate evaluation exactness.
    cand = sampler.evaluate_candidate(U[0])
    assert np.array_equal(cand.c_final, gf2_matvec(inst.A, cand.u_final))
    assert cand.weight == hamming_weight(cand.c_final)

    # 3) Deterministic update formula + clipping.
    A_small = np.array([[1,0,1,0],[0,1,1,0],[1,1,0,1]], dtype=np.uint8)
    s2 = CrossEntropySampler(A_small, W=1, p_init=0.5, p_min=0.2, p_max=0.8, smoothing=0.5, seed=seed)
    elites = [
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.array([1,0,1,0], dtype=np.uint8), np.zeros(3, dtype=np.uint8), 0, 0, False),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.array([1,1,1,0], dtype=np.uint8), np.zeros(3, dtype=np.uint8), 0, 0, False),
    ]
    p_old = s2.p.copy()
    info = s2.update_from_elites(elites, smoothing=0.5)
    elite_mean = np.array([1.0, 0.5, 1.0, 0.0], dtype=np.float64)
    expected = np.clip((1.0 - 0.5) * p_old + 0.5 * elite_mean, s2.p_min, s2.p_max)
    assert np.allclose(info["p_new"], expected, atol=1e-12)
    assert np.all((info["p_new"] >= s2.p_min) & (info["p_new"] <= s2.p_max))

    # 4) Entropy finite/nonnegative and changes after update.
    ent0 = float(info["entropy_old"])
    ent1 = float(info["entropy_new"])
    assert np.isfinite(ent0) and np.isfinite(ent1)
    assert ent0 >= 0.0 and ent1 >= 0.0
    assert abs(ent1 - ent0) > 1e-12

    # 5) Elite selection ordering and minimum 1.
    fake = [
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 7, 0, False),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 3, 1, True),
        CrossEntropyCandidate(np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8), np.zeros(3, dtype=np.uint8), 3, 0, False),
    ]
    elites_small = sampler.select_elites(fake, elite_frac=0.01)
    assert len(elites_small) >= 1
    assert elites_small[0].weight == 3 and elites_small[0].descent_steps == 1

    # 6) Coefficient-tracked descent exactness.
    A_id = np.eye(4, dtype=np.uint8)
    bank = DirectionBank(A_id)
    bank.add_one(np.array([1,0,0,0], dtype=np.uint8))
    u_start = np.array([1,1,0,0], dtype=np.uint8)
    s3 = CrossEntropySampler(A_id, W=1, p_init=0.5, seed=seed)
    cand_d = s3.evaluate_candidate(u_start, bank=bank, max_descent_steps=4)
    assert cand_d.descent_steps >= 1
    assert cand_d.weight < hamming_weight(gf2_matvec(A_id, u_start))
    assert np.array_equal(cand_d.c_final, gf2_matvec(A_id, cand_d.u_final))

    # 7) No bank mutation.
    V_before = bank.V.copy()
    D_before = bank.D.copy()
    M_before = bank.D.shape[0]
    _ = s3.evaluate_candidate(u_start, bank=bank, max_descent_steps=2)
    _ = s3.evaluate_batch(5, bank=bank, max_descent_steps=2)
    assert bank.D.shape[0] == M_before
    assert np.array_equal(bank.V, V_before)
    assert np.array_equal(bank.D, D_before)

    # 8) Round helper contract and bounds.
    round_out = run_cross_entropy_round(sampler, num_samples=10, bank=None, max_descent_steps=0)
    for key in ["candidates", "elites", "update_info", "best_candidate"]:
        assert key in round_out
    assert np.all((sampler.p >= sampler.p_min) & (sampler.p <= sampler.p_max))
    bc = round_out["best_candidate"]
    assert bc is not None
    assert np.array_equal(bc.c_final, gf2_matvec(sampler.A, bc.u_final))

    # 9) No certification claim: heuristic best-so-far only.
    assert round_out["best_candidate"].metadata.get("heuristic") == "cross_entropy_restart"


run_cross_entropy_sampler_self_tests()


## 10. Strategy-level Q-controller skeleton

In [ ]:
# @title Strategy-level Q-controller skeleton (macro-action selector; exact algebra remains symbolic)
from dataclasses import dataclass, field
from typing import Optional

if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn


if TORCH_AVAILABLE:
    class QControllerDQN(nn.Module):
        def __init__(self, obs_dim, num_actions=8, hidden_dim=256):
            super().__init__()
            self.trunk = nn.Sequential(
                nn.Linear(obs_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )
            self.value = nn.Linear(hidden_dim, 1)
            self.advantage = nn.Linear(hidden_dim, num_actions)

        def forward(self, obs, action_mask=None):
            h = self.trunk(obs)
            v = self.value(h)
            a = self.advantage(h)
            q = v + a - a.mean(dim=-1, keepdim=True)
            if action_mask is not None:
                q = q.masked_fill(~action_mask.bool(), -1e9)
            return q


@dataclass
class SearchControllerState:
    inst: SpanInstance
    W: int
    u_current: np.ndarray
    c_current: np.ndarray
    bank: DirectionBank
    gradient_engine: GradientEngine
    archive: FailedMinimaArchive
    ce_sampler: CrossEntropySampler
    best_u: Optional[np.ndarray]
    best_c: Optional[np.ndarray]
    best_weight: int
    step: int
    terminal: bool = False
    terminal_reason: str = ""
    last_solver_result: Optional[SolverResult] = None
    last_action_info: dict = field(default_factory=dict)


@dataclass
class StepResult:
    observation: np.ndarray
    action_mask: np.ndarray
    reward: float
    done: bool
    info: dict


def _weight(x: np.ndarray) -> int:
    return int(np.sum(x, dtype=np.int64))


def initialize_controller_state(inst, M_init: int = 32, seed: int = 0, ce_sampler: Optional[CrossEntropySampler] = None) -> SearchControllerState:
    rng = np.random.default_rng(seed)
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(M_init, rng)
    bank.verify()
    gradient_engine = GradientEngine()
    archive = FailedMinimaArchive(inst.A, inst.W)
    archive.verify()
    if ce_sampler is None:
        ce_sampler = CrossEntropySampler(inst.A, W=inst.W, seed=seed)
    ce_sampler.verify()
    u0 = ce_sampler.sample_coefficients(1)[0]
    c0 = gf2_matvec(inst.A, u0)
    return SearchControllerState(inst=inst, W=inst.W, u_current=u0.copy(), c_current=c0.copy(), bank=bank, gradient_engine=gradient_engine, archive=archive, ce_sampler=ce_sampler, best_u=None, best_c=None, best_weight=inst.A.shape[0] + 1, step=0)


def controller_observation(state) -> np.ndarray:
    N, r = state.inst.A.shape
    c = state.c_current
    wcur = _weight(c)
    M = state.bank.D.shape[0]
    if M > 0:
        deltas = np.array([directional_delta(c, d) for d in state.bank.D], dtype=np.int64)
        dweights = np.array([_weight(d) for d in state.bank.D], dtype=np.float32)
        min_delta = float(np.min(deltas))
        nneg = int(np.sum(deltas < 0))
        frac_neg = float(nneg / max(1, M))
        dmin, dmean, dmax = float(np.min(dweights)), float(np.mean(dweights)), float(np.max(dweights))
    else:
        min_delta = 0.0
        nneg = 0
        frac_neg = 0.0
        dmin = dmean = dmax = 0.0
    best_w = state.best_weight if state.best_c is not None else 0
    failed_count = len(state.archive.entries)
    failed_best = min((e["weight"] for e in state.archive.entries), default=0)
    entropy = float(state.ce_sampler.entropy())
    solver_found = 1.0 if (state.last_solver_result is not None and state.last_solver_result.found) else 0.0
    solver_runtime = float(state.last_solver_result.runtime_s) if state.last_solver_result is not None else 0.0
    obs = np.array([
        N, r, state.W, state.W / max(1, N), r / max(1, N), M, M / max(1, N),
        wcur, wcur / max(1, state.W), wcur / max(1, N), best_w, best_w / max(1, state.W),
        min_delta, nneg, frac_neg, dmin, dmean, dmax, failed_count, failed_best, entropy,
        solver_found, solver_runtime, state.step, 1.0 if state.terminal else 0.0,
    ], dtype=np.float32)
    return obs


def controller_action_mask(state, direction_ranker=None, coefficient_generator=None, local_solver_available: Optional[bool] = None) -> np.ndarray:
    if local_solver_available is None:
        local_solver_available = ORTOOLS_AVAILABLE
    m = np.zeros(8, dtype=bool)
    m[0] = state.bank.best_descent(state.c_current) is not None
    m[1] = (direction_ranker is not None) and TORCH_AVAILABLE and (state.bank.D.shape[0] > 0)
    m[2] = True
    m[3] = (coefficient_generator is not None) and TORCH_AVAILABLE
    m[4] = bool(local_solver_available)
    m[5] = True
    m[6] = True
    m[7] = len(state.archive.entries) > 0
    return m


def choose_action_epsilon_greedy(q_model, observation, action_mask, epsilon: float = 0.1, rng=None, device="cpu") -> int:
    if rng is None:
        rng = np.random.default_rng(0)
    valid = np.where(action_mask)[0]
    if len(valid) == 0:
        raise RuntimeError("No available controller actions.")
    if rng.random() < epsilon:
        return int(rng.choice(valid))
    if q_model is None:
        return int(valid[0])
    obs_t = torch.tensor(observation[None, :], dtype=torch.float32, device=device)
    mask_t = torch.tensor(action_mask[None, :], dtype=torch.bool, device=device)
    with torch.no_grad():
        q = q_model(obs_t, action_mask=mask_t)[0].detach().cpu().numpy()
    return int(np.argmax(q))


def controller_step(state, action: int, rng=None, direction_ranker=None, coefficient_generator=None, K_add: int = 8, max_descent_steps: int = 5, solver_time_limit_s: float = 5.0) -> StepResult:
    if rng is None:
        rng = np.random.default_rng(0)
    mask = controller_action_mask(state, direction_ranker=direction_ranker, coefficient_generator=coefficient_generator)
    if not (0 <= int(action) <= 7):
        raise ValueError("Action must be in [0,7].")
    if not mask[action]:
        raise ValueError(f"Action {action} is currently masked out.")

    info = {"action": int(action), "moved": False}
    old_weight = _weight(state.c_current)
    old_best = state.best_weight

    if action == 0:
        idx = state.bank.best_descent(state.c_current)
        c_new, moved = state.gradient_engine.apply_best(state.c_current, state.bank)
        if moved and idx is not None:
            state.c_current = c_new
            state.u_current = (state.u_current ^ state.bank.V[idx]).astype(np.uint8)
            info.update({"moved": True, "idx": int(idx)})
        else:
            info["no_descent"] = True
    elif action == 1:
        gfeat = global_features_for_ranker(
            state.inst.A,
            state.W,
            state.c_current,
            state.bank,
        )
        ranking = rank_directions_with_model(
            direction_ranker,
            state.c_current,
            state.bank,
            gfeat,
        )
        idx = exact_best_direction_from_ranking(state.c_current, state.bank, ranking)
        info["neural_action_executed"] = True
        info["ranked_direction_count"] = int(len(ranking))
        info["selected_idx"] = None if idx is None else int(idx)
        if idx is not None:
            d = state.bank.D[idx]
            candidate = (state.c_current ^ d).astype(np.uint8)
            if directional_delta(state.c_current, d) < 0 and _weight(candidate) > 0:
                state.c_current = candidate
                state.u_current = (state.u_current ^ state.bank.V[idx]).astype(np.uint8)
                info["moved"] = True
            else:
                info["no_ranked_descent"] = True
        else:
            info["no_ranked_descent"] = True
    elif action == 2:
        prev = state.bank.D.shape[0]
        state.bank.add_random_span_directions(K_add, rng)
        info["added"] = state.bank.D.shape[0] - prev
    elif action == 3:
        proposals = propose_coefficients_with_model(
            coefficient_generator,
            state.inst.A,
            state.W,
            state.u_current,
            state.c_current,
            bank=state.bank,
            num_samples=K_add,
            seed=int(rng.integers(1_000_000)),
        )
        added = 0
        for proposal in proposals:
            v = proposal["v"]
            if state.bank.add_one(v):
                added += 1
        state.bank.verify()
        info["neural_action_executed"] = True
        info["proposal_count"] = int(len(proposals))
        info["added"] = int(added)
        info["added_count"] = int(added)
    elif action == 4:
        solver = LocalMinimumSolver(random_seed=int(rng.integers(1_000_000)))
        res = solver.solve(state.inst.A, state.W, state.bank, time_limit_s=solver_time_limit_s)
        state.last_solver_result = res
        info["solver_found"] = bool(res.found)
        if res.found:
            state.c_current = res.c.copy()
            state.u_current = res.u.copy()
            info["moved"] = True
    elif action == 5:
        u = state.ce_sampler.sample_coefficients(1)[0]
        state.u_current = u.copy()
        state.c_current = gf2_matvec(state.inst.A, state.u_current)
        info["restart"] = True
    elif action == 6:
        n_samples = 16 if (HEADLESS or SMOKE) else 32
        out = run_cross_entropy_round(state.ce_sampler, num_samples=n_samples, bank=state.bank, max_descent_steps=max_descent_steps)
        best = out["best_candidate"]
        info["ce_updated"] = True
        if best is not None and _weight(best.c_final) > 0 and _weight(best.c_final) < _weight(state.c_current):
            state.u_current = best.u_final.copy()
            state.c_current = best.c_final.copy()
            info["moved"] = True
    elif action == 7:
        target = state.archive.latest()
        if target is not None:
            proposal = random_attack_failed_minimum(state.inst.A, target["c"], state.bank, num_trials=16, rng=rng)
            if proposal is not None:
                added = state.bank.add_one(proposal["v"])
                info["added"] = int(bool(added))

    # Exact invariants
    assert np.array_equal(state.c_current, gf2_matvec(state.inst.A, state.u_current))
    state.bank.verify()

    w = _weight(state.c_current)
    if w > 0 and w < state.best_weight:
        state.best_weight = w
        state.best_c = state.c_current.copy()
        state.best_u = state.u_current.copy()

    report = None
    if w > 0 and w > state.W:
        report = local_minimum_report(state.c_current, state.bank)
        if report.get("is_local", False):
            state.archive.add(state.u_current, state.c_current, state.bank)

    if w > 0 and w <= state.W:
        state.terminal = True
        state.terminal_reason = "found_threshold_solution"

    reward = float(old_weight - w) + (10.0 if (w > 0 and w <= state.W) else 0.0) - 0.01
    state.step += 1
    state.last_action_info = info
    info.update({"old_weight": old_weight, "new_weight": w, "best_weight_before": old_best, "best_weight_after": state.best_weight, "terminal": state.terminal, "terminal_reason": state.terminal_reason})
    if report is not None:
        info["local_report"] = report
    return StepResult(observation=controller_observation(state), action_mask=controller_action_mask(state, direction_ranker, coefficient_generator), reward=reward, done=state.terminal, info=info)


def run_short_controller_episode(inst, num_steps: int = 8, seed: int = 0, use_models: bool = False) -> list[dict]:
    rng = np.random.default_rng(seed)
    state = initialize_controller_state(inst, seed=seed)
    trace = []
    for _ in range(num_steps):
        mask = controller_action_mask(state)
        a = int(rng.choice(np.where(mask)[0]))
        out = controller_step(state, a, rng=rng)
        row = {"action": a, "reward": out.reward, "current_weight": _weight(state.c_current), "best_weight": state.best_weight, "num_directions": int(state.bank.D.shape[0]), "failed_minima_count": len(state.archive.entries), "terminal": state.terminal, "terminal_reason": state.terminal_reason}
        trace.append(row)
        if state.terminal:
            break
    return trace


In [ ]:
# @title Q-controller self-tests (transition/coherence smoke only; no learning-quality claim)
def run_q_controller_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable; cannot run Q-controller self-tests.")
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    Fo = 25
    model = QControllerDQN(obs_dim=Fo, num_actions=8, hidden_dim=64)
    obs = torch.randn(4, Fo)
    mask = torch.tensor([[1,1,1,1,1,1,1,1],[1,0,1,0,1,0,1,0],[0,1,0,1,0,1,0,1],[1,1,0,0,1,1,0,0]], dtype=torch.bool)
    q = model(obs, action_mask=mask)
    assert q.shape == (4, 8)
    assert torch.all(q[~mask] <= -1e8)

    inst = generate_planted_span_instance(N=24, r=10, W=4, seed=seed, hide=True)
    state = initialize_controller_state(inst, M_init=16, seed=seed)
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))
    state.bank.verify(); state.archive.verify(); state.ce_sampler.verify()
    obs0 = controller_observation(state)
    assert np.all(np.isfinite(obs0))
    am = controller_action_mask(state)
    assert am.shape == (8,)

    assert am[2] and am[5] and am[6]
    assert am[4] == bool(ORTOOLS_AVAILABLE)
    assert not controller_action_mask(state, direction_ranker=None, coefficient_generator=None)[1]
    assert not controller_action_mask(state, direction_ranker=None, coefficient_generator=None)[3]

    ranker_bank = DirectionBank(inst.A)
    ranker_bank.add_random_span_directions(8, rng)
    ranker_bank.verify()
    s_rank = initialize_controller_state(inst, M_init=8, seed=seed+11)
    gfeat = global_features_for_ranker(inst.A, s_rank.W, s_rank.c_current, s_rank.bank)
    dir_feat = s_rank.bank.features_for_ranker(s_rank.c_current)
    ranker = DirectionRanker(dir_feat_dim=int(dir_feat.shape[1]), global_feat_dim=int(gfeat.shape[0]), hidden_dim=16)
    w_before = _weight(s_rank.c_current)
    out_rank = controller_step(s_rank, 1, rng=rng, direction_ranker=ranker)
    assert out_rank.info.get("neural_action_executed", False)
    if out_rank.info.get("moved", False):
        assert np.array_equal(s_rank.c_current, gf2_matvec(inst.A, s_rank.u_current))
        assert _weight(s_rank.c_current) < w_before
    else:
        assert out_rank.info.get("no_ranked_descent", False)

    s_gen = initialize_controller_state(inst, M_init=8, seed=seed+12)
    bfeat = basis_features_for_generator(inst.A, s_gen.u_current, s_gen.c_current)
    ggen = global_features_for_generator(inst.A, s_gen.W, s_gen.u_current, s_gen.c_current, s_gen.bank)
    gen = CoefficientGenerator(basis_feat_dim=int(bfeat.shape[1]), global_feat_dim=int(ggen.shape[0]), hidden_dim=16)
    prev_dirs = int(s_gen.bank.D.shape[0])
    out_gen = controller_step(s_gen, 3, rng=rng, coefficient_generator=gen, K_add=4)
    assert out_gen.info.get("neural_action_executed", False)
    assert int(s_gen.bank.D.shape[0]) >= prev_dirs or int(out_gen.info.get("added", 0)) == 0
    s_gen.bank.verify()
    assert np.array_equal(s_gen.c_current, gf2_matvec(inst.A, s_gen.u_current))

    A = np.eye(6, dtype=np.uint8)
    inst2 = SpanInstance(A=A, W=2, u_star=np.array([1,1,0,0,0,0],dtype=np.uint8), c_star=np.array([1,1,0,0,0,0],dtype=np.uint8), metadata={})
    s2 = initialize_controller_state(inst2, M_init=0, seed=seed)
    s2.u_current = np.array([1,1,1,0,0,0], dtype=np.uint8)
    s2.c_current = gf2_matvec(A, s2.u_current)
    s2.bank.add_one(np.array([0,0,1,0,0,0], dtype=np.uint8))
    w_old = _weight(s2.c_current)
    r0 = controller_step(s2, 0, rng=rng)
    assert np.array_equal(s2.c_current, gf2_matvec(A, s2.u_current))
    if r0.info.get("moved", False):
        assert _weight(s2.c_current) < w_old
        assert _weight(s2.c_current) > 0

    prev = state.bank.D.shape[0]
    controller_step(state, 2, rng=rng)
    assert state.bank.D.shape[0] >= prev
    state.bank.verify()

    controller_step(state, 5, rng=rng)
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))
    controller_step(state, 6, rng=rng)
    assert np.all((state.ce_sampler.p >= 0.0) & (state.ce_sampler.p <= 1.0))
    assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))

    if ORTOOLS_AVAILABLE:
        tiny = generate_planted_span_instance(N=12, r=6, W=3, seed=seed+1, hide=True)
        s3 = initialize_controller_state(tiny, M_init=4, seed=seed+1)
        o = controller_step(s3, 4, rng=rng, solver_time_limit_s=1.0)
        if o.info.get("solver_found", False):
            assert np.array_equal(s3.c_current, gf2_matvec(tiny.A, s3.u_current))
            assert _weight(s3.c_current) <= s3.W

    s4 = initialize_controller_state(inst2, M_init=0, seed=seed+3)
    s4.u_current = np.array([1,1,1,1,0,0], dtype=np.uint8)
    s4.c_current = gf2_matvec(A, s4.u_current)
    s4.bank.add_one(np.array([1,0,0,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,1,0,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,0,1,0,0,0], dtype=np.uint8))
    s4.bank.add_one(np.array([0,0,0,1,0,0], dtype=np.uint8))
    out4 = controller_step(s4, 2, rng=rng, K_add=0)
    if out4.info.get("local_report", {}).get("is_local", False):
        assert len(s4.archive.entries) >= 1

    tr = run_short_controller_episode(inst, num_steps=5, seed=seed)
    for row in tr:
        for k in ["action","reward","current_weight","best_weight","num_directions","failed_minima_count","terminal","terminal_reason"]:
            assert k in row
    print("Q-controller smoke trace:")
    for row in tr:
        print(row)
    print("Q-controller self-tests passed (transition/coherence smoke only; no learning-quality claim).")


if TORCH_AVAILABLE:
    run_q_controller_self_tests()
else:
    msg = "Torch unavailable; cannot run Q-controller self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


## 11. Anytime hybrid solver wrapper

In [ ]:
# @title Anytime hybrid solver wrapper (bounded heuristic search with exact validity checks)
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class HybridSolverConfig:
    M_init: int = 32
    K_add: int = 8
    max_steps: int = 32
    max_descent_steps: int = 8
    solver_time_limit_s: float = 5.0
    use_direction_ranker: bool = False
    use_coefficient_generator: bool = False
    use_q_controller: bool = False
    epsilon: float = 0.2
    seed: int = 0


@dataclass
class HybridSolverTraceEntry:
    step: int
    action: int
    action_name: str
    reward: float
    current_weight: int
    best_weight: int
    num_directions: int
    failed_minima_count: int
    terminal: bool
    terminal_reason: str
    info: dict = field(default_factory=dict)


@dataclass
class HybridSolverResult:
    status: str
    best_u: Optional[np.ndarray]
    best_c: Optional[np.ndarray]
    best_weight: Optional[int]
    found: bool
    trace: list
    final_state: SearchControllerState
    metadata: dict = field(default_factory=dict)


def verify_candidate_solution(A, W, u, c, *, require_threshold: bool = False) -> None:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u, name="u", copy=False)
    c_bin = as_binary_uint8(c, name="c", copy=False)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")
    N, r = A_bin.shape
    if u_bin.shape != (r,):
        raise ValueError(f"u shape mismatch: expected ({r},), got {u_bin.shape}")
    if c_bin.shape != (N,):
        raise ValueError(f"c shape mismatch: expected ({N},), got {c_bin.shape}")
    c_exact = gf2_matvec(A_bin, u_bin)
    if not np.array_equal(c_bin, c_exact):
        raise ValueError("candidate mismatch: c != A u over GF(2)")
    w = hamming_weight(c_bin)
    if w == 0:
        raise ValueError("candidate codeword must be nonzero")
    if require_threshold and w > int(W):
        raise ValueError(f"candidate weight {w} exceeds threshold W={int(W)}")


def action_name(action: int) -> str:
    names = {
        0: "exact_gradient",
        1: "neural_ranked_direction",
        2: "add_random_directions",
        3: "add_neural_directions",
        4: "local_minimum_solver",
        5: "ce_restart",
        6: "ce_update",
        7: "attack_failed_minimum",
    }
    if int(action) not in names:
        raise ValueError(f"invalid action index {action}; expected integer in [0, 7]")
    return names[int(action)]


def default_macro_action_policy(state, action_mask, rng) -> int:
    mask = np.asarray(action_mask, dtype=bool).reshape(-1)
    if mask.shape != (8,):
        raise ValueError(f"action_mask must have shape (8,), got {mask.shape}")
    available = np.flatnonzero(mask)
    if available.size == 0:
        raise ValueError("no available macro actions")

    step = int(state.step)
    if mask[0]:
        return 0
    if (step % 3 == 0) and mask[2]:
        return 2
    if (step % 4 == 0) and mask[6]:
        return 6
    if (step % 5 == 0) and mask[4]:
        return 4
    if mask[7]:
        return 7
    return int(rng.choice(available))


def run_hybrid_solver(inst, config: Optional[HybridSolverConfig] = None, direction_ranker=None, coefficient_generator=None, q_model=None) -> HybridSolverResult:
    cfg = HybridSolverConfig() if config is None else config
    rng = np.random.default_rng(cfg.seed)

    state = initialize_controller_state(inst, M_init=cfg.M_init, seed=cfg.seed)
    trace = []

    for step in range(cfg.max_steps):
        obs = controller_observation(state)
        am = controller_action_mask(
            state,
            direction_ranker=direction_ranker if cfg.use_direction_ranker else None,
            coefficient_generator=coefficient_generator if cfg.use_coefficient_generator else None,
        )

        if cfg.use_q_controller and (q_model is not None):
            action = choose_action_epsilon_greedy(q_model, obs, am, epsilon=cfg.epsilon, rng=rng)
        else:
            action = default_macro_action_policy(state, am, rng)

        sr = controller_step(
            state,
            action,
            rng=rng,
            direction_ranker=direction_ranker if cfg.use_direction_ranker else None,
            coefficient_generator=coefficient_generator if cfg.use_coefficient_generator else None,
            K_add=cfg.K_add,
            max_descent_steps=cfg.max_descent_steps,
            solver_time_limit_s=cfg.solver_time_limit_s,
        )

        if action == 4 and state.last_solver_result is not None:
            verify_solver_result(inst.A, inst.W, state.bank, state.last_solver_result)

        if not np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current)):
            raise RuntimeError("exact invariant violated: c_current != A u_current")

        term = False
        reason = ""
        if state.best_c is not None and hamming_weight(state.best_c) <= inst.W and hamming_weight(state.best_c) > 0:
            term = True
            reason = "found_threshold_solution"
            state.terminal = True
            state.terminal_reason = reason
        elif sr.done:
            term = True
            reason = state.terminal_reason or "controller_done"

        trace.append(HybridSolverTraceEntry(
            step=step,
            action=int(action),
            action_name=action_name(int(action)),
            reward=float(sr.reward),
            current_weight=hamming_weight(state.c_current),
            best_weight=int(state.best_weight),
            num_directions=int(state.bank.D.shape[0]),
            failed_minima_count=len(state.archive.entries),
            terminal=bool(term),
            terminal_reason=str(reason),
            info=dict(sr.info),
        ))

        if term:
            break

    if state.best_c is not None and state.best_weight <= inst.W:
        verify_candidate_solution(inst.A, inst.W, state.best_u, state.best_c, require_threshold=True)
        status = "valid_solution"
        found = True
    elif state.best_c is not None:
        verify_candidate_solution(inst.A, inst.W, state.best_u, state.best_c, require_threshold=False)
        status = "best_found"
        found = False
    else:
        status = "no_solution_found"
        found = False

    return HybridSolverResult(
        status=status,
        best_u=None if state.best_u is None else state.best_u.copy(),
        best_c=None if state.best_c is None else state.best_c.copy(),
        best_weight=None if state.best_c is None else int(state.best_weight),
        found=found,
        trace=trace,
        final_state=state,
        metadata={"heuristic": True, "claims": "bounded anytime search; no optimality or infeasibility certificate"},
    )


In [ ]:
# @title Hybrid solver self-tests (bounded anytime wrapper checks)
def run_hybrid_solver_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # 1) verify_candidate_solution checks.
    A = np.eye(4, dtype=np.uint8)
    u = np.array([1, 0, 1, 0], dtype=np.uint8)
    c = gf2_matvec(A, u)
    verify_candidate_solution(A, 2, u, c, require_threshold=False)
    try:
        verify_candidate_solution(A, 2, np.zeros(4, dtype=np.uint8), np.zeros(4, dtype=np.uint8))
        raise AssertionError("expected zero-vector check to fail")
    except ValueError:
        pass
    try:
        verify_candidate_solution(A, 2, u, np.array([1,1,1,1], dtype=np.uint8))
        raise AssertionError("expected mismatched c check to fail")
    except ValueError:
        pass
    try:
        verify_candidate_solution(A, 1, u, c, require_threshold=True)
        raise AssertionError("expected threshold check to fail")
    except ValueError:
        pass

    # 2) action_name checks.
    names = [action_name(i) for i in range(8)]
    assert len(set(names)) == 8 and all(len(n) > 0 for n in names)
    try:
        action_name(8)
        raise AssertionError("invalid action should raise")
    except ValueError:
        pass

    # 3) default policy respects mask.
    inst = generate_planted_span_instance(N=20, r=8, W=4, seed=seed, hide=True)
    s = initialize_controller_state(inst, M_init=8, seed=seed)
    for _ in range(12):
        mask = controller_action_mask(s)
        a = default_macro_action_policy(s, mask, rng)
        assert mask[a]
        singleton = np.zeros(8, dtype=bool); singleton[6] = True
        assert default_macro_action_policy(s, singleton, rng) == 6
        s.step += 1

    # 4) Tiny planted run.
    inst_small = generate_planted_span_instance(N=24, r=10, W=4, seed=seed + 1, hide=True)
    cfg = HybridSolverConfig(M_init=24, K_add=4, max_steps=8, max_descent_steps=4, solver_time_limit_s=2.0,
                             use_direction_ranker=False, use_coefficient_generator=False, use_q_controller=False, seed=seed+1)
    result = run_hybrid_solver(inst_small, config=cfg)
    assert result.status in {"valid_solution", "best_found", "no_solution_found"}
    assert len(result.trace) <= cfg.max_steps
    assert np.array_equal(result.final_state.c_current, gf2_matvec(inst_small.A, result.final_state.u_current))
    if result.best_c is not None:
        assert np.array_equal(result.best_c, gf2_matvec(inst_small.A, result.best_u))
        assert hamming_weight(result.best_c) > 0
    if result.found:
        assert result.best_weight <= inst_small.W

    # 5) Solver fast path.
    if ORTOOLS_AVAILABLE:
        tiny = generate_planted_span_instance(N=14, r=7, W=3, seed=seed + 2, hide=True)
        s2 = initialize_controller_state(tiny, M_init=6, seed=seed+2)
        out = controller_step(s2, 4, rng=rng, solver_time_limit_s=1.0)
        if out.info.get("solver_found", False):
            verify_candidate_solution(tiny.A, tiny.W, s2.u_current, s2.c_current, require_threshold=True)
    else:
        print("OR-Tools unavailable; skipping solver-fast-path hybrid test.")

    # 6) Trace contract and compact print.
    required = ["step","action","action_name","reward","current_weight","best_weight","num_directions","failed_minima_count","terminal","terminal_reason"]
    for t in result.trace:
        for k in required:
            assert hasattr(t, k)
    print("Hybrid solver compact trace:")
    for t in result.trace:
        print(t.action_name, round(t.reward, 4), t.current_weight, t.best_weight, t.num_directions, t.failed_minima_count, t.terminal)

    # 7) No optimality claim.
    assert result.status != "valid_solution" or result.metadata.get("heuristic", False)
    print("Hybrid solver self-tests passed (valid_solution means verified threshold hit, not certified optimum).")


In [ ]:
run_hybrid_solver_self_tests()

## 12. Small benchmark/evaluation harness

In [ ]:
# @title Small benchmark/evaluation harness (tiny planted cases; evaluation plumbing only)
from dataclasses import dataclass, field
from typing import Optional
import os
import time


@dataclass
class BenchmarkCase:
    name: str
    N: int
    r: int
    W: int
    seed: int
    hide: bool = True
    M_init: int = 24
    K_add: int = 4
    max_steps: int = 8
    max_descent_steps: int = 4
    solver_time_limit_s: float = 2.0
    use_direction_ranker: bool = False
    use_coefficient_generator: bool = False
    use_q_controller: bool = False


@dataclass
class BenchmarkRunResult:
    case: BenchmarkCase
    status: str
    found: bool
    best_weight: Optional[int]
    threshold_W: int
    runtime_s: float
    num_trace_steps: int
    num_directions_final: int
    failed_minima_count: int
    solver_calls: int
    solver_statuses: list
    ce_entropy_final: float
    best_c_verified: bool
    trace: list
    metadata: dict = field(default_factory=dict)


def make_tiny_benchmark_suite(smoke: bool = True) -> list[BenchmarkCase]:
    if smoke:
        return [
            BenchmarkCase(name='smoke_24_10_w4_s0', N=24, r=10, W=4, seed=0),
            BenchmarkCase(name='smoke_32_12_w4_s1', N=32, r=12, W=4, seed=1),
        ]
    return [
        BenchmarkCase(name='tiny_24_10_w3_s0', N=24, r=10, W=3, seed=0),
        BenchmarkCase(name='tiny_32_12_w4_s1', N=32, r=12, W=4, seed=1),
        BenchmarkCase(name='tiny_48_16_w5_s2', N=48, r=16, W=5, seed=2),
    ]


def run_benchmark_case(case: BenchmarkCase) -> BenchmarkRunResult:
    inst = generate_planted_span_instance(N=case.N, r=case.r, W=case.W, seed=case.seed, hide=case.hide)
    cfg = HybridSolverConfig(
        M_init=case.M_init,
        K_add=case.K_add,
        max_steps=case.max_steps,
        max_descent_steps=case.max_descent_steps,
        solver_time_limit_s=case.solver_time_limit_s,
        use_direction_ranker=case.use_direction_ranker,
        use_coefficient_generator=case.use_coefficient_generator,
        use_q_controller=case.use_q_controller,
        seed=case.seed,
    )
    t0 = time.time()
    result = run_hybrid_solver(inst, cfg)
    runtime_s = float(time.time() - t0)

    trace = list(result.trace) if result.trace is not None else []
    solver_calls = 0
    solver_statuses = []
    for entry in trace:
        if getattr(entry, 'action_name', '') == 'local_minimum_solver':
            solver_calls += 1
            info = getattr(entry, 'info', {}) or {}
            solver_statuses.append(info.get('status', 'unknown'))

    fs = result.final_state
    num_directions_final = int(len(getattr(fs, 'direction_bank', []) or []))
    failed_minima_count = int(len(getattr(fs, 'failed_minima_archive', []) or []))
    ce_entropy_final = float(getattr(fs, 'ce_entropy', 0.0) or 0.0)

    best_c_verified = False
    if result.best_c is not None:
        verify_candidate_solution(inst.A, case.W, result.best_u, result.best_c, require_threshold=result.found)
        best_c_verified = True

    return BenchmarkRunResult(
        case=case,
        status=str(result.status),
        found=bool(result.found),
        best_weight=None if result.best_weight is None else int(result.best_weight),
        threshold_W=int(case.W),
        runtime_s=runtime_s,
        num_trace_steps=len(trace),
        num_directions_final=num_directions_final,
        failed_minima_count=failed_minima_count,
        solver_calls=solver_calls,
        solver_statuses=solver_statuses,
        ce_entropy_final=ce_entropy_final,
        best_c_verified=best_c_verified,
        trace=trace,
        metadata={'heuristic': result.status != 'valid_solution'},
    )


def run_benchmark_suite(cases: list[BenchmarkCase]) -> list[BenchmarkRunResult]:
    return [run_benchmark_case(case) for case in cases]


def summarize_benchmark_results(results: list[BenchmarkRunResult]) -> list[dict]:
    rows = []
    for rr in results:
        rows.append({
            'name': rr.case.name,
            'N': rr.case.N,
            'r': rr.case.r,
            'W': rr.case.W,
            'seed': rr.case.seed,
            'status': rr.status,
            'found': rr.found,
            'best_weight': rr.best_weight,
            'threshold_W': rr.threshold_W,
            'runtime_s': rr.runtime_s,
            'num_trace_steps': rr.num_trace_steps,
            'num_directions_final': rr.num_directions_final,
            'failed_minima_count': rr.failed_minima_count,
            'solver_calls': rr.solver_calls,
            'ce_entropy_final': rr.ce_entropy_final,
            'best_c_verified': rr.best_c_verified,
        })
    return rows


def print_benchmark_summary(results: list[BenchmarkRunResult]) -> None:
    rows = summarize_benchmark_results(results)
    if ('pd' in globals()) and (pd is not None):
        print(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


def run_benchmark_harness_self_tests(seed=0) -> None:
    smoke_cases = make_tiny_benchmark_suite(smoke=True)
    assert len(smoke_cases) > 0
    for case in smoke_cases:
        assert case.N > 0 and case.r > 0 and case.W > 0 and case.r <= case.N

    before_tmp = set(os.listdir('/tmp'))
    case = smoke_cases[int(seed) % len(smoke_cases)]
    rr = run_benchmark_case(case)
    assert rr.status in {'valid_solution', 'best_found', 'no_solution_found'}
    if rr.best_weight is not None:
        assert rr.best_c_verified
    assert len(rr.trace) <= case.max_steps
    assert rr.status != 'certified_optimum'

    rows = summarize_benchmark_results([rr])
    assert len(rows) == 1
    required_keys = {
        'name','N','r','W','seed','status','found','best_weight','threshold_W','runtime_s',
        'num_trace_steps','num_directions_final','failed_minima_count','solver_calls',
        'ce_entropy_final','best_c_verified'
    }
    assert required_keys.issubset(rows[0].keys())

    print_benchmark_summary([rr])

    after_tmp = set(os.listdir('/tmp'))
    assert before_tmp == after_tmp, 'benchmark harness must not create files in /tmp'


In [ ]:
run_benchmark_harness_self_tests()

## 13. Baseline comparison and ablation table

In [ ]:
# @title Baseline comparison and ablation table (smoke-safe heuristic ablations only)
@dataclass
class AblationVariant:
    name: str
    description: str
    enabled_actions: tuple
    use_direction_ranker: bool = False
    use_coefficient_generator: bool = False
    use_q_controller: bool = False
    M_init: int = 24
    K_add: int = 4
    max_steps: int = 8
    max_descent_steps: int = 4
    solver_time_limit_s: float = 2.0


@dataclass
class AblationResult:
    case_name: str
    variant_name: str
    status: str
    found: bool
    best_weight: Optional[int]
    threshold_W: int
    runtime_s: float
    num_trace_steps: int
    num_directions_final: int
    failed_minima_count: int
    solver_calls: int
    ce_entropy_final: float
    best_c_verified: bool
    metadata: dict = field(default_factory=dict)


def make_default_ablation_variants(smoke: bool = True) -> list[AblationVariant]:
    base = [
        AblationVariant('gradient_only', 'Exact descent + random direction expansion only.', (0, 2)),
        AblationVariant('ce_restart', 'Exact descent + random directions + CE restart/update.', (0, 2, 5, 6)),
        AblationVariant('solver_assisted', 'Exact descent + random directions + local-minimum solver.', (0, 2, 4)),
        AblationVariant('attack_assisted', 'Exact descent + random directions + failed-minimum attack.', (0, 2, 7)),
        AblationVariant('full_symbolic_hybrid', 'Exact descent + random + solver + CE + failed-minimum attack.', (0, 2, 4, 5, 6, 7)),
    ]
    if TORCH_AVAILABLE:
        base.extend([
            AblationVariant(
                'neural_ranker_untrained',
                'Untrained/random-weight neural ranker smoke plumbing (not a performance model).',
                (0, 1, 2),
                use_direction_ranker=True,
            ),
            AblationVariant(
                'neural_generator_untrained',
                'Untrained/random-weight neural coefficient generator smoke plumbing.',
                (0, 2, 3),
                use_coefficient_generator=True,
            ),
        ])
    if smoke:
        return base[:5]
    return base


def choose_ablation_action(state, action_mask, variant: AblationVariant, rng) -> int:
    allowed = np.zeros(8, dtype=bool)
    for a in variant.enabled_actions:
        if not (0 <= int(a) <= 7):
            raise ValueError(f'Invalid action id in variant {variant.name}: {a}')
        allowed[int(a)] = True
    mask = np.asarray(action_mask, dtype=bool)
    available = np.where(mask & allowed)[0]
    if available.size == 0:
        raise ValueError(f'No enabled actions are currently available for variant={variant.name}.')

    priorities = [
        (0, True),
        (4, state.step % 3 == 0),
        (7, True),
        (6, True),
        (5, True),
        (1, True),
        (3, True),
        (2, True),
    ]
    for action, cond in priorities:
        if cond and action in available:
            return int(action)
    return int(rng.choice(available))


def run_ablation_variant_on_case(case: BenchmarkCase, variant: AblationVariant) -> AblationResult:
    inst = generate_planted_span_instance(N=case.N, r=case.r, W=case.W, seed=case.seed, hide=case.hide)
    rng = np.random.default_rng(case.seed)
    state = initialize_controller_state(inst, M_init=variant.M_init, seed=case.seed)

    direction_ranker = DirectionRanker(inst.A.shape[0]) if (variant.use_direction_ranker and TORCH_AVAILABLE) else None
    coefficient_generator = CoefficientGenerator(inst.A.shape[1], inst.A.shape[0]) if (variant.use_coefficient_generator and TORCH_AVAILABLE) else None

    t0 = time.time()
    trace = []
    found = False
    status = 'no_solution_found'
    solver_calls = 0

    for _ in range(int(variant.max_steps)):
        action_mask = controller_action_mask(state, direction_ranker=direction_ranker, coefficient_generator=coefficient_generator)
        action = choose_ablation_action(state, action_mask, variant, rng)
        if int(action) == 4:
            solver_calls += 1
        step_result = controller_step(
            state,
            action=action,
            rng=rng,
            direction_ranker=direction_ranker,
            coefficient_generator=coefficient_generator,
            K_add=variant.K_add,
            max_descent_steps=variant.max_descent_steps,
            solver_time_limit_s=variant.solver_time_limit_s,
        )
        assert np.array_equal(state.c_current, gf2_matvec(inst.A, state.u_current))
        trace.append({'step': int(state.step), 'action': int(action), 'reward': float(step_result.reward)})

        if state.best_weight <= case.W:
            found = True
            status = 'valid_solution'
            break
        if bool(step_result.done):
            break

    runtime_s = float(time.time() - t0)
    if status != 'valid_solution':
        status = 'best_found' if int(state.best_weight) < inst.A.shape[0] + 1 else 'no_solution_found'

    best_weight = None if int(state.best_weight) >= inst.A.shape[0] + 1 else int(state.best_weight)
    best_c_verified = False
    if state.best_c is not None and state.best_u is not None:
        verify_candidate_solution(inst.A, case.W, state.best_u, state.best_c, require_threshold=found)
        best_c_verified = True

    ce_entropy_attr = getattr(state.ce_sampler, 'entropy', 0.0)
    ce_entropy_final = float(ce_entropy_attr() if callable(ce_entropy_attr) else (ce_entropy_attr or 0.0))

    return AblationResult(
        case_name=case.name,
        variant_name=variant.name,
        status=status,
        found=found,
        best_weight=best_weight,
        threshold_W=int(case.W),
        runtime_s=runtime_s,
        num_trace_steps=len(trace),
        num_directions_final=int(state.bank.D.shape[0]),
        failed_minima_count=int(len(state.archive.entries)),
        solver_calls=solver_calls,
        ce_entropy_final=ce_entropy_final,
        best_c_verified=best_c_verified,
        metadata={
            'heuristic_only': True,  # Ablation outcomes are heuristic and not certified optimum.
            'trace': trace,
        },
    )


def run_ablation_table(cases: Optional[list[BenchmarkCase]] = None, variants: Optional[list[AblationVariant]] = None, smoke: bool = True) -> list[AblationResult]:
    if cases is None:
        cases = make_tiny_benchmark_suite(smoke=smoke)
    if variants is None:
        variants = make_default_ablation_variants(smoke=smoke)

    if smoke:
        cases = list(cases)[:2]
        variants = [v for v in variants if v.name in {'gradient_only', 'ce_restart', 'solver_assisted', 'attack_assisted', 'full_symbolic_hybrid'}]

    results = []
    for case in cases:
        for variant in variants:
            results.append(run_ablation_variant_on_case(case, variant))
    return results


def summarize_ablation_results(results: list[AblationResult]) -> list[dict]:
    return [{
        'case_name': rr.case_name,
        'variant_name': rr.variant_name,
        'status': rr.status,
        'found': rr.found,
        'best_weight': rr.best_weight,
        'threshold_W': rr.threshold_W,
        'runtime_s': rr.runtime_s,
        'num_trace_steps': rr.num_trace_steps,
        'num_directions_final': rr.num_directions_final,
        'failed_minima_count': rr.failed_minima_count,
        'solver_calls': rr.solver_calls,
        'ce_entropy_final': rr.ce_entropy_final,
        'best_c_verified': rr.best_c_verified,
    } for rr in results]


def print_ablation_table(results: list[AblationResult]) -> None:
    rows = summarize_ablation_results(results)
    if ('pd' in globals()) and (pd is not None):
        df = pd.DataFrame(rows).sort_values(['case_name', 'variant_name']).reset_index(drop=True)
        print(df)
    else:
        for row in sorted(rows, key=lambda x: (x['case_name'], x['variant_name'])):
            print(row)


def run_ablation_self_tests(seed=0) -> None:
    before_tmp = set(os.listdir('/tmp'))

    variants = make_default_ablation_variants(smoke=True)
    assert len(variants) > 0
    names = [v.name for v in variants]
    assert len(set(names)) == len(names)
    for variant in variants:
        assert len(variant.enabled_actions) > 0
        assert all(isinstance(a, int) and 0 <= a <= 7 for a in variant.enabled_actions)

    rng = np.random.default_rng(seed)
    state = type('S', (), {'step': 0})()
    mask = np.array([True, False, True, False, True, False, False, False], dtype=bool)
    v = AblationVariant('tmp', 'tmp', (0, 2, 4))
    action = choose_ablation_action(state, mask, v, rng)
    assert action in {0, 2, 4} and bool(mask[action])
    try:
        choose_ablation_action(state, np.zeros(8, dtype=bool), v, rng)
        raise AssertionError('expected ValueError when no action is available')
    except ValueError:
        pass

    tiny_case = BenchmarkCase(name='ablation_tiny', N=24, r=10, W=4, seed=0, max_steps=4)
    run_variants = [AblationVariant('gradient_only', 'g', (0, 2), max_steps=4)]
    if ORTOOLS_AVAILABLE:
        run_variants.append(AblationVariant('solver_assisted', 's', (0, 2, 4), max_steps=4))
    results = [run_ablation_variant_on_case(tiny_case, rv) for rv in run_variants]

    for rr in results:
        assert rr.status in {'valid_solution', 'best_found', 'no_solution_found'}
        if rr.best_weight is not None:
            assert rr.best_c_verified
            trace = rr.metadata.get('trace', [])
            assert isinstance(trace, list)
        assert rr.metadata.get('heuristic_only', False), 'ablation results are heuristic and not certified optimum'

    rows = summarize_ablation_results(results)
    assert len(rows) == len(results)
    required_keys = {
        'case_name','variant_name','status','found','best_weight','threshold_W','runtime_s',
        'num_trace_steps','num_directions_final','failed_minima_count','solver_calls',
        'ce_entropy_final','best_c_verified'
    }
    assert required_keys.issubset(rows[0].keys())

    print_ablation_table(results)

    after_tmp = set(os.listdir('/tmp'))
    assert before_tmp == after_tmp, 'ablation section must not write files to /tmp'


In [ ]:
run_ablation_self_tests()

## 14. Lightweight performance profiling harness

In [ ]:
# @title Lightweight performance profiling harness (smoke-safe baseline timings only)

@dataclass
class TimingResult:
    name: str
    size_label: str
    seconds: float
    metadata: dict = field(default_factory=dict)


@dataclass
class PerformanceProfile:
    results: list
    metadata: dict = field(default_factory=dict)


def _time_call(fn, *args, repeat: int = 3, warmup: int = 1, **kwargs) -> tuple[float, Any]:
    if repeat < 1:
        raise ValueError("repeat must be >= 1")
    if warmup < 0:
        raise ValueError("warmup must be >= 0")

    out = None
    for _ in range(int(warmup)):
        out = fn(*args, **kwargs)

    best = float('inf')
    for _ in range(int(repeat)):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        dt = time.perf_counter() - t0
        best = min(best, float(dt))
    return best, out


def profile_gf2_primitives(seed=0, smoke: bool = True) -> list[TimingResult]:
    rng = np.random.default_rng(seed)
    sizes = [(32, 12), (64, 16)] if smoke else [(64, 16), (96, 24)]
    repeats = 2 if smoke else 3
    out: list[TimingResult] = []

    for N, r in sizes:
        A = rng.integers(0, 2, size=(N, r), dtype=np.uint8)
        u = rng.integers(0, 2, size=(r,), dtype=np.uint8)
        B = rng.integers(0, 2, size=(r, max(2, r // 2)), dtype=np.uint8)

        t, c = _time_call(gf2_matvec, A, u, repeat=repeats, warmup=1)
        c_ref = ((A.astype(np.int64) @ u.astype(np.int64)) % 2).astype(np.uint8)
        assert np.array_equal(c, c_ref)
        out.append(TimingResult("gf2_matvec", f"N={N},r={r}", t))

        t, C = _time_call(gf2_matmul, A, B, repeat=repeats, warmup=1)
        C_ref = ((A.astype(np.int64) @ B.astype(np.int64)) % 2).astype(np.uint8)
        assert np.array_equal(C, C_ref)
        out.append(TimingResult("gf2_matmul", f"N={N},r={r},k={B.shape[1]}", t))

        t, rk = _time_call(gf2_rank, A, repeat=repeats, warmup=1)
        assert 0 <= int(rk) <= min(N, r)
        out.append(TimingResult("gf2_rank", f"N={N},r={r}", t, metadata={"rank": int(rk)}))

        c_rand = rng.integers(0, 2, size=(N,), dtype=np.uint8)
        d_rand = rng.integers(0, 2, size=(N,), dtype=np.uint8)
        t, delta = _time_call(directional_delta, c_rand, d_rand, repeat=repeats, warmup=1)
        delta_ref = int(hamming_weight(d_rand) - 2 * int(np.sum(c_rand & d_rand)))
        assert int(delta) == delta_ref
        out.append(TimingResult("directional_delta", f"N={N}", t))

    inv_sizes = [8, 12]
    for n in inv_sizes:
        while True:
            M = rng.integers(0, 2, size=(n, n), dtype=np.uint8)
            if gf2_rank(M) == n:
                break
        t, Minv = _time_call(gf2_inverse, M, repeat=repeats, warmup=1)
        I = gf2_matmul(M, Minv)
        assert np.array_equal(I, np.eye(n, dtype=np.uint8))
        out.append(TimingResult("gf2_inverse", f"n={n}", t))

    return out


def profile_direction_bank_and_descent(seed=0, smoke: bool = True) -> list[TimingResult]:
    inst = generate_planted_span_instance(N=32, r=12, W=4, seed=seed)
    rng = np.random.default_rng(seed + 1)
    bank = DirectionBank(inst.A)
    ge = GradientEngine()
    M_init = 32 if smoke else 48
    repeats = 2 if smoke else 3

    t_add, _ = _time_call(bank.add_random_span_directions, M_init, rng, repeat=repeats, warmup=1)
    bank.verify()

    u0 = rng.integers(0, 2, size=(inst.A.shape[1],), dtype=np.uint8)
    c0 = gf2_matvec(inst.A, u0)
    t_deltas, vals = _time_call(bank.deltas, c0, repeat=repeats, warmup=1)
    assert vals.shape == (bank.D.shape[0],)

    t_best, idx = _time_call(bank.best_descent, c0, repeat=repeats, warmup=1)
    if idx is not None:
        assert 0 <= int(idx) < bank.D.shape[0]

    t_desc, desc_pair = _time_call(ge.descend, c0, bank, max_steps=6 if smoke else 12, repeat=1, warmup=0)
    c_final, hist = desc_pair
    assert np.isin(c_final, [0, 1]).all()
    assert hamming_weight(c_final) <= hamming_weight(c0)

    out = [
        TimingResult("DirectionBank.add_random_span_directions", "N=32,r=12,M=32", t_add),
        TimingResult("DirectionBank.deltas", "N=32,r=12", t_deltas),
        TimingResult("DirectionBank.best_descent", "N=32,r=12", t_best),
        TimingResult("GradientEngine.descend", "N=32,r=12,steps<=6", t_desc, metadata={"steps": int(len(hist))}),
    ]
    return out


def profile_solver_smoke(seed=0, smoke: bool = True) -> list[TimingResult]:
    if not ORTOOLS_AVAILABLE:
        msg = "OR-Tools unavailable; cannot profile LocalMinimumSolver in smoke/headless mode."
        if HEADLESS or smoke:
            raise RuntimeError(msg)
        print(msg)
        return []

    inst = generate_planted_span_instance(N=20, r=8, W=3, seed=seed)
    rng = np.random.default_rng(seed + 2)
    bank = DirectionBank(inst.A)
    bank.add_random_span_directions(24, rng)
    solver = LocalMinimumSolver(random_seed=seed)

    t, result = _time_call(solver.solve, inst.A, inst.W, bank, time_limit_s=2.0, repeat=1, warmup=0)
    if result.found:
        verify_solver_result(inst.A, inst.W, bank, result)
    return [TimingResult("LocalMinimumSolver.solve", "N=20,r=8,W=3,t<=2s", t, metadata={"found": bool(result.found)})]


def profile_hybrid_wrapper_smoke(seed=0, smoke: bool = True) -> list[TimingResult]:
    inst = generate_planted_span_instance(N=24, r=10, W=4, seed=seed)
    cfg = HybridSolverConfig(
        seed=seed,
        M_init=24,
        K_add=4,
        max_steps=6,
        max_descent_steps=4,
        solver_time_limit_s=2.0,
        use_q_controller=False,
        use_direction_ranker=False,
        use_coefficient_generator=False,
    )
    t, result = _time_call(run_hybrid_solver, inst, cfg, repeat=1, warmup=0)
    assert result.status in {"valid_solution", "best_found", "no_solution_found"}
    if result.best_u is not None and result.best_c is not None and hamming_weight(result.best_c) > 0:
        verify_candidate_solution(inst.A, inst.W, result.best_u, result.best_c, require_threshold=False)
    return [TimingResult("run_hybrid_solver", "N=24,r=10,W=4,steps<=6", t, metadata={"status": result.status})]


def run_performance_profile(seed=0, smoke: Optional[bool] = None) -> PerformanceProfile:
    use_smoke = SMOKE if smoke is None else bool(smoke)
    results: list[TimingResult] = []
    results.extend(profile_gf2_primitives(seed=seed, smoke=use_smoke))
    results.extend(profile_direction_bank_and_descent(seed=seed, smoke=use_smoke))
    results.extend(profile_solver_smoke(seed=seed, smoke=use_smoke))
    results.extend(profile_hybrid_wrapper_smoke(seed=seed, smoke=use_smoke))
    return PerformanceProfile(results=results, metadata={"seed": int(seed), "smoke": bool(use_smoke)})


def summarize_performance_profile(profile: PerformanceProfile) -> list[dict]:
    return [
        {
            "name": tr.name,
            "size_label": tr.size_label,
            "seconds": float(tr.seconds),
            "metadata": dict(tr.metadata),
        }
        for tr in profile.results
    ]


def print_performance_profile(profile: PerformanceProfile) -> None:
    rows = summarize_performance_profile(profile)
    try:
        import pandas as _pd
        cols = ["name", "size_label", "seconds", "metadata"]
        print(_pd.DataFrame(rows, columns=cols).head(24).to_string(index=False))
    except Exception:
        for row in rows[:24]:
            print(row)


def run_performance_profile_self_tests(seed=0) -> None:
    # Timings here are smoke-baseline sanity checks only, not performance claims.
    t, val = _time_call(lambda x: x + 1, 41, repeat=2, warmup=1)
    assert val == 42
    assert t >= 0.0 and np.isfinite(t)

    before_files = set(Path('.').iterdir())
    profile = run_performance_profile(seed=seed, smoke=True)
    after_files = set(Path('.').iterdir())

    assert len(profile.results) > 0
    for tr in profile.results:
        assert isinstance(tr, TimingResult)
        assert np.isfinite(tr.seconds)
        assert tr.seconds >= 0.0

    summary = summarize_performance_profile(profile)
    assert len(summary) == len(profile.results)
    for row in summary:
        for key in ("name", "size_label", "seconds", "metadata"):
            assert key in row

    print_performance_profile(profile)
    assert before_files == after_files


In [ ]:
run_performance_profile_self_tests()

## 15. Packed-bit GF(2) prototype helpers

In [ ]:
# @title Packed-bit GF(2) prototype helpers (experimental, correctness-first)

@dataclass
class PackedBinaryMatrix:
    shape: tuple
    packed: np.ndarray
    bitorder: str = "little"


@dataclass
class PackedBinaryVector:
    length: int
    packed: np.ndarray
    bitorder: str = "little"


def pack_binary_vector(x, *, bitorder: str = "little") -> PackedBinaryVector:
    x_bin = as_binary_uint8(x)
    if x_bin.ndim != 1:
        raise ValueError("x must be a rank-1 binary vector")
    packed = np.packbits(x_bin, bitorder=bitorder)
    return PackedBinaryVector(length=int(x_bin.shape[0]), packed=packed, bitorder=bitorder)


def unpack_binary_vector(px: PackedBinaryVector) -> np.ndarray:
    bits = np.unpackbits(px.packed, bitorder=px.bitorder)
    out = bits[: px.length].astype(np.uint8, copy=False)
    return as_binary_uint8(out)


def pack_binary_matrix_rows(A, *, bitorder: str = "little") -> PackedBinaryMatrix:
    A_bin = as_binary_uint8(A)
    if A_bin.ndim != 2:
        raise ValueError("A must be rank-2")
    packed = np.packbits(A_bin, axis=1, bitorder=bitorder)
    return PackedBinaryMatrix(shape=tuple(A_bin.shape), packed=packed, bitorder=bitorder)


def unpack_binary_matrix_rows(pA: PackedBinaryMatrix) -> np.ndarray:
    r, n = pA.shape
    bits = np.unpackbits(pA.packed, axis=1, bitorder=pA.bitorder)
    out = bits[:, :n].astype(np.uint8, copy=False)
    if out.shape != (r, n):
        out = out.reshape((r, n))
    return as_binary_uint8(out)


def packed_hamming_weight(px: PackedBinaryVector) -> int:
    return int(np.sum(unpack_binary_vector(px), dtype=np.int64))


def packed_support_intersection(px: PackedBinaryVector, py: PackedBinaryVector) -> int:
    if px.length != py.length:
        raise ValueError("packed vectors must have the same length")
    if px.bitorder != py.bitorder:
        raise ValueError("packed vectors must have matching bitorder")
    x = unpack_binary_vector(px)
    y = unpack_binary_vector(py)
    return int(np.sum((x & y).astype(np.uint8), dtype=np.int64))


def packed_directional_delta(pc: PackedBinaryVector, pd: PackedBinaryVector) -> int:
    return packed_hamming_weight(pd) - 2 * packed_support_intersection(pc, pd)


def gf2_matvec_packed_rows(A, u) -> np.ndarray:
    A_bin = as_binary_uint8(A)
    u_bin = as_binary_uint8(u)
    if A_bin.ndim != 2:
        raise ValueError("A must be rank-2")
    if u_bin.ndim != 1:
        raise ValueError("u must be rank-1")
    r, n = A_bin.shape
    if u_bin.shape[0] != n:
        raise ValueError("dimension mismatch in gf2_matvec_packed_rows")

    pA = pack_binary_matrix_rows(A_bin)
    pu = pack_binary_vector(u_bin, bitorder=pA.bitorder)
    out = np.zeros(r, dtype=np.uint8)
    for i in range(r):
        row_packed = PackedBinaryVector(length=n, packed=pA.packed[i], bitorder=pA.bitorder)
        parity = packed_support_intersection(row_packed, pu) & 1
        out[i] = np.uint8(parity)
    return out


def batch_deltas_packed(c, D) -> np.ndarray:
    c_bin = as_binary_uint8(c)
    D_bin = as_binary_uint8(D)
    if c_bin.ndim != 1:
        raise ValueError("c must be rank-1")
    if D_bin.ndim != 2:
        raise ValueError("D must be rank-2")
    m, n = D_bin.shape
    if c_bin.shape[0] != n:
        raise ValueError("dimension mismatch in batch_deltas_packed")
    if m == 0:
        return np.empty((0,), dtype=int)

    pc = pack_binary_vector(c_bin)
    out = np.empty((m,), dtype=int)
    for i in range(m):
        pd = pack_binary_vector(D_bin[i], bitorder=pc.bitorder)
        out[i] = packed_directional_delta(pc, pd)
    return out


def run_packed_bit_helpers_self_tests(seed=0) -> None:
    # Optional prototype helpers: exact uint8 kernels remain authoritative and solver wiring is unchanged.
    rng = np.random.default_rng(seed)

    lengths = [0, 1, 2, 7, 8, 9, 15, 16, 17, 31, 32, 33]
    for n in lengths:
        x = rng.integers(0, 2, size=(n,), dtype=np.uint8)
        px = pack_binary_vector(x)
        xr = unpack_binary_vector(px)
        assert xr.dtype == np.uint8
        assert xr.shape == (n,)
        assert np.array_equal(as_binary_uint8(xr), xr)
        assert np.array_equal(x, xr)

    shapes = [(0, 0), (1, 1), (2, 7), (3, 8), (4, 9), (5, 17)]
    for r, n in shapes:
        A = rng.integers(0, 2, size=(r, n), dtype=np.uint8)
        pA = pack_binary_matrix_rows(A)
        Ar = unpack_binary_matrix_rows(pA)
        assert Ar.dtype == np.uint8
        assert Ar.shape == (r, n)
        assert np.array_equal(A, Ar)

    for n in [1, 7, 8, 9, 17, 33]:
        zero = np.zeros(n, dtype=np.uint8)
        one_hot = np.zeros(n, dtype=np.uint8)
        one_hot[min(1, n - 1)] = 1
        ones = np.ones(n, dtype=np.uint8)
        for x in [zero, one_hot, ones, rng.integers(0, 2, size=n, dtype=np.uint8)]:
            assert packed_hamming_weight(pack_binary_vector(x)) == hamming_weight(x)

    for n in [1, 2, 7, 8, 9, 15, 16, 17, 31, 33]:
        for _ in range(8):
            x = rng.integers(0, 2, size=n, dtype=np.uint8)
            y = rng.integers(0, 2, size=n, dtype=np.uint8)
            assert packed_support_intersection(pack_binary_vector(x), pack_binary_vector(y)) == support_intersection(x, y)

    for n in [1, 7, 8, 9, 16, 17, 31, 33]:
        for _ in range(8):
            c = rng.integers(0, 2, size=n, dtype=np.uint8)
            d = rng.integers(0, 2, size=n, dtype=np.uint8)
            pdelta = packed_directional_delta(pack_binary_vector(c), pack_binary_vector(d))
            assert pdelta == directional_delta(c, d)
            assert pdelta == hamming_weight(c ^ d) - hamming_weight(c)

    for n in [1, 2, 7, 8, 9, 16, 31]:
        for r in [1, 3, 8, 13]:
            A = rng.integers(0, 2, size=(r, n), dtype=np.uint8)
            u = rng.integers(0, 2, size=(n,), dtype=np.uint8)
            y_ref = gf2_matvec(A, u)
            y_packed = gf2_matvec_packed_rows(A, u)
            assert np.array_equal(y_ref, y_packed)

    for n in [1, 7, 8, 9, 17, 31, 33]:
        c = rng.integers(0, 2, size=(n,), dtype=np.uint8)
        D_empty = np.zeros((0, n), dtype=np.uint8)
        empty_deltas = batch_deltas_packed(c, D_empty)
        assert empty_deltas.shape == (0,)
        assert empty_deltas.dtype.kind in ("i", "u")

        D = rng.integers(0, 2, size=(11, n), dtype=np.uint8)
        ref = np.array([directional_delta(c, D[i]) for i in range(D.shape[0])], dtype=int)
        got = batch_deltas_packed(c, D)
        assert np.array_equal(ref, got)

    inst = generate_planted_span_instance(N=20, r=8, W=4, seed=int(rng.integers(0, 10_000)), hide=True)
    A = inst.A
    bank = DirectionBank(A)
    bank.add_random_span_directions(K=16, rng=rng)
    c = gf2_matvec(A, rng.integers(0, 2, size=(A.shape[1],), dtype=np.uint8))
    assert np.array_equal(bank.deltas(c), batch_deltas_packed(c, bank.D))

    A0 = rng.integers(0, 2, size=(6, 9), dtype=np.uint8)
    u0 = rng.integers(0, 2, size=(9,), dtype=np.uint8)
    D0 = rng.integers(0, 2, size=(5, 9), dtype=np.uint8)
    c0 = rng.integers(0, 2, size=(9,), dtype=np.uint8)
    A_before = A0.copy()
    u_before = u0.copy()
    D_before = D0.copy()
    c_before = c0.copy()
    _ = gf2_matvec_packed_rows(A0, u0)
    _ = batch_deltas_packed(c0, D0)
    assert np.array_equal(A0, A_before)
    assert np.array_equal(u0, u_before)
    assert np.array_equal(D0, D_before)
    assert np.array_equal(c0, c_before)

    print("Packed-bit GF(2) prototype helper self-tests passed.")


In [ ]:
run_packed_bit_helpers_self_tests()

## 16. Packed batch-delta comparison harness

In [ ]:
# @title Packed batch-delta comparison harness (validation + measurement only)

@dataclass
class PackedDeltaComparisonCase:
    name: str
    N: int
    r: int
    M: int
    seed: int


@dataclass
class PackedDeltaComparisonResult:
    case: PackedDeltaComparisonCase
    rowwise_seconds: float
    vectorized_seconds: float
    packed_seconds: float
    packed_vs_rowwise_speedup: float
    packed_vs_vectorized_speedup: float
    vectorized_vs_rowwise_speedup: float
    exact_match: bool
    max_abs_difference: int
    metadata: dict = field(default_factory=dict)


def make_packed_delta_comparison_cases(smoke: bool = True) -> list[PackedDeltaComparisonCase]:
    if smoke:
        return [
            PackedDeltaComparisonCase(name="smoke_n32_r12_m32", N=32, r=12, M=32, seed=101),
            PackedDeltaComparisonCase(name="smoke_n64_r16_m64", N=64, r=16, M=64, seed=102),
            PackedDeltaComparisonCase(name="smoke_n65_r18_m64", N=65, r=18, M=64, seed=103),
            PackedDeltaComparisonCase(name="smoke_n96_r20_m96", N=96, r=20, M=96, seed=104),
        ]
    return [
        PackedDeltaComparisonCase(name="bench_n64_r16_m64", N=64, r=16, M=64, seed=201),
        PackedDeltaComparisonCase(name="bench_n96_r24_m128", N=96, r=24, M=128, seed=202),
        PackedDeltaComparisonCase(name="bench_n128_r28_m256", N=128, r=28, M=256, seed=203),
        PackedDeltaComparisonCase(name="bench_n192_r40_m256", N=192, r=40, M=256, seed=204),
    ]


def reference_batch_deltas(c, D) -> np.ndarray:
    c_arr = as_binary_uint8(c)
    D_arr = as_binary_uint8(D)
    if c_arr.ndim != 1:
        raise ValueError("c must have shape [N]")
    if D_arr.ndim != 2:
        raise ValueError("D must have shape [M, N]")
    if D_arr.shape[1] != c_arr.shape[0]:
        raise ValueError("D.shape[1] must equal len(c)")
    if D_arr.shape[0] == 0:
        return np.empty((0,), dtype=np.int64)
    return np.array([directional_delta(c_arr, D_arr[i]) for i in range(D_arr.shape[0])], dtype=np.int64)


def vectorized_bank_deltas(c, bank) -> np.ndarray:
    c_arr = as_binary_uint8(c)
    if c_arr.ndim != 1:
        raise ValueError("c must have shape [N]")
    vec = np.asarray(bank.deltas(c_arr))
    if vec.ndim != 1:
        raise ValueError("bank.deltas(c) must return shape [M]")
    if vec.shape[0] != bank.D.shape[0]:
        raise ValueError("bank.deltas(c) length must match number of directions")
    if not np.issubdtype(vec.dtype, np.integer):
        raise ValueError("bank.deltas(c) must return an integer dtype")
    return vec.astype(np.int64, copy=False)


def run_packed_delta_comparison_case(case: PackedDeltaComparisonCase) -> PackedDeltaComparisonResult:
    rng = np.random.default_rng(case.seed)
    A = gf2_random_matrix(case.N, case.r, rng=rng)
    bank = DirectionBank(A)

    for _ in range(case.M):
        coeff = rng.integers(0, 2, size=case.r, dtype=np.uint8)
        bank.add_one(coeff)

    u = rng.integers(0, 2, size=case.r, dtype=np.uint8)
    c = gf2_matvec(A, u)

    rowwise = reference_batch_deltas(c, bank.D)
    vectorized = vectorized_bank_deltas(c, bank)
    packed = batch_deltas_packed(c, bank.D)

    assert np.array_equal(rowwise, vectorized), "rowwise reference must match DirectionBank.deltas"
    assert np.array_equal(rowwise, packed), "rowwise reference must match packed prototype"

    rowwise_seconds, _ = _time_call(reference_batch_deltas, c, bank.D, repeat=3, warmup=1)
    vectorized_seconds, _ = _time_call(vectorized_bank_deltas, c, bank, repeat=3, warmup=1)
    packed_seconds, _ = _time_call(batch_deltas_packed, c, bank.D, repeat=3, warmup=1)

    exact_match = np.array_equal(rowwise, vectorized) and np.array_equal(rowwise, packed)
    if rowwise.size == 0:
        max_abs_difference = 0
    else:
        max_abs_difference = int(np.max(np.abs(rowwise.astype(np.int64) - packed.astype(np.int64))))

    packed_vs_rowwise_speedup = float(rowwise_seconds / packed_seconds) if packed_seconds > 0 else float(np.inf)
    packed_vs_vectorized_speedup = float(vectorized_seconds / packed_seconds) if packed_seconds > 0 else float(np.inf)
    vectorized_vs_rowwise_speedup = float(rowwise_seconds / vectorized_seconds) if vectorized_seconds > 0 else float(np.inf)

    return PackedDeltaComparisonResult(
        case=case,
        rowwise_seconds=float(rowwise_seconds),
        vectorized_seconds=float(vectorized_seconds),
        packed_seconds=float(packed_seconds),
        packed_vs_rowwise_speedup=packed_vs_rowwise_speedup,
        packed_vs_vectorized_speedup=packed_vs_vectorized_speedup,
        vectorized_vs_rowwise_speedup=vectorized_vs_rowwise_speedup,
        exact_match=bool(exact_match),
        max_abs_difference=max_abs_difference,
        metadata={"benchmark_only": True},
    )


def run_packed_delta_comparison_suite(
    cases: Optional[list[PackedDeltaComparisonCase]] = None,
    smoke: bool = True,
) -> list[PackedDeltaComparisonResult]:
    active_cases = cases if cases is not None else make_packed_delta_comparison_cases(smoke=smoke)
    return [run_packed_delta_comparison_case(case) for case in active_cases]


def summarize_packed_delta_comparisons(results: list[PackedDeltaComparisonResult]) -> list[dict]:
    rows = []
    for res in results:
        rows.append({
            "name": res.case.name,
            "N": res.case.N,
            "r": res.case.r,
            "M": res.case.M,
            "rowwise_seconds": res.rowwise_seconds,
            "vectorized_seconds": res.vectorized_seconds,
            "packed_seconds": res.packed_seconds,
            "packed_vs_rowwise_speedup": res.packed_vs_rowwise_speedup,
            "packed_vs_vectorized_speedup": res.packed_vs_vectorized_speedup,
            "vectorized_vs_rowwise_speedup": res.vectorized_vs_rowwise_speedup,
            "exact_match": res.exact_match,
            "max_abs_difference": res.max_abs_difference,
        })
    return rows


def print_packed_delta_comparison_table(results: list[PackedDeltaComparisonResult]) -> None:
    rows = summarize_packed_delta_comparisons(results)
    if "pd" in globals() and globals()["pd"] is not None:
        print(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)


def run_packed_delta_comparison_self_tests(seed=0) -> None:
    # Benchmark-only harness: this validates optional packed prototype timing and does not replace default solver kernels.
    smoke_cases = make_packed_delta_comparison_cases(smoke=True)
    assert smoke_cases, "Expected non-empty smoke cases"
    assert any((c.N % 8) != 0 for c in smoke_cases), "Need at least one non-byte-aligned N"

    rng = np.random.default_rng(seed)
    N = 17
    c = rng.integers(0, 2, size=N, dtype=np.uint8)
    D = rng.integers(0, 2, size=(9, N), dtype=np.uint8)
    ref = reference_batch_deltas(c, D)
    rowwise = np.array([directional_delta(c, D[i]) for i in range(D.shape[0])], dtype=np.int64)
    assert np.array_equal(ref, rowwise)

    A = gf2_random_matrix(N, 7, rng=rng)
    bank = DirectionBank(A)
    for _ in range(5):
        bank.add_one(rng.integers(0, 2, size=7, dtype=np.uint8))
    c_bank = gf2_matvec(A, rng.integers(0, 2, size=7, dtype=np.uint8))
    vec = vectorized_bank_deltas(c_bank, bank)
    assert np.array_equal(vec, np.asarray(bank.deltas(c_bank), dtype=np.int64))

    D_empty = np.empty((0, N), dtype=np.uint8)
    ref_empty = reference_batch_deltas(c, D_empty)
    packed_empty = batch_deltas_packed(c, D_empty)
    assert ref_empty.dtype == np.int64 and ref_empty.shape == (0,)
    assert packed_empty.dtype == np.int64 and packed_empty.shape == (0,)
    assert np.array_equal(ref_empty, packed_empty)

    case_result = run_packed_delta_comparison_case(smoke_cases[0])
    assert case_result.exact_match is True
    assert case_result.max_abs_difference == 0
    assert np.isfinite(case_result.rowwise_seconds) and case_result.rowwise_seconds >= 0.0
    assert np.isfinite(case_result.vectorized_seconds) and case_result.vectorized_seconds >= 0.0
    assert np.isfinite(case_result.packed_seconds) and case_result.packed_seconds >= 0.0
    assert (np.isfinite(case_result.packed_vs_rowwise_speedup) and case_result.packed_vs_rowwise_speedup >= 0.0) or np.isinf(case_result.packed_vs_rowwise_speedup)
    assert (np.isfinite(case_result.packed_vs_vectorized_speedup) and case_result.packed_vs_vectorized_speedup >= 0.0) or np.isinf(case_result.packed_vs_vectorized_speedup)
    assert (np.isfinite(case_result.vectorized_vs_rowwise_speedup) and case_result.vectorized_vs_rowwise_speedup >= 0.0) or np.isinf(case_result.vectorized_vs_rowwise_speedup)

    summary = summarize_packed_delta_comparisons([case_result])
    required_keys = {
        "name", "N", "r", "M", "rowwise_seconds", "vectorized_seconds", "packed_seconds",
        "packed_vs_rowwise_speedup", "packed_vs_vectorized_speedup", "vectorized_vs_rowwise_speedup",
        "exact_match", "max_abs_difference"
    }
    assert required_keys.issubset(summary[0].keys())

    print_packed_delta_comparison_table([case_result])

In [ ]:
run_packed_delta_comparison_self_tests()

## 17. Fast packed-popcount delta prototype

In [ ]:
# @title Fast packed-popcount delta prototype (experimental, benchmark-only)

_POPCOUNT_U8_TABLE = None


def popcount_u8_table() -> np.ndarray:
    global _POPCOUNT_U8_TABLE
    if _POPCOUNT_U8_TABLE is None:
        vals = np.arange(256, dtype=np.uint8)
        _POPCOUNT_U8_TABLE = np.unpackbits(vals[:, None], axis=1, bitorder="little").sum(axis=1).astype(np.uint8)
    return _POPCOUNT_U8_TABLE


def packed_hamming_weight_fast(px: PackedBinaryVector) -> int:
    if px.length == 0:
        return 0
    table = popcount_u8_table()
    return int(table[px.packed].sum(dtype=np.int64))


def packed_support_intersection_fast(px: PackedBinaryVector, py: PackedBinaryVector) -> int:
    if px.length != py.length:
        raise ValueError("packed vectors must share the same length")
    if px.bitorder != py.bitorder:
        raise ValueError("packed vectors must share the same bitorder")
    if px.length == 0:
        return 0
    table = popcount_u8_table()
    return int(table[np.bitwise_and(px.packed, py.packed)].sum(dtype=np.int64))


def packed_directional_delta_fast(pc: PackedBinaryVector, pd: PackedBinaryVector) -> int:
    return packed_hamming_weight_fast(pd) - 2 * packed_support_intersection_fast(pc, pd)


def batch_deltas_packed_fast(c, D) -> np.ndarray:
    c_bin = as_binary_uint8(c)
    D_bin = as_binary_uint8(D)
    if c_bin.ndim != 1:
        raise ValueError("c must be rank-1")
    if D_bin.ndim != 2:
        raise ValueError("D must be rank-2")
    if D_bin.shape[1] != c_bin.shape[0]:
        raise ValueError("D width must match c length")
    M, N = D_bin.shape
    if M == 0:
        return np.zeros((0,), dtype=np.int64)

    pc = pack_binary_vector(c_bin)
    pD = pack_binary_matrix_rows(D_bin)
    table = popcount_u8_table()

    direction_weights = table[pD.packed].sum(axis=1, dtype=np.int64)
    intersections = table[np.bitwise_and(pD.packed, pc.packed[None, :])].sum(axis=1, dtype=np.int64)
    deltas = direction_weights - 2 * intersections
    return np.asarray(deltas, dtype=np.int64)


def compare_delta_backends_extended(c, D, bank=None) -> dict:
    rowwise = reference_batch_deltas(c, D)
    if bank is None:
        N = as_binary_uint8(c).shape[0]
        bank_local = DirectionBank(np.eye(N, dtype=np.uint8))
        for d_row in as_binary_uint8(D):
            bank_local.add_one(d_row)
        vectorized = bank_local.deltas(c)
    else:
        vectorized = bank.deltas(c)
    packed_slow = batch_deltas_packed(c, D)
    packed_fast = batch_deltas_packed_fast(c, D)

    exact_match = bool(
        np.array_equal(rowwise, vectorized)
        and np.array_equal(rowwise, packed_slow)
        and np.array_equal(rowwise, packed_fast)
    )
    max_abs_difference = int(
        np.max(np.abs(np.concatenate([
            (rowwise - vectorized).astype(np.int64),
            (rowwise - packed_slow).astype(np.int64),
            (rowwise - packed_fast).astype(np.int64),
        ]))) if rowwise.size else 0
    )

    rowwise_seconds, _ = _time_call(lambda: reference_batch_deltas(c, D), repeat=3, warmup=1)
    if bank is None:
        N = as_binary_uint8(c).shape[0]
        bank_local_for_time = DirectionBank(np.eye(N, dtype=np.uint8))
        for d_row in as_binary_uint8(D):
            bank_local_for_time.add_one(d_row)
        vectorized_fn = lambda: bank_local_for_time.deltas(c)
    else:
        vectorized_fn = lambda: bank.deltas(c)
    vectorized_seconds, _ = _time_call(vectorized_fn, repeat=3, warmup=1)
    packed_slow_seconds, _ = _time_call(lambda: batch_deltas_packed(c, D), repeat=3, warmup=1)
    packed_fast_seconds, _ = _time_call(lambda: batch_deltas_packed_fast(c, D), repeat=3, warmup=1)

    return {
        "rowwise_seconds": float(rowwise_seconds),
        "vectorized_seconds": float(vectorized_seconds),
        "packed_slow_seconds": float(packed_slow_seconds),
        "packed_fast_seconds": float(packed_fast_seconds),
        "packed_fast_vs_vectorized_speedup": float(vectorized_seconds / packed_fast_seconds) if packed_fast_seconds > 0 else float("inf"),
        "packed_fast_vs_slow_speedup": float(packed_slow_seconds / packed_fast_seconds) if packed_fast_seconds > 0 else float("inf"),
        "exact_match": exact_match,
        "max_abs_difference": max_abs_difference,
    }


def run_fast_packed_popcount_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    table = popcount_u8_table()
    assert isinstance(table, np.ndarray)
    assert table.shape == (256,)
    assert int(table[0]) == 0 and int(table[255]) == 8 and int(table[1]) == 1 and int(table[128]) == 1

    lengths = [0, 1, 7, 8, 9, 15, 16, 17, 31, 32, 33, 65, 97]
    for N in lengths:
        for _ in range(6):
            x = rng.integers(0, 2, size=N, dtype=np.uint8)
            px = pack_binary_vector(x)
            assert packed_hamming_weight_fast(px) == hamming_weight(x)

            y = rng.integers(0, 2, size=N, dtype=np.uint8)
            py = pack_binary_vector(y)
            assert packed_support_intersection_fast(px, py) == support_intersection(x, y)
            assert packed_directional_delta_fast(px, py) == directional_delta(x, y)
            assert packed_directional_delta_fast(px, py) == hamming_weight(np.bitwise_xor(x, y)) - hamming_weight(x)

    for N in [65, 97]:
        for M in [0, 1, 32]:
            c = rng.integers(0, 2, size=N, dtype=np.uint8)
            D = rng.integers(0, 2, size=(M, N), dtype=np.uint8)
            c_before = c.copy()
            D_before = D.copy()

            got = batch_deltas_packed_fast(c, D)
            assert got.shape == (M,)
            assert got.dtype == np.int64

            ref = reference_batch_deltas(c, D)
            assert np.array_equal(got, ref)

            if M > 0:
                A_eye = np.eye(N, dtype=np.uint8)
                bank = DirectionBank(A_eye)
                for d_row in D:
                    bank.add_one(d_row)
                assert np.array_equal(got, bank.deltas(c).astype(np.int64))

            assert np.array_equal(c, c_before)
            assert np.array_equal(D, D_before)

    N, M = 65, 32
    c = rng.integers(0, 2, size=N, dtype=np.uint8)
    D = rng.integers(0, 2, size=(M, N), dtype=np.uint8)
    comparison = compare_delta_backends_extended(c, D, bank=None)
    assert comparison["exact_match"] is True
    assert comparison["max_abs_difference"] == 0
    for key in ["rowwise_seconds", "vectorized_seconds", "packed_slow_seconds", "packed_fast_seconds"]:
        assert np.isfinite(comparison[key]) and comparison[key] >= 0.0

    # Experimental benchmark-only path: this does not replace solver defaults.
    print({k: comparison[k] for k in [
        "rowwise_seconds",
        "vectorized_seconds",
        "packed_slow_seconds",
        "packed_fast_seconds",
        "packed_fast_vs_vectorized_speedup",
        "packed_fast_vs_slow_speedup",
        "exact_match",
        "max_abs_difference",
    ]})



def run_extended_delta_backend_comparison_suite(
    cases: Optional[list[PackedDeltaComparisonCase]] = None,
    smoke: bool = True,
) -> list[dict]:
    if cases is None:
        if smoke:
            cases = [
                PackedDeltaComparisonCase(name="smoke_n32_r12_m32", N=32, r=12, M=32, seed=301),
                PackedDeltaComparisonCase(name="smoke_n65_r16_m64", N=65, r=16, M=64, seed=302),
            ]
        else:
            cases = [
                PackedDeltaComparisonCase(name="n32_r12_m32", N=32, r=12, M=32, seed=301),
                PackedDeltaComparisonCase(name="n65_r16_m64", N=65, r=16, M=64, seed=302),
                PackedDeltaComparisonCase(name="n97_r20_m96", N=97, r=20, M=96, seed=303),
            ]

    results = []
    for case in cases:
        rng = np.random.default_rng(case.seed)
        A = gf2_random_matrix(case.N, case.r, rng=rng)
        bank = DirectionBank(A)
        for _ in range(case.M):
            coeff = rng.integers(0, 2, size=case.r, dtype=np.uint8)
            bank.add_one(coeff)
        u = rng.integers(0, 2, size=case.r, dtype=np.uint8)
        c = gf2_matvec(A, u)
        D = bank.D
        metrics = compare_delta_backends_extended(c, D, bank=bank)
        results.append({
            "name": case.name,
            "N": int(case.N),
            "r": int(case.r),
            "M": int(case.M),
            **metrics,
        })
    return results


def summarize_extended_delta_backend_comparisons(results) -> list[dict]:
    summary = []
    for row in results:
        rowwise = float(row["rowwise_seconds"])
        vectorized = float(row["vectorized_seconds"])
        packed_slow = float(row["packed_slow_seconds"])
        packed_fast = float(row["packed_fast_seconds"])
        summary.append({
            "name": row["name"],
            "N": int(row["N"]),
            "r": int(row["r"]),
            "M": int(row["M"]),
            "rowwise_seconds": rowwise,
            "vectorized_seconds": vectorized,
            "packed_slow_seconds": packed_slow,
            "packed_fast_seconds": packed_fast,
            "vectorized_vs_rowwise_speedup": (rowwise / vectorized) if vectorized > 0 else float("inf"),
            "packed_slow_vs_vectorized_speedup": (vectorized / packed_slow) if packed_slow > 0 else float("inf"),
            "packed_fast_vs_vectorized_speedup": (vectorized / packed_fast) if packed_fast > 0 else float("inf"),
            "packed_fast_vs_slow_speedup": (packed_slow / packed_fast) if packed_fast > 0 else float("inf"),
            "exact_match": bool(row["exact_match"]),
            "max_abs_difference": int(row["max_abs_difference"]),
        })
    return summary


def print_extended_delta_backend_comparison_table(results) -> None:
    summary = summarize_extended_delta_backend_comparisons(results)
    columns = [
        "name", "N", "r", "M", "rowwise_seconds", "vectorized_seconds",
        "packed_slow_seconds", "packed_fast_seconds", "vectorized_vs_rowwise_speedup",
        "packed_slow_vs_vectorized_speedup", "packed_fast_vs_vectorized_speedup",
        "packed_fast_vs_slow_speedup", "exact_match", "max_abs_difference",
    ]
    header = " | ".join(columns)
    print(header)
    print("-" * len(header))
    for row in summary:
        values = []
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                values.append(f"{val:.6g}")
            else:
                values.append(str(val))
        print(" | ".join(values))


def run_extended_delta_backend_comparison_self_tests(seed=0) -> None:
    _ = seed
    results = run_extended_delta_backend_comparison_suite(smoke=True)
    assert isinstance(results, list) and len(results) > 0
    assert any((int(r["N"]) % 8) != 0 for r in results)

    required_timing_keys = ["rowwise_seconds", "vectorized_seconds", "packed_slow_seconds", "packed_fast_seconds"]
    for row in results:
        assert row["exact_match"] is True
        assert int(row["max_abs_difference"]) == 0
        for key in required_timing_keys:
            assert np.isfinite(row[key]) and float(row[key]) >= 0.0

    summary = summarize_extended_delta_backend_comparisons(results)
    required_summary_keys = {
        "name", "N", "r", "M", "rowwise_seconds", "vectorized_seconds", "packed_slow_seconds",
        "packed_fast_seconds", "vectorized_vs_rowwise_speedup", "packed_slow_vs_vectorized_speedup",
        "packed_fast_vs_vectorized_speedup", "packed_fast_vs_slow_speedup", "exact_match", "max_abs_difference",
    }
    for row in summary:
        assert required_summary_keys.issubset(set(row.keys()))

    # Benchmark-only instrumentation check: this does not replace default solver kernels.
    print_extended_delta_backend_comparison_table(results)


In [ ]:
run_extended_delta_backend_comparison_self_tests()
run_fast_packed_popcount_self_tests()

## 18. Delta backend scaling study

In [ ]:
# @title Delta backend scaling study (benchmark-only, solver defaults unchanged)

from dataclasses import dataclass, field
from typing import Optional


@dataclass
class DeltaScalingCase:
    name: str
    N: int
    r: int
    M: int
    seed: int


@dataclass
class DeltaScalingResult:
    case: DeltaScalingCase
    rowwise_seconds: float
    vectorized_seconds: float
    packed_slow_seconds: float
    packed_fast_seconds: float
    vectorized_vs_rowwise_speedup: float
    packed_fast_vs_vectorized_speedup: float
    packed_fast_vs_slow_speedup: float
    exact_match: bool
    max_abs_difference: int
    metadata: dict = field(default_factory=dict)


def _safe_speedup(baseline_seconds: float, candidate_seconds: float) -> float:
    if candidate_seconds == 0.0:
        return np.inf
    return baseline_seconds / candidate_seconds


def make_delta_scaling_cases(smoke: bool = True) -> list[DeltaScalingCase]:
    if smoke:
        return [
            DeltaScalingCase(name='smoke_n65_r16_m64', N=65, r=16, M=64, seed=101),
            DeltaScalingCase(name='smoke_n129_r24_m128', N=129, r=24, M=128, seed=102),
            DeltaScalingCase(name='smoke_n257_r32_m128', N=257, r=32, M=128, seed=103),
        ]
    return [
        DeltaScalingCase(name='std_n129_r24_m128', N=129, r=24, M=128, seed=201),
        DeltaScalingCase(name='std_n257_r32_m256', N=257, r=32, M=256, seed=202),
        DeltaScalingCase(name='std_n513_r48_m512', N=513, r=48, M=512, seed=203),
    ]


def run_delta_scaling_case(case: DeltaScalingCase) -> DeltaScalingResult:
    rng = np.random.default_rng(case.seed)
    A = gf2_random_matrix(case.N, case.r, rng)
    bank = DirectionBank(A)

    for _ in range(case.M):
        u_d = rng.integers(0, 2, size=case.r, dtype=np.uint8)
        bank.add_one(u_d)

    u = rng.integers(0, 2, size=case.r, dtype=np.uint8)
    c = gf2_matvec(A, u)

    rowwise = reference_batch_deltas(c, bank.D)
    vectorized = bank.deltas(c)
    packed_slow = batch_deltas_packed(c, bank.D)
    packed_fast = batch_deltas_packed_fast(c, bank.D)

    stacked = np.vstack([rowwise, vectorized, packed_slow, packed_fast]).astype(np.int64)
    max_abs_difference = int(np.max(np.abs(stacked - stacked[0]))) if stacked.size else 0
    exact_match = bool(
        np.array_equal(rowwise, vectorized)
        and np.array_equal(rowwise, packed_slow)
        and np.array_equal(rowwise, packed_fast)
    )
    if not exact_match:
        raise AssertionError('Delta backend mismatch in scaling study case.')

    timing_cfg = dict(repeat=3, warmup=1)
    rowwise_seconds, _ = _time_call(lambda: reference_batch_deltas(c, bank.D), **timing_cfg)
    vectorized_seconds, _ = _time_call(lambda: bank.deltas(c), **timing_cfg)
    packed_slow_seconds, _ = _time_call(lambda: batch_deltas_packed(c, bank.D), **timing_cfg)
    packed_fast_seconds, _ = _time_call(lambda: batch_deltas_packed_fast(c, bank.D), **timing_cfg)

    return DeltaScalingResult(
        case=case,
        rowwise_seconds=rowwise_seconds,
        vectorized_seconds=vectorized_seconds,
        packed_slow_seconds=packed_slow_seconds,
        packed_fast_seconds=packed_fast_seconds,
        vectorized_vs_rowwise_speedup=_safe_speedup(rowwise_seconds, vectorized_seconds),
        packed_fast_vs_vectorized_speedup=_safe_speedup(vectorized_seconds, packed_fast_seconds),
        packed_fast_vs_slow_speedup=_safe_speedup(packed_slow_seconds, packed_fast_seconds),
        exact_match=exact_match,
        max_abs_difference=max_abs_difference,
        metadata={
            'benchmark_only': True,
            'note': 'Scaling study does not replace solver kernels or defaults.',
        },
    )


def run_delta_scaling_study(
    cases: Optional[list[DeltaScalingCase]] = None,
    smoke: bool = True,
) -> list[DeltaScalingResult]:
    use_cases = make_delta_scaling_cases(smoke=smoke) if cases is None else cases
    return [run_delta_scaling_case(case) for case in use_cases]


def summarize_delta_scaling_results(results: list[DeltaScalingResult]) -> list[dict]:
    rows = []
    for res in results:
        rows.append({
            'name': res.case.name,
            'N': res.case.N,
            'r': res.case.r,
            'M': res.case.M,
            'rowwise_seconds': res.rowwise_seconds,
            'vectorized_seconds': res.vectorized_seconds,
            'packed_slow_seconds': res.packed_slow_seconds,
            'packed_fast_seconds': res.packed_fast_seconds,
            'vectorized_vs_rowwise_speedup': res.vectorized_vs_rowwise_speedup,
            'packed_fast_vs_vectorized_speedup': res.packed_fast_vs_vectorized_speedup,
            'packed_fast_vs_slow_speedup': res.packed_fast_vs_slow_speedup,
            'exact_match': res.exact_match,
            'max_abs_difference': res.max_abs_difference,
        })
    return rows


def print_delta_scaling_table(results: list[DeltaScalingResult]) -> None:
    rows = summarize_delta_scaling_results(results)
    if ('pd' in globals()):
        display(pd.DataFrame(rows))
        return
    for row in rows:
        print(row)


def run_delta_scaling_self_tests(seed=0) -> None:
    _ = seed
    cases = make_delta_scaling_cases(smoke=True)
    assert len(cases) > 0
    assert any((case.N % 8) != 0 for case in cases)

    smallest_case = min(cases, key=lambda c: (c.N, c.M, c.r))
    result = run_delta_scaling_case(smallest_case)

    assert result.exact_match is True
    assert result.max_abs_difference == 0

    timings = [
        result.rowwise_seconds,
        result.vectorized_seconds,
        result.packed_slow_seconds,
        result.packed_fast_seconds,
    ]
    assert all(np.isfinite(t) and t >= 0.0 for t in timings)
    assert np.isfinite(result.packed_fast_vs_vectorized_speedup) or np.isinf(result.packed_fast_vs_vectorized_speedup)

    rows = summarize_delta_scaling_results([result])
    required_keys = {
        'name', 'N', 'r', 'M',
        'rowwise_seconds', 'vectorized_seconds', 'packed_slow_seconds', 'packed_fast_seconds',
        'vectorized_vs_rowwise_speedup', 'packed_fast_vs_vectorized_speedup', 'packed_fast_vs_slow_speedup',
        'exact_match', 'max_abs_difference',
    }
    assert required_keys.issubset(rows[0].keys())

    print_delta_scaling_table([result])

    # Benchmark-only guardrail: this study validates timing/equality and does not replace solver kernels.
    assert result.metadata.get('benchmark_only', False) is True


In [ ]:
run_delta_scaling_self_tests()

## 19. Supervised neural-guidance data generator

In [ ]:
# @title Supervised neural-guidance data generator (tiny supervised plumbing checks only)

@dataclass
class RankerTrainingExample:
    direction_features: np.ndarray
    global_features: np.ndarray
    target_scores: np.ndarray
    valid_descent_mask: np.ndarray
    deltas: np.ndarray
    metadata: dict = field(default_factory=dict)


@dataclass
class CoefficientTrainingExample:
    basis_features: np.ndarray
    global_features: np.ndarray
    target_v: np.ndarray
    u_current: np.ndarray
    c_current: np.ndarray
    metadata: dict = field(default_factory=dict)


@dataclass
class NeuralGuidanceDataset:
    ranker_examples: list
    coefficient_examples: list
    metadata: dict = field(default_factory=dict)


def make_ranker_training_example(inst, bank, u_current) -> RankerTrainingExample:
    inst.verify()
    bank.verify()
    assert np.array_equal(bank.A, inst.A), "bank.A must equal inst.A"
    r = inst.A.shape[1]
    u = np.asarray(u_current, dtype=np.uint8).reshape(-1)
    assert u.shape == (r,), "u_current must have shape [r]"
    assert np.all((u == 0) | (u == 1)), "u_current must be binary"
    c_current = inst.matvec(u)

    direction_features = bank.features_for_ranker(c_current).astype(np.float32, copy=False)
    global_features = global_features_for_ranker(inst.A, inst.W, c_current, bank).astype(np.float32, copy=False)

    deltas = bank.deltas(c_current).astype(np.int32, copy=False)
    next_states = c_current[None, :] ^ bank.D
    nonzero_target = np.any(next_states != 0, axis=1)
    valid_descent_mask = (deltas < 0) & nonzero_target

    N = int(inst.A.shape[0])
    target_scores = (-deltas.astype(np.float32)) / float(max(1, N))
    target_scores = target_scores.astype(np.float32, copy=False)
    target_scores = np.where(nonzero_target, target_scores, np.float32(-1.0))

    return RankerTrainingExample(
        direction_features=direction_features,
        global_features=global_features,
        target_scores=target_scores,
        valid_descent_mask=valid_descent_mask.astype(bool, copy=False),
        deltas=deltas,
        metadata={"u_current": u.copy(), "c_current": c_current.copy(), "inst": inst, "bank": bank},
    )


def make_coefficient_training_example(inst, u_current, bank=None) -> CoefficientTrainingExample:
    inst.verify()
    r = inst.A.shape[1]
    u = np.asarray(u_current, dtype=np.uint8).reshape(-1)
    assert u.shape == (r,), "u_current must have shape [r]"
    assert np.all((u == 0) | (u == 1)), "u_current must be binary"

    c_current = inst.matvec(u)
    target_v = planted_coefficient_target(inst, u)
    lhs = gf2_matvec(inst.A, target_v)
    rhs = c_current ^ inst.c_star
    assert np.array_equal(lhs, rhs), "exact identity A @ target_v == c_current XOR c_star failed"

    basis_features = basis_features_for_generator(inst.A, u, c_current).astype(np.float32, copy=False)
    global_features = global_features_for_generator(inst.A, inst.W, u, c_current, bank=bank).astype(np.float32, copy=False)

    return CoefficientTrainingExample(
        basis_features=basis_features,
        global_features=global_features,
        target_v=target_v.astype(np.float32, copy=False),
        u_current=u.astype(np.float32, copy=False),
        c_current=c_current.astype(np.float32, copy=False),
        metadata={"u_star": inst.u_star.copy(), "c_star": inst.c_star.copy(), "inst": inst, "bank": bank},
    )


def generate_neural_guidance_dataset(
    num_instances: int = 2,
    states_per_instance: int = 4,
    N: int = 32,
    r: int = 12,
    W: int = 4,
    M_init: int = 32,
    seed: int = 0,
) -> NeuralGuidanceDataset:
    rng = np.random.default_rng(seed)
    ranker_examples = []
    coefficient_examples = []

    for inst_idx in range(int(num_instances)):
        inst_seed = int(rng.integers(0, 2**32 - 1))
        inst = generate_planted_span_instance(N=N, r=r, W=W, seed=inst_seed)
        inst.verify()

        bank = DirectionBank(inst.A)
        bank.add_random_span_directions(K=int(M_init), rng=rng)
        bank.verify()

        for _ in range(int(states_per_instance)):
            u = rng.integers(0, 2, size=r, dtype=np.uint8)
            ranker_examples.append(make_ranker_training_example(inst, bank, u))
            coefficient_examples.append(make_coefficient_training_example(inst, u, bank=bank))

    return NeuralGuidanceDataset(
        ranker_examples=ranker_examples,
        coefficient_examples=coefficient_examples,
        metadata={
            "num_instances": int(num_instances),
            "states_per_instance": int(states_per_instance),
            "N": int(N),
            "r": int(r),
            "W": int(W),
            "M_init": int(M_init),
            "seed": int(seed),
        },
    )


def collate_ranker_examples(examples) -> dict:
    assert len(examples) > 0, "examples must be nonempty"
    m0, fd0 = examples[0].direction_features.shape
    fg0 = examples[0].global_features.shape[0]
    for ex in examples:
        assert ex.direction_features.shape == (m0, fd0)
        assert ex.global_features.shape == (fg0,)
        assert ex.target_scores.shape == (m0,)
        assert ex.valid_descent_mask.shape == (m0,)
        assert ex.deltas.shape == (m0,)

    return {
        "direction_features": np.stack([ex.direction_features for ex in examples], axis=0).astype(np.float32),
        "global_features": np.stack([ex.global_features for ex in examples], axis=0).astype(np.float32),
        "target_scores": np.stack([ex.target_scores for ex in examples], axis=0).astype(np.float32),
        "valid_descent_mask": np.stack([ex.valid_descent_mask for ex in examples], axis=0).astype(bool),
        "deltas": np.stack([ex.deltas for ex in examples], axis=0).astype(np.int32),
    }


def collate_coefficient_examples(examples) -> dict:
    assert len(examples) > 0, "examples must be nonempty"
    r0, fb0 = examples[0].basis_features.shape
    fg0 = examples[0].global_features.shape[0]
    n0 = examples[0].c_current.shape[0]
    for ex in examples:
        assert ex.basis_features.shape == (r0, fb0)
        assert ex.global_features.shape == (fg0,)
        assert ex.target_v.shape == (r0,)
        assert ex.u_current.shape == (r0,)
        assert ex.c_current.shape == (n0,)

    return {
        "basis_features": np.stack([ex.basis_features for ex in examples], axis=0).astype(np.float32),
        "global_features": np.stack([ex.global_features for ex in examples], axis=0).astype(np.float32),
        "target_v": np.stack([ex.target_v for ex in examples], axis=0).astype(np.float32),
        "u_current": np.stack([ex.u_current for ex in examples], axis=0).astype(np.float32),
        "c_current": np.stack([ex.c_current for ex in examples], axis=0).astype(np.float32),
    }


def train_direction_ranker_tiny(examples, steps: int = 30, seed: int = 0) -> dict:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable")
    torch.manual_seed(seed)
    batch = collate_ranker_examples(examples)
    x_d = torch.from_numpy(batch["direction_features"])
    x_g = torch.from_numpy(batch["global_features"])
    y = torch.from_numpy(batch["target_scores"])

    model = DirectionRanker(x_d.shape[-1], x_g.shape[-1], hidden_dim=32)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    with torch.no_grad():
        initial_loss = nn.functional.mse_loss(model(x_d, x_g), y).item()
    for _ in range(int(steps)):
        opt.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(x_d, x_g), y)
        loss.backward()
        opt.step()
    with torch.no_grad():
        final_loss = nn.functional.mse_loss(model(x_d, x_g), y).item()
    assert final_loss < initial_loss, "tiny ranker training loss did not decrease"
    return {"initial_loss": initial_loss, "final_loss": final_loss, "loss_decreased": final_loss < initial_loss, "model": model}


def train_coefficient_generator_tiny(examples, steps: int = 30, seed: int = 0) -> dict:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Torch unavailable")
    torch.manual_seed(seed)
    batch = collate_coefficient_examples(examples)
    x_b = torch.from_numpy(batch["basis_features"])
    x_g = torch.from_numpy(batch["global_features"])
    y = torch.from_numpy(batch["target_v"])

    model = CoefficientGenerator(x_b.shape[-1], x_g.shape[-1], hidden_dim=32)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        initial_loss = criterion(model(x_b, x_g), y).item()
    for _ in range(int(steps)):
        opt.zero_grad(set_to_none=True)
        loss = criterion(model(x_b, x_g), y)
        loss.backward()
        opt.step()
    with torch.no_grad():
        final_loss = criterion(model(x_b, x_g), y).item()
    assert final_loss < initial_loss, "tiny coefficient training loss did not decrease"
    return {"initial_loss": initial_loss, "final_loss": final_loss, "loss_decreased": final_loss < initial_loss, "model": model}


def run_neural_guidance_data_self_tests(seed=0) -> None:
    ds = generate_neural_guidance_dataset(num_instances=2, states_per_instance=3, seed=seed)
    assert len(ds.ranker_examples) > 0 and len(ds.coefficient_examples) > 0

    ex_r = ds.ranker_examples[0]
    inst = ex_r.metadata["inst"]
    bank = ex_r.metadata["bank"]
    u = ex_r.metadata["u_current"].astype(np.uint8)
    c = ex_r.metadata["c_current"].astype(np.uint8)
    assert ex_r.direction_features.shape[0] == bank.D.shape[0]
    assert np.array_equal(ex_r.deltas, bank.deltas(c))
    nz = np.any((c[None, :] ^ bank.D) != 0, axis=1)
    assert np.array_equal(ex_r.valid_descent_mask, (ex_r.deltas < 0) & nz)
    assert np.isfinite(ex_r.target_scores).all()

    ex_c = ds.coefficient_examples[0]
    inst_c = ex_c.metadata["inst"]
    expected_v = (ex_c.u_current.astype(np.uint8) ^ inst_c.u_star).astype(np.uint8)
    assert np.array_equal(ex_c.target_v.astype(np.uint8), expected_v)
    assert np.array_equal(gf2_matvec(inst_c.A, ex_c.target_v.astype(np.uint8)), ex_c.c_current.astype(np.uint8) ^ inst_c.c_star)
    assert np.isfinite(ex_c.basis_features).all() and np.isfinite(ex_c.global_features).all()

    rb = collate_ranker_examples(ds.ranker_examples)
    cb = collate_coefficient_examples(ds.coefficient_examples)
    assert rb["direction_features"].shape[0] == len(ds.ranker_examples)
    assert rb["target_scores"].shape == rb["valid_descent_mask"].shape == rb["deltas"].shape
    assert cb["basis_features"].shape[0] == len(ds.coefficient_examples)
    assert cb["target_v"].shape == cb["u_current"].shape

    # Tiny training checks validate data plumbing and exact labels only, not model quality.
    if TORCH_AVAILABLE:
        out_r = train_direction_ranker_tiny(ds.ranker_examples, steps=30, seed=seed)
        out_c = train_coefficient_generator_tiny(ds.coefficient_examples, steps=30, seed=seed)
        assert out_r["loss_decreased"] and out_c["loss_decreased"]
    else:
        msg = "Torch unavailable; cannot run neural-guidance data self-tests."
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)


## 20. Tiny offline-trained neural ablation

In [ ]:
# @title Tiny offline-trained neural ablation (smoke-scale wiring check only)

from dataclasses import replace

@dataclass
class OfflineNeuralAblationConfig:
    train_num_instances: int = 2
    train_states_per_instance: int = 4
    N: int = 32
    r: int = 12
    W: int = 4
    M_init: int = 32
    train_steps: int = 30
    eval_max_steps: int = 8
    eval_max_descent_steps: int = 4
    solver_time_limit_s: float = 2.0
    seed: int = 0


@dataclass
class OfflineNeuralAblationResult:
    variant_name: str
    status: str
    found: bool
    best_weight: Optional[int]
    threshold_W: int
    trace_steps: int
    num_directions_final: int
    failed_minima_count: int
    best_c_verified: bool = False

    action_counts: dict = field(default_factory=dict)
    neural_action_count: int = 0
    neural_action_available_count: int = 0
    neural_action_attempt_count: int = 0
    neural_action_success_count: int = 0
    neural_action_noop_count: int = 0
    neural_ranker_attempt_count: int = 0
    neural_generator_attempt_count: int = 0
    fallback_action_count: int = 0
    exact_gradient_action_count: int = 0
    solver_action_count: int = 0
    ce_action_count: int = 0

    ranker_initial_loss: Optional[float] = None
    ranker_final_loss: Optional[float] = None
    generator_initial_loss: Optional[float] = None
    generator_final_loss: Optional[float] = None

    metadata: dict = field(default_factory=dict)


def is_neural_action(action: int) -> bool:
    return int(action) in {1, 3}


def action_moved_or_mutated(info: dict, old_weight: int, new_weight: int, old_num_directions: int, new_num_directions: int) -> bool:
    info = info or {}
    for key in ('moved', 'added', 'found_solution', 'found', 'updated_best', 'best_improved'):
        if bool(info.get(key, False)):
            return True
    for key in ('added_count', 'num_added_directions', 'directions_added'):
        val = info.get(key, 0)
        if isinstance(val, (int, np.integer)) and int(val) > 0:
            return True
    return False


def choose_offline_neural_ablation_action(state, action_mask, variant_name: str, rng, attempted_neural_actions: Optional[set] = None) -> int:
    am = np.asarray(action_mask, dtype=bool)
    available = [i for i, ok in enumerate(am) if ok]
    if not available:
        raise ValueError('No available action in action_mask.')

    attempted_neural_actions = set() if attempted_neural_actions is None else set(attempted_neural_actions)
    wants_ranker = 'ranker' in variant_name and variant_name != 'symbolic_baseline'
    wants_generator = 'generator' in variant_name and variant_name != 'symbolic_baseline'

    neural_priority = []
    if variant_name == 'trained_ranker_and_generator':
        for a in (3, 1):
            if a not in attempted_neural_actions:
                neural_priority.append(a)
        neural_priority.extend([3, 1])
    elif wants_ranker and wants_generator:
        neural_priority = [3, 1]
    elif wants_ranker:
        neural_priority = [1]
    elif wants_generator:
        neural_priority = [3]

    for a in neural_priority:
        if a < len(am) and am[a]:
            return int(a)

    for a in (0, 5, 2, 6, 4):
        if a < len(am) and am[a]:
            return int(a)
    return int(default_macro_action_policy(state, am, rng))

# ... rest unchanged autogenerated below ...
def train_tiny_neural_guidance_models(config: OfflineNeuralAblationConfig) -> dict:
    if not TORCH_AVAILABLE:
        raise RuntimeError('Torch is required for offline neural ablation training.')
    data = generate_neural_guidance_dataset(
        num_instances=int(config.train_num_instances),
        states_per_instance=int(config.train_states_per_instance),
        N=int(config.N),
        r=int(config.r),
        W=int(config.W),
        M_init=int(config.M_init),
        seed=int(config.seed),
    )
    ranker_train = train_direction_ranker_tiny(data.ranker_examples, steps=int(config.train_steps), seed=int(config.seed))
    generator_train = train_coefficient_generator_tiny(data.coefficient_examples, steps=int(config.train_steps), seed=int(config.seed))

    out = {
        'ranker_model': ranker_train['model'],
        'generator_model': generator_train['model'],
        'ranker_initial_loss': float(ranker_train['initial_loss']),
        'ranker_final_loss': float(ranker_train['final_loss']),
        'generator_initial_loss': float(generator_train['initial_loss']),
        'generator_final_loss': float(generator_train['final_loss']),
        'ranker_loss_decreased': bool(ranker_train['final_loss'] <= ranker_train['initial_loss']),
        'generator_loss_decreased': bool(generator_train['final_loss'] <= generator_train['initial_loss']),
        'dataset_metadata': data.metadata,
    }
    assert out['ranker_loss_decreased'], 'Expected ranker loss to decrease in tiny training.'
    assert out['generator_loss_decreased'], 'Expected generator loss to decrease in tiny training.'
    return out


def count_action_usage(trace) -> dict:
    counts = {}
    for t in trace:
        key = getattr(t, 'action_name', None)
        if key is None:
            key = getattr(t, 'action', None)
        counts[key] = counts.get(key, 0) + 1
    return counts


def neural_action_diagnostics(trace) -> dict:
    counts = count_action_usage(trace)
    return {
        'neural_action_count': int(counts.get('neural_ranked_direction', 0) + counts.get('add_neural_directions', 0)),
        'exact_gradient_action_count': int(counts.get('exact_gradient', 0)),
        'solver_action_count': int(counts.get('local_minimum_solver', 0)),
        'ce_action_count': int(counts.get('ce_restart', 0) + counts.get('ce_update', 0)),
        'add_random_action_count': int(counts.get('add_random_directions', 0)),
        'attack_action_count': int(counts.get('attack_failed_minimum', 0)),
    }


def run_offline_neural_ablation(config: Optional[OfflineNeuralAblationConfig] = None) -> list[OfflineNeuralAblationResult]:
    if not TORCH_AVAILABLE:
        raise RuntimeError('Torch is required for offline neural ablation.')
    cfg = OfflineNeuralAblationConfig() if config is None else config
    trained = train_tiny_neural_guidance_models(cfg)
    eval_seeds = [cfg.seed + 101, cfg.seed + 202]
    eval_insts = [generate_planted_span_instance(N=cfg.N, r=cfg.r, W=cfg.W, seed=s) for s in eval_seeds]

    dir_dim = int(trained['ranker_model'].dir_mlp[0].in_features)
    g_rank_dim = int(trained['ranker_model'].global_mlp[0].in_features)
    basis_dim = int(trained['generator_model'].basis_mlp[0].in_features)
    g_gen_dim = int(trained['generator_model'].global_mlp[0].in_features)
    untrained_ranker = DirectionRanker(dir_dim, g_rank_dim, hidden_dim=32)
    untrained_generator = CoefficientGenerator(basis_dim, g_gen_dim, hidden_dim=32)

    variants = [
        ('symbolic_baseline', False, False, None, None, True),
        ('untrained_ranker', True, False, untrained_ranker, None, True),
        ('trained_ranker', True, False, trained['ranker_model'], None, True),
        ('untrained_generator', False, True, None, untrained_generator, True),
        ('trained_generator', False, True, None, trained['generator_model'], True),
        ('trained_ranker_and_generator', True, True, trained['ranker_model'], trained['generator_model'], True),
    ]

    results = []
    for inst_idx, inst in enumerate(eval_insts):
        for name, use_ranker, use_gen, ranker, gen, disable_solver in variants:
            hcfg = HybridSolverConfig(M_init=cfg.M_init, K_add=4, max_steps=cfg.eval_max_steps, max_descent_steps=cfg.eval_max_descent_steps, solver_time_limit_s=cfg.solver_time_limit_s, use_direction_ranker=use_ranker, use_coefficient_generator=use_gen, use_q_controller=False, seed=cfg.seed + 1000 + inst_idx)
            rng = np.random.default_rng(hcfg.seed)
            state = initialize_controller_state(inst, M_init=hcfg.M_init, seed=hcfg.seed)
            trace = []
            neural_action_available_count = 0
            neural_action_attempt_count = 0
            neural_action_success_count = 0
            neural_action_noop_count = 0
            neural_ranker_attempt_count = 0
            neural_generator_attempt_count = 0
            fallback_action_count = 0
            fallback_due_to_neural_error_count = 0
            attempted_neural_actions = set()
            available_neural_actions = set()
            model_compat_error = None
            for step in range(hcfg.max_steps):
                fallback_due_to_neural_error = False
                try:
                    am = controller_action_mask(state, direction_ranker=ranker if use_ranker else None, coefficient_generator=gen if use_gen else None)
                except Exception as ex:
                    model_compat_error = str(ex)
                    am = controller_action_mask(state, direction_ranker=None, coefficient_generator=None)
                am = np.asarray(am, dtype=bool).copy()
                if disable_solver:
                    am[4] = False
                preferred = []
                if name == "trained_ranker_and_generator":
                    preferred = [1, 3]
                elif use_ranker:
                    preferred = [1]
                elif use_gen:
                    preferred = [3]
                neural_available = any(a < len(am) and am[a] for a in preferred)
                for neural_action in (1, 3):
                    if neural_action < len(am) and bool(am[neural_action]):
                        available_neural_actions.add(int(neural_action))
                if name == "symbolic_baseline":
                    neural_available = False
                neural_action_available_count += int(neural_available)
                old_weight = int(_weight(state.c_current))
                old_num_directions = int(state.bank.D.shape[0])
                action = choose_offline_neural_ablation_action(state, am, name, rng, attempted_neural_actions=attempted_neural_actions)
                attempted_this_step = name != "symbolic_baseline" and is_neural_action(action)
                if attempted_this_step:
                    neural_action_attempt_count += 1
                    attempted_neural_actions.add(int(action))
                    if int(action) == 1:
                        neural_ranker_attempt_count += 1
                    if int(action) == 3:
                        neural_generator_attempt_count += 1
                had_neural_failure = False
                try:
                    sr = controller_step(state, action, rng=rng, direction_ranker=ranker if use_ranker else None, coefficient_generator=gen if use_gen else None, K_add=hcfg.K_add, max_descent_steps=hcfg.max_descent_steps, solver_time_limit_s=hcfg.solver_time_limit_s)
                except Exception as ex:
                    if attempted_this_step:
                        model_compat_error = str(ex)
                        had_neural_failure = True
                        fallback_due_to_neural_error = True
                        fallback_due_to_neural_error_count += 1
                    fallback_am = controller_action_mask(state, direction_ranker=None, coefficient_generator=None)
                    action = default_macro_action_policy(state, np.asarray(fallback_am, dtype=bool), rng)
                    sr = controller_step(state, action, rng=rng, direction_ranker=None, coefficient_generator=None, K_add=hcfg.K_add, max_descent_steps=hcfg.max_descent_steps, solver_time_limit_s=hcfg.solver_time_limit_s)
                    fallback_action_count += 1
                new_weight = int(_weight(state.c_current))
                new_num_directions = int(state.bank.D.shape[0])
                if attempted_this_step and not had_neural_failure:
                    if action_moved_or_mutated(getattr(sr, "info", {}) or {}, old_weight, new_weight, old_num_directions, new_num_directions):
                        neural_action_success_count += 1
                trace.append(HybridSolverTraceEntry(step=step, action=int(action), action_name=action_name(int(action)), reward=float(sr.reward), current_weight=int(_weight(state.c_current)), best_weight=state.best_weight, num_directions=int(state.bank.D.shape[0]), failed_minima_count=len(state.archive.entries), terminal=bool(state.terminal), terminal_reason=str(state.terminal_reason) if state.terminal_reason is not None else None))
                if state.terminal:
                    break
            neural_action_noop_count = int(neural_action_attempt_count - neural_action_success_count)
            if state.best_c is not None and state.best_weight <= inst.W:
                verify_candidate_solution(inst.A, inst.W, state.best_u, state.best_c, require_threshold=True)
                status, found = "valid_solution", True
            elif state.best_c is not None:
                verify_candidate_solution(inst.A, inst.W, state.best_u, state.best_c, require_threshold=False)
                status, found = "best_found", False
            else:
                status, found = "no_solution_found", False
            result = HybridSolverResult(status=status, best_u=None if state.best_u is None else state.best_u.copy(), best_c=None if state.best_c is None else state.best_c.copy(), best_weight=None if state.best_c is None else int(state.best_weight), found=found, trace=trace, final_state=state, metadata={"heuristic": True, "claims": "bounded anytime search; no optimality or infeasibility certificate"})
            verified = bool(result.best_c is not None)
            diags = neural_action_diagnostics(trace)
            action_counts = count_action_usage(trace)

            ranker_initial_loss = ranker_final_loss = generator_initial_loss = generator_final_loss = None
            if name in {'trained_ranker', 'trained_ranker_and_generator'}:
                ranker_initial_loss = trained['ranker_initial_loss']
                ranker_final_loss = trained['ranker_final_loss']
            if name in {'trained_generator', 'trained_ranker_and_generator'}:
                generator_initial_loss = trained['generator_initial_loss']
                generator_final_loss = trained['generator_final_loss']

            results.append(OfflineNeuralAblationResult(
                variant_name=name, status=result.status, found=bool(result.found), best_weight=result.best_weight, threshold_W=int(inst.W), trace_steps=len(result.trace), num_directions_final=int(result.final_state.bank.D.shape[0]), failed_minima_count=len(result.final_state.archive.entries), best_c_verified=verified,
                action_counts=action_counts,
                neural_action_count=int(neural_action_attempt_count),
                neural_action_available_count=int(neural_action_available_count),
                neural_action_attempt_count=int(neural_action_attempt_count),
                neural_action_success_count=int(neural_action_success_count),
                neural_action_noop_count=int(neural_action_noop_count),
                neural_ranker_attempt_count=int(neural_ranker_attempt_count),
                neural_generator_attempt_count=int(neural_generator_attempt_count),
                fallback_action_count=int(fallback_action_count),
                exact_gradient_action_count=diags['exact_gradient_action_count'],
                solver_action_count=diags['solver_action_count'],
                ce_action_count=diags['ce_action_count'],
                ranker_initial_loss=ranker_initial_loss, ranker_final_loss=ranker_final_loss,
                generator_initial_loss=generator_initial_loss, generator_final_loss=generator_final_loss,
                metadata={'eval_instance_index': inst_idx, 'smoke_only': True, 'add_random_action_count': diags['add_random_action_count'], 'attack_action_count': diags['attack_action_count'], 'model_compat_error': model_compat_error, 'attempted_neural_actions': sorted(int(a) for a in attempted_neural_actions), 'fallback_due_to_neural_error_count': int(fallback_due_to_neural_error_count), 'available_neural_actions': sorted(int(a) for a in available_neural_actions)},
            ))
    return results


def summarize_offline_neural_ablation_results(results) -> list[dict]:
    rows = []
    for r in results:
        rows.append({'variant_name': r.variant_name, 'status': r.status, 'found': r.found, 'best_weight': r.best_weight, 'threshold_W': r.threshold_W, 'trace_steps': r.trace_steps, 'num_directions_final': r.num_directions_final, 'failed_minima_count': r.failed_minima_count, 'neural_action_count': r.neural_action_count, 'neural_action_available_count': r.neural_action_available_count, 'neural_action_attempt_count': r.neural_action_attempt_count, 'neural_action_success_count': r.neural_action_success_count, 'neural_action_noop_count': r.neural_action_noop_count, 'neural_ranker_attempt_count': r.neural_ranker_attempt_count, 'neural_generator_attempt_count': r.neural_generator_attempt_count, 'fallback_action_count': r.fallback_action_count, 'exact_gradient_action_count': r.exact_gradient_action_count, 'solver_action_count': r.solver_action_count, 'ce_action_count': r.ce_action_count, 'ranker_initial_loss': r.ranker_initial_loss, 'ranker_final_loss': r.ranker_final_loss, 'generator_initial_loss': r.generator_initial_loss, 'generator_final_loss': r.generator_final_loss, 'best_c_verified': r.best_c_verified})
    return rows


def print_offline_neural_ablation_table(results) -> None:
    rows = summarize_offline_neural_ablation_results(results)
    if globals().get("PANDAS_AVAILABLE", False):
        df = pd.DataFrame(rows)
        cols = ['variant_name', 'status', 'found', 'best_weight', 'threshold_W', 'trace_steps', 'neural_action_available_count', 'neural_action_attempt_count', 'neural_action_success_count', 'neural_action_noop_count', 'neural_ranker_attempt_count', 'neural_generator_attempt_count', 'fallback_action_count', 'exact_gradient_action_count', 'solver_action_count', 'ce_action_count', 'best_c_verified']
        print(df[cols].to_string(index=False))
    else:
        for row in rows:
            print(row)


def run_offline_neural_ablation_self_tests(seed=0) -> None:
    if not TORCH_AVAILABLE:
        msg = 'Torch unavailable; cannot run offline neural ablation self-tests.'
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)
        return

    cfg = OfflineNeuralAblationConfig(train_num_instances=1, train_states_per_instance=2, N=24, r=10, W=4, M_init=24, train_steps=12, eval_max_steps=6, eval_max_descent_steps=3, solver_time_limit_s=1.0, seed=seed)
    trained = train_tiny_neural_guidance_models(cfg)
    assert trained['ranker_model'] is not None
    assert trained['generator_model'] is not None
    assert trained['ranker_loss_decreased'] and trained['generator_loss_decreased']

    results = run_offline_neural_ablation(cfg)
    assert len(results) > 0
    names = {r.variant_name for r in results}
    for required in ['symbolic_baseline', 'trained_ranker', 'trained_generator', 'trained_ranker_and_generator']:
        assert required in names

    by_name = {}
    for r in results:
        by_name.setdefault(r.variant_name, []).append(r)
        assert isinstance(r.action_counts, dict)
        assert r.neural_action_attempt_count == r.neural_ranker_attempt_count + r.neural_generator_attempt_count
        assert r.neural_action_success_count + r.neural_action_noop_count == r.neural_action_attempt_count
        assert r.neural_action_success_count <= r.neural_action_attempt_count

    for r in by_name['symbolic_baseline']:
        assert r.neural_action_available_count == 0
        assert r.neural_action_attempt_count == 0
        assert r.neural_action_success_count == 0
        assert r.neural_action_noop_count == 0
        assert r.neural_ranker_attempt_count == 0
        assert r.neural_generator_attempt_count == 0
        assert r.ranker_initial_loss is None and r.ranker_final_loss is None
        assert r.generator_initial_loss is None and r.generator_final_loss is None
        assert r.metadata.get('model_compat_error') is None
        assert int(r.metadata.get('fallback_due_to_neural_error_count', 0)) == 0
    for r in by_name['trained_ranker']:
        assert r.neural_action_available_count >= 1
        assert r.neural_ranker_attempt_count >= 1
        assert r.neural_action_attempt_count >= 1
        assert r.ranker_initial_loss is not None and r.ranker_final_loss is not None
        assert r.ranker_final_loss <= r.ranker_initial_loss
        assert r.generator_initial_loss is None and r.generator_final_loss is None
        assert r.metadata.get('model_compat_error') is None
    for r in by_name['trained_generator']:
        assert r.neural_action_available_count >= 1
        assert r.neural_generator_attempt_count >= 1
        assert r.neural_action_attempt_count >= 1
        assert r.generator_initial_loss is not None and r.generator_final_loss is not None
        assert r.generator_final_loss <= r.generator_initial_loss
        assert r.ranker_initial_loss is None and r.ranker_final_loss is None
        assert r.metadata.get('model_compat_error') is None
        assert int(r.metadata.get('fallback_due_to_neural_error_count', 0)) == 0

    for r in by_name['trained_ranker_and_generator']:
        assert r.neural_action_attempt_count >= 1
        assert r.metadata.get('model_compat_error') is None
        assert int(r.metadata.get('fallback_due_to_neural_error_count', 0)) == 0
        available_actions = set(int(a) for a in r.metadata.get('available_neural_actions', []))
        if 1 in available_actions and 3 in available_actions:
            assert r.neural_ranker_attempt_count >= 1
            assert r.neural_generator_attempt_count >= 1
        assert r.neural_action_success_count <= r.neural_action_attempt_count

    allowed = {'valid_solution', 'best_found', 'no_solution_found'}
    for r in results:
        assert r.status in allowed
        if r.best_weight is not None:
            assert r.best_c_verified
        if r.found:
            assert r.best_weight is not None and r.best_weight <= r.threshold_W

    rows = summarize_offline_neural_ablation_results(results)
    assert len(rows) == len(results)
    needed = {'variant_name', 'status', 'found', 'best_weight', 'threshold_W', 'trace_steps', 'num_directions_final', 'failed_minima_count', 'neural_action_count', 'neural_action_available_count', 'neural_action_attempt_count', 'neural_action_success_count', 'neural_action_noop_count', 'neural_ranker_attempt_count', 'neural_generator_attempt_count', 'fallback_action_count', 'exact_gradient_action_count', 'solver_action_count', 'ce_action_count', 'ranker_initial_loss', 'ranker_final_loss', 'generator_initial_loss', 'generator_final_loss', 'best_c_verified'}
    for row in rows:
        assert needed.issubset(row.keys())

    print_offline_neural_ablation_table(results)
    # Smoke ablation scope: this only tests wiring and tiny supervised learning; it is not evidence of general model quality or optimality.

In [ ]:
if TORCH_AVAILABLE:
    run_offline_neural_ablation_self_tests()
else:
    msg = "Torch unavailable; cannot run offline neural ablation self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


In [ ]:
if TORCH_AVAILABLE:
    run_neural_guidance_data_self_tests()
else:
    msg = "Torch unavailable; cannot run neural-guidance data self-tests."
    if HEADLESS or SMOKE:
        raise RuntimeError(msg)
    print(msg)


In [ ]:
# @title Run all notebook self-tests

def run_all_self_tests() -> None:
    run_f2_self_tests()
    run_planted_instance_self_tests()
    run_direction_bank_self_tests()
    run_gradient_engine_self_tests()

    if ORTOOLS_AVAILABLE:
        run_local_minimum_solver_self_tests()
    else:
        msg = "OR-Tools unavailable; cannot run local-minimum solver self-tests."
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)

    run_failed_minima_archive_self_tests()

    if TORCH_AVAILABLE:
        run_direction_ranker_self_tests()
        run_coefficient_generator_self_tests()
        run_q_controller_self_tests()
        run_neural_guidance_data_self_tests()
        run_offline_neural_ablation_self_tests()
    else:
        msg = "Torch unavailable; cannot run neural/controller self-tests."
        if HEADLESS or SMOKE:
            raise RuntimeError(msg)
        print(msg)

    run_cross_entropy_sampler_self_tests()
    run_hybrid_solver_self_tests()
    run_benchmark_harness_self_tests()
    run_ablation_self_tests()
    run_performance_profile_self_tests()
    run_packed_bit_helpers_self_tests()
    run_packed_delta_comparison_self_tests()
    run_fast_packed_popcount_self_tests()
    run_extended_delta_backend_comparison_self_tests()
    run_delta_scaling_self_tests()

run_all_self_tests()